# [실습] 요율표 가정 점검 — 표준오차를 다시 잰다

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 오늘 우리가 하게 될 일

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">모형추정 심화통계 · Day 4</div>
<h1 style="color:#000000;margin:10px 0">요율표 가정 확인 — 표준오차의 재측정</h1>
<div style="font-size:1.05em;color:#656565">세그먼트 8개의 점추정에 신뢰구간을 산출하고, 그 구간이 전제하는 가정(등분산·독립)까지 확인합니다</div>
</div>

## 오늘 할 일

어제 우리는 계약마다 다른 사고율을 매기는 포아송 GLM 으로 요율 배율 8개를 얻었고, 어느 후보를 요율표로 삼을지는 오늘 교차검증 뒤로 미뤄 두었습니다. 오늘 아침 재무팀에서 메일이 도착했습니다.

> **"요율표 초안 잘 받았습니다. 그런데 숫자가 전부 점 하나로만 적혀 있네요. 이사회에서 '이 수치가 얼마나 믿을 만하냐'고 물으면 뭐라고 답합니까?"**

점추정 하나로는 답할 수 없는 질문이고, 그 답이 **구간**입니다. 오늘 구간을 산출할 값은 어제 모형의 예측이 아니라 세그먼트마다 사고 건수 합을 노출 합으로 나눈 사고율입니다. **세그먼트**는 요율을 따로 매기는 조건 하나와 그 조건에 속한 계약 전부를 말하고, 요율표의 한 행이 됩니다.

오늘 사용할 세그먼트는 도시화 등급 6개와 지역 2개, 모두 8개입니다. 도시화 등급 6개가 계약 678,013건을 남김없이 나누고 지역 R43·R42 의 계약은 그 6개 안에 들어 있어, 이 8개는 요율표 한 행이 될 후보이지 서로 겹치지 않는 8행이 아닙니다.

세그먼트마다 부트스트랩 신뢰구간을 산출하고, 계수 40개에 정규화의 패널티 항을 적용해 보고, 교차검증으로 어제의 순위를 새 데이터에서 다시 매깁니다. 과제에서는 그 구간이 어떤 가정 위에서 계산되었는지를 잔차로 확인하고, 얼마나 넓혀 보고할지까지 정합니다.

코드는 미션 3개와 과제 1개 모두 AI 에게 맡깁니다. 그 가운데 미션 2 는 전임 담당자가 남긴 코드를 읽고 어디가 틀렸는지 찾아, 무엇을 어떻게 고칠지 지시로 적는 함정 미션입니다. 우리는 **시킬 일을 정하고 결과를 확인하는 역할**을 맡습니다.

## 오늘의 데이터

**freMTPL2freq** 는 프랑스 자동차 책임보험(Motor Third-Party Liability)의 실제 계약 기록입니다. 계약 한 건이 한 행이고 678,013행 × 12컬럼이며, 공개 저장소 OpenML 에서 로그인 없이 내려받습니다.

오늘 쓰는 컬럼은 역할이 2가지입니다. 사고율을 계산하는 컬럼이 4일 동안 사용한 `ClaimNb` · `Exposure` 2개이고, 모형의 설명변수가 7개입니다. 설명변수 7개는 미션 2 에서 범주형을 0/1 더미 열로 펼친 숫자 행렬, 곧 이 노트북에서 **설계행렬**이라 부르는 행렬의 열 40개가 되고(절편까지 계수 41개), 미션 3 과 과제는 이 가운데 핵심 4개인 `BonusMalus` · `DrivAge` · `Area` · `VehGas` 를 formula 로 적어 적합합니다. 7개를 모두 적는 것은 미션 3 의 확장 후보 하나입니다.

| 컬럼 | 뜻 | 단위 | 받자마자 고정하는 규칙 |
|---|---|---|---|
| `ClaimNb` | 보험 기간 중 사고(청구) 건수 | 건 | 4건 상한 처리 |
| `Exposure` | 그 계약이 실제로 유지된 기간, 곧 **노출** | 년 | 1.0년 상한 처리 |
| 설명변수 7개 | 할증점수 `BonusMalus` · 운전자 연령 `DrivAge` · 도시화 등급 `Area` · 연료 `VehGas` · 차량 브랜드 `VehBrand` · 지역 `Region` · 인구밀도 `Density` | 변수마다 다름 | 범주형 4개는 문자열을 정리해 범주형으로, `Density` 는 로그를 취해 사용 |

계약 4건 중 3건이 1년을 채우지 못하고 종료되므로, 계약마다 다른 관측 기간인 노출이 사고율의 분모입니다. 이 규칙 위에서 나온 **전체 계약(회사 전체)의 연간 사고율 0.100614 건/년** 이 오늘 모든 구간을 읽는 기준선입니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_001.png" width="720" height="180"/>
<p align="center"><em>▲ 4일 동안 사용한 freMTPL2freq 를 전처리 전 그대로 그리면 패널 3개 모두 꼬리가 길게 뻗는다 — 계약당 사고 건수는 16건까지, 사고 건수를 노출로 나눈 빈도는 700 건/년 부근까지 분포한다. <code>ClaimNb</code> 를 4건에서, <code>Exposure</code> 를 1.0년에서 상한 처리한 이유가 이 꼬리다 (출처: scikit-learn documentation)</em></p>

## 이 노트북을 쓰는 법

**환경 안내 한 가지** — ▶ 실행은 Google Colab 연결 안내를 따르고, AI 튜터는 상단 「Claude 도우미」 설치 뒤 사용합니다(온보딩 레슨 참고).

**맨 위 설치·폰트 셀은 어느 쪽에서 열었든 먼저 한 번 실행합니다.** 채점 셀을 준비하는 방법만 2가지입니다.

| 어디서 열었나 | 채점 셀을 어떻게 하나 |
|---|---|
| LMS 에서 열었을 때 | Colab 이 연결되면 자동으로 준비됩니다 — 코드 패널 위쪽의 「채점 준비됨」이 표시된 뒤 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 출력되지 않으면 그 「채점 준비됨」을 누르세요 |
| 내려받아 열었을 때 | 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다(코드는 접혀 있어도 실행됩니다) |

미션마다 코드 셀이 2개입니다. 앞은 **준비 셀**(그대로 실행), 뒤는 **PROMPT 셀**(시킬 일을 적는 곳)입니다. 시킬 일을 `PROMPT` 에 내 말로 적어 AI 튜터에게 전달하면 `# --- 여기부터 AI 코드 ---` 행 아래가 AI 코드가 들어갈 위치이고, 다른 AI 에서 받은 코드도 그 행 아래에 직접 붙여 넣습니다.

실행하는 순간 **실행 결과(출력)에 채점 결과**가 출력됩니다. 기준을 충족하면 그 아래 「🔓 완성본 풀이 보기」가 열려 내 코드와 비교할 수 있고, 충족하지 못하면 충족하지 못한 항목과 AI 튜터에 보낼 질문이 표시됩니다.

채점 코드는 변수 이름이 아니라 셀이 출력한 값과 그림을 확인하므로 변수 이름은 무엇으로 지어도 됩니다. 그래프를 만든 미션은 내 그림과 참조 그림이 나란히 한 장으로 표시됩니다 — 왼쪽이 내 그림입니다.

「준비 셀」은 미션마다 쓸 변수를 처음부터 만듭니다. 며칠 뒤에 뒤쪽 미션만 열어도 맨 위 설치·폰트 셀 다음에 그 미션의 「준비 셀」부터 실행하면 됩니다.

**주고받기 규칙 2가지** — AI 와 막혔을 때는 이 2가지만 지킵니다.

| 이럴 때 | 이렇게 합니다 |
|---|---|
| 같은 말을 2회째 되풀이하게 될 때 | 멈춥니다. 말을 바꾸지 말고 `PROMPT` 에 **빠진 조건을 하나** 추가합니다 |
| 그래도 안 될 때 | 앞 대화를 이어서 쓰지 말고, 요구를 한 번에 묶어 다시 적어 새 메시지로 보냅니다 |

## 오늘 참조하는 이론

본문의 「이론 n.」 이 이론의 어느 제목이고 거기서 무슨 값을 가져오는지 정리했습니다.

| 이론 | 오늘 쓰는 곳 | 가져오는 값 |
|---|---|---|
| 1. 표준오차 | 미션 1 배경 | 점추정에 표준오차의 2배를 더하고 빼는 방식으로는 구간을 설정하지 못하는 세그먼트 크기 |
| 2. 부트스트랩 | 미션 1 | 재표본 수의 실무 하한 1,000 · 백분위 구간 |
| 4. 신뢰구간을 이용한 의사결정 | 미션 1 배경 · 오늘의 정리 | 계약 1,326건인 지역 R43 의 표기 · 실무 기준(ROPE) |
| 5. 잔차 진단 | 과제 | 예측값 순으로 나눈 구간 15개 · 이탈도 잔차 |
| 6. 정규화와 교차검증 | 미션 2 · 미션 3 | alpha = 0.000001 에서의 표준화 계수 40개의 크기 합 **1.0880** |

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Illustration_bootstrap.svg" width="640" height="195"/>
<p align="center"><em>▲ 오늘 사용하는 주요 방법 — 원 표본에서 복원추출로 표본을 여러 개 생성해 통계량의 분포를 구성하는 부트스트랩 (출처: Wikipedia)</em></p>

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "scipy>=1.11" "matplotlib>=3.7" "statsmodels>=0.14" "scikit-learn>=1.3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_poisson_deviance
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150      # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 기준선=남색, 관측=파랑, 구간·강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다

print("사용 폰트:", FONT,
      "| 색 팔레트 준비 완료 — 채점 셀 준비 방식은 실행 환경에 따라 다릅니다"
      " (위 「이 노트북을 쓰는 법」 의 표 참고)")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 붙잡아 바로 아래에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 행들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6eyJ2YWx1ZXMiOnsi7KeA7JetIFI0MyDsnZggOTUlIOq1rOqwhCDtlZ"
           "jtlZwgKOqxtC/rhYQpIjpbMC4wNywwLjA4LCLqsbQv64WEIl0sIuyngOyXrSBS"
           "NDMg7J2YIDk1JSDqtazqsIQg7IOB7ZWcICjqsbQv64WEKSI6WzAuMTIzLDAuMT"
           "M2LCLqsbQv64WEIl19LCJwbG90Ijp7ImtpbmQiOiJpbnRlcnZhbCIsIm5fc3Bh"
           "bnMiOls4LDIwXSwibl9wb2ludHMiOls4LDI0XSwic3Bhbl9tYXgiOlswLjA0LD"
           "AuMDc1XSwic3Bhbl9taW4iOlswLjAsMC4wMTJdLCJ4X3JhbmdlIjpbMC4wNzIs"
           "MC4xNDhdLCJ2bGluZV94IjpbMC4wOTk1LDAuMTAxN10sIuy2leudvOuyqCI6dH"
           "J1ZX19LCJtMiI6eyJ2YWx1ZXMiOnsi6rWQ7LCo6rKA7KadIOydtO2DiOuPhOqw"
           "gCDqsIDsnqUg64Ku7J2AIOyngOygkOydmCDsnbTtg4jrj4QiOlswLjYwNiwwLj"
           "YxXSwi67KM7KCQ7J20IOqxsOydmCDsl4bripQgYWxwaGEg7JeQ7ISc7J2YIO2R"
           "nOykgO2ZlCDqs4TsiJggNDDqsJzsnZgg7YGs6riwIO2VqSI6WzEuMDIsMS4xNl"
           "19LCJwbG90Ijp7ImtpbmQiOiJiYXIiLCJuIjo2LCJ5bWF4IjpbMC42MTUsMC42"
           "MzJdLCLstpXrnbzrsqgiOnRydWV9fSwibTMiOnsidmFsdWVzIjp7Iu2ZleyepS"
           "DrqqjtmJUoZnVsbCnsnZgg7Y+065OcIO2Pieq3oCDsnbTtg4jrj4QiOlswLjYw"
           "NzgsMC42MDg0NV0sImNvcmUgLSBmdWxsIOynneyngOydgCDssKjsnbTsnZgg7Y"
           "+J6regIC8g7ZGc7KSA7Jik7LCoICjrsLApIjpbNi4wLDExLjAsIuuwsCJdfSwi"
           "cGxvdCI6eyJraW5kIjoiYmFyIiwibiI6NSwieW1heCI6WzAuMDAyNSwwLjAwND"
           "VdLCLstpXrnbzrsqgiOnRydWV9fSwi6rO87KCcIjp7InZhbHVlcyI6eyLtlaDs"
           "pp3soJDsiJgg6rOE7IiY7J2YIO2RnOykgOyYpOywqCDrsLDsiJgg4oCUIOq1sO"
           "ynkSDDtyDtj6zslYTshqEgKOuwsCkiOls1LjAsNS42LCLrsLAiXSwi67aE7IKw"
           "w7ftj4nqt6DsnbQgMSDsnITsqr3snbgg7Lm4IOyImCAo7Lm4KSI6WzE1LDE1LC"
           "LsubgiXX0sInBsb3QiOnsia2luZCI6ImludGVydmFsIiwibl9zcGFucyI6WzIy"
           "LDIyXSwibl9wb2ludHMiOlsyMiwyMl0sInNwYW5fbWF4IjpbMC4wNDUsMC4wN1"
           "0sInNwYW5fbWluIjpbMC4wMDMsMC4wMDddLCJ4X3JhbmdlIjpbLTAuMDI5LDAu"
           "MDI4XSwidmxpbmVfeCI6Wy0wLjAwMDUsMC4wMDA1XSwi7LaV652867KoIjp0cn"
           "VlfX19")
_REVEAL = {"m1":                   # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
           "8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIOOAjEFJIOy9lOuTnCDsi5zsnp"
           "Eg7ZaJ44CNIOyVhOuemOyXkCDrk6TslrTsmYDslbwg7ZWY64qUIOy9lOuTnOuK"
           "lCDsnbQg7KCV64+E7J6F64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrtoDrpb"
           "gg7ZSE66Gs7ZSE7Yq4IOyYiAoKICAgIOydtOuvuCDrp4zrk6TslrTsoLgg7J6I"
           "64qUIGRmLCBTRUdNRU5UUywgQiwgU0VFRCwgTEFNX0FMTCDsnYQg6re464yA66"
           "GcIOyTsOqzoCDri6Tsi5wg66eM65Ok7KeA64qUIOunkOyVhCDspJguIFNFR01F"
           "TlRTIOuKlCDshLjqt7jrqLztirgg7J2066aE7J2EIOq3uCDshLjqt7jrqLztir"
           "jsnZgg6rOE7JW97J2EIOqzoOultOuKlCDrtojrpqzslrgKICAgIOuniOyKpO2B"
           "rOyXkCDsnbTslrQg65GUIOyCrOyghCA46rCc7JW8LiDshLjqt7jrqLztirjrp4"
           "jri6Qg6re4IOyEuOq3uOuovO2KuOydmCDqs4Tslb3snYQg67O17JuQ7LaU7Lac"
           "66GcIOybkCDtkZzrs7jqs7wg6rCZ7J2AIO2BrOq4sOunjO2BvCBCIOuyiCDri6"
           "Tsi5wg672R7JWELCDsnqztkZzrs7jrp4jri6Qg7IKs6rOgIOqxtOyImCBDbGFp"
           "bU5iIOydmCDtlansnYQg64W47LacCiAgICBFeHBvc3VyZSDsnZgg7ZWp7Jy866"
           "GcIOuCmOuIiCDsgqzqs6DsnKjsnYQg66qo7JWEIOykmCDigJQg64Kc7IiY64qU"
           "IFNFRUQg66GcIOyLnOyeke2VmOqzoCwg67aE66qo64qUIOqzhOyVvSDsiJjqsI"
           "Ag7JWE64uI6528IOuFuOy2nCDtlansnbTslbwuIOq3uCBCIOqwnOyXkOyEnCAy"
           "LjUlIOyZgCA5Ny41JSDrsLHrtoTsnITrpbwKICAgIOyekOuluCDqsoPsnbQg6r"
           "e4IOyEuOq3uOuovO2KuOydmCA5NSUg6rWs6rCE7J206rOgLCDshLjqt7jrqLzt"
           "irgg7J2066aEIMK3IOqzhOyVvSDsiJggwrcg7IKs6rOgIOqxtOyImCDCtyDrhb"
           "jstpwgwrcg7KCQ7LaU7KCVIOyCrOqzoOycqCDCtyDqtazqsIQg7ZWY7ZWcIGxv"
           "OTUgwrcg6rWs6rCEIOyDge2VnCBoaTk1IMK3IOyerO2RnOuzuCDrtoTtj6zsnZ"
           "gKICAgIO2RnOykgO2OuOywqOulvCDtlZwg7ZaJ7Jy866GcIOuLtOydgCDtkZzr"
           "pbwg66eM65Ok7Ja0IOq1rOqwhCDquLjsnbQg7Je0IGDtj61gIOqzvCDsoJDstp"
           "TsoJXsnLzroZwg64KY64iIIOyDgeuMgCDqtazqsIQg6ri47J20IOyXtCBg7IOB"
           "64yA7Y+tYCDsnYQg642U7ZW0IOykmC4g6re4IOuLpOydjCAi7KeA7JetIFI0My"
           "A5NSUg6rWs6rCECiAgICA9ICjtlZjtlZwpIH4gKOyDge2VnCkg6rG0L+uFhCIg"
           "7J2EIOyGjOyImOygkCDslYTrnpggNeyekOumrOq5jOyngCDtlZwg7ZaJLCBBcm"
           "VhIEMg7JmAIFJlZ2lvbiBSNDMg7J2YIOyDgeuMgCDqtazqsIQg6ri47J2066W8"
           "IOuCmOuegO2eiCDrhpPsnYAg7ZWcIO2WiSwgUjQzIOydmCDrtoDtirjsiqTtir"
           "jrnqkg7ZGc7KSA7Jik7LCo66W8CiAgICDqsbQv64WEIOuLqOychOuhnCDtlZwg"
           "7ZaJIHByaW50IO2VtCDspJguIOq3uOuemO2UhOuKlCDtlZwg7J6l66eMIOKAlC"
           "DshLjqt7jrqLztirggOOqwnOulvCDshLjroZzroZwg67Cw7LmY7ZWY6rOgIOyE"
           "uOq3uOuovO2KuOuniOuLpCBsbzk1IOu2gO2EsCBoaTk1IOq5jOyngOulvCDqtb"
           "XsnYAg6rCA66GcIOunieuMgOuhnCDquIvqs6Ag7KCQ7LaU7KCVIOychOy5mOyX"
           "kAogICAg66eI7Luk66W8IO2VmOuCmOyUqSDtkZzsi5ztlZwg65KkLCBMQU1fQU"
           "xMIOychOy5mOyXkCDshLjroZwg7KCQ7ISg7J2EIOq4i+qzoCDshLjroZzstpUg"
           "64iI6riI7JeQIOyEuOq3uOuovO2KuCDsnbTrpoTsnYQg7KCB6rOgIOuRkCDstp"
           "Xsl5Ag652867Ko7J2EIOuLrOyVhCDspJguCgogICAgZGVmIGJvb3RfbGFtYmRh"
           "KHksIEUsIEIsIHNlZWQpOiAgICAgICAgICAgICAgICAgICAgIyDqt7gg7IS46r"
           "e466i87Yq47J2YIOyerO2RnOuzuCDsgqzqs6DsnKggQuuyjAogICAgICAgIHJu"
           "ZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgICAgIG4gPSBsZW"
           "4oeSkgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyDsnqzt"
           "kZzrs7gg7YGs6riw64qUIOybkCDtkZzrs7jqs7wg6rCZ64ukCiAgICAgICAgb3"
           "V0ID0gbnAuZW1wdHkoQikKICAgICAgICBmb3IgYiBpbiByYW5nZShCKToKICAg"
           "ICAgICAgICAgaSA9IHJuZy5pbnRlZ2VycygwLCBuLCBuKSAgICAgICAgICAgIC"
           "AgICAgICMg67O17JuQ7LaU7LacIOKAlCDqsJnsnYAg6rOE7JW97J20IOuRkCDr"
           "sogg672R7Z6QIOyImCDsnojri6QKICAgICAgICAgICAgb3V0W2JdID0geVtpXS"
           "5zdW0oKSAvIEVbaV0uc3VtKCkgICAgICAgICAgICMg67aE66qo64qUIOqzhOyV"
           "vSDsiJjqsIAg7JWE64uI6528IOuFuOy2nCDtlakKICAgICAgICByZXR1cm4gb3"
           "V0CgogICAgcm93cyA9IFtdCiAgICBmb3IgbmFtZSwgbWFzayBpbiBTRUdNRU5U"
           "Uy5pdGVtcygpOgogICAgICAgIGQgPSBkZlttYXNrXQogICAgICAgIHMgPSBib2"
           "90X2xhbWJkYShkWyJDbGFpbU5iIl0udG9fbnVtcHkoZmxvYXQpLCBkWyJFeHBv"
           "c3VyZSJdLnRvX251bXB5KGZsb2F0KSwgQiwgU0VFRCkKICAgICAgICBsbywgaG"
           "kgPSBucC5wZXJjZW50aWxlKHMsIDIuNSksIG5wLnBlcmNlbnRpbGUocywgOTcu"
           "NSkgICAgICMg7JaR7Kq97J2EIDIuNSXslKkg7J6Q66W4IOuTseq8rOumrCDqta"
           "zqsIQKICAgICAgICByb3dzLmFwcGVuZCh7IuyEuOq3uOuovO2KuCI6IG5hbWUs"
           "ICJuIjogbGVuKGQpLCAiY2xhaW1zIjogaW50KGRbIkNsYWltTmIiXS5zdW0oKS"
           "ksCiAgICAgICAgICAgICAgICAgICAgICJleHBvc3VyZSI6IGZsb2F0KGRbIkV4"
           "cG9zdXJlIl0uc3VtKCkpLAogICAgICAgICAgICAgICAgICAgICAibGFtX2hhdC"
           "I6IGRbIkNsYWltTmIiXS5zdW0oKSAvIGRbIkV4cG9zdXJlIl0uc3VtKCksICJs"
           "bzk1IjogbG8sICJoaTk1IjogaGksCiAgICAgICAgICAgICAgICAgICAgICJzZS"
           "I6IGZsb2F0KHMuc3RkKGRkb2Y9MSkpfSkgICAgICAgICAgICAgICAgICAgIyDr"
           "toDtirjsiqTtirjrnqkg67aE7Y+s7J2YIO2RnOykgO2OuOywqAoKICAgIGJvb3"
           "RfY2kgPSBwZC5EYXRhRnJhbWUocm93cykKICAgIGJvb3RfY2lbIu2PrSJdID0g"
           "Ym9vdF9jaVsiaGk5NSJdIC0gYm9vdF9jaVsibG85NSJdCiAgICBib290X2NpWy"
           "Lsg4HrjIDtj60iXSA9IGJvb3RfY2lbIu2PrSJdIC8gYm9vdF9jaVsibGFtX2hh"
           "dCJdCiAgICByID0gYm9vdF9jaS5zZXRfaW5kZXgoIuyEuOq3uOuovO2KuCIpCi"
           "AgICBwcmludChmIuyngOyXrSBSNDMgOTUlIOq1rOqwhCA6IHtyLmxvY1snUmVn"
           "aW9uIFI0MycsICdsbzk1J106LjVmfSB+ICIKICAgICAgICAgIGYie3IubG9jWy"
           "dSZWdpb24gUjQzJywgJ2hpOTUnXTouNWZ9IOqxtC/rhYQiKQogICAgcHJpbnQo"
           "ZiLsg4HrjIAg7Y+tIDogQXJlYSBDIHtyLmxvY1snQXJlYSBDJywgJ+yDgeuMgO"
           "2PrSddOi40Zn0g64yAICIKICAgICAgICAgIGYiUmVnaW9uIFI0MyB7ci5sb2Nb"
           "J1JlZ2lvbiBSNDMnLCAn7IOB64yA7Y+tJ106LjRmfSIpCiAgICBwcmludChmIl"
           "I0MyDrtoDtirjsiqTtirjrnqkg7ZGc7KSA7Jik7LCoID0ge3IubG9jWydSZWdp"
           "b24gUjQzJywgJ3NlJ106LjVmfSDqsbQv64WEIikKCiAgICB5cG9zID0gbnAuYX"
           "JhbmdlKGxlbihib290X2NpKSlbOjotMV0gICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAgIyDsnITsl5DshJwg7JWE656Y66GcIOyEuOq3uOuovO2KuCA46rCcCi"
           "AgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XLCA0LjAp"
           "KQogICAgYXguaGxpbmVzKHlwb3MsIGJvb3RfY2lbImxvOTUiXSwgYm9vdF9jaV"
           "siaGk5NSJdLCBjb2xvcj1DT0xbIu2VmOuKmOyDiSJdLCBsdz00KQogICAgYXgu"
           "cGxvdChib290X2NpWyJsYW1faGF0Il0sIHlwb3MsICJvIiwgY29sb3I9Q09MWy"
           "Lrgqjsg4kiXSkgICAgICAgIyDsoJDstpTsoJUKICAgIGF4LmF4dmxpbmUoTEFN"
           "X0FMTCwgY29sb3I9Q09MWyLtmozsg4kiXSwgbHM9IjoiKSAgICAgICAgICAgIC"
           "AgICAgICMg7KCE7LK0IOqzhOyVvSDquLDspIDshKAKICAgIGF4LnNldF95dGlj"
           "a3MoeXBvcykKICAgIGF4LnNldF95dGlja2xhYmVscyhib290X2NpWyLshLjqt7"
           "jrqLztirgiXSkKICAgIGF4LnNldF94bGFiZWwoIuyXsOqwhCDsgqzqs6DsnKgg"
           "KOqxtC/rhYQpIikKICAgIGF4LnNldF95bGFiZWwoIuyalOycqO2RnOydmCDshL"
           "jqt7jrqLztirgiKQogICAgcGx0LnNob3coKQoK7ZW07ISdCgotIOyYiOyDgSDs"
           "p4jrrLjsnZgg64u17J2AIOuRkCDrsojsp7gg44CM7Je0IOuwsCDslYjtjI7jgI"
           "0g7J207JeI7Iq164uI64ukIOKAlCDslYTrnpjsl5DshJwg6re4IOuwsOyImOul"
           "vCDsiKvsnpDroZwg7ZmV7J247ZW0IOuztOyEuOyalC4KLSDsp4Dsl60gUjQzIO"
           "ydmCA5NSUg6rWs6rCE7J2AIDAuMDc2MTQgfiAwLjEyNzc5IOqxtC/rhYTsnoXr"
           "i4jri6QuIOygkOy2lOyglSAwLjEwMTEwIOydgCDsoITssrQg6rOE7JW9IOyCrO"
           "qzoOycqCAwLjEwMDYxNCDsmYAg6rGw7J2YIOqwmeyVhCDrs7TsnbTsp4Drp4ws"
           "IOq1rOqwhOydmCDslYTrnpgg64Gd6rO8IOychCDrgZ3snYAg7JqU7Jyo66GcIO"
           "2ZmOyCsO2VmOuptCAxLjfrsLAg7LCo7J207J6F64uI64ukIOKAlCDsnbQg7IS4"
           "6re466i87Yq47JeQIO2Pieq3oCDsmpTsnKjsnYQg66ek6ri47KeAIO2VoOymne"
           "ydhCDrp6TquLjsp4DripQg7J20IOuNsOydtO2EsOunjOycvOuhnCDsoJXtlaAg"
           "7IiYIOyXhuyKteuLiOuLpC4KLSDqtazqsIQg6ri47J2066W8IOyEuOq3uOuovO"
           "2KuOuBvOumrCDruYTqtZDtlZjroKTrqbQg7KCQ7LaU7KCV7Jy866GcIOuCmOuI"
           "iCDsg4HrjIAg6rWs6rCEIOq4uOydtOulvCDsgqzsmqntlanri4jri6QuIEFyZW"
           "EgQyDripQgMC4wMzkzLCBSNDMg7J2AIDAuNTEwOSDroZwg7JW9IDEzLjDrsLAg"
           "7LCo7J207J6F64uI64ukLiDqt7gg7LCo7J2066W8IOunjOuToCDqsoPsnYAg6r"
           "OE7JW9IOyImOqwgCDslYTri4jrnbwg64iE7KCB65CcIOyCrOqzoCDqsbTsiJjs"
           "noXri4jri6Qg4oCUIEFyZWEgQyA5LDg3NeqxtCDrjIAgUjQzIDU36rG07J206r"
           "OgLCDsnbTroaAgMS4g7ZGc7KSA7Jik7LCo7J2YIOyLnSBTRSjOu8yCKSA9IOKI"
           "mijsgqzqs6Ag7ZWpKSDDtyDrhbjstpwg7ZWp7J2EIOygkOy2lOygleycvOuhnC"
           "DrgpjriITrqbQg7IOB64yAIO2RnOykgOyYpOywqOqwgCAxIMO3IOKImijsgqzq"
           "s6Ag7ZWpKSDsnbQg65CY66+A66GcLCDiiJooOTg3NSAvIDU3KSDiiYggMTMuMi"
           "DrsLDqsIAg6re464yA66GcIOyDgeuMgCDqtazqsIQg6ri47J207J2YIOuwsOyI"
           "mOuhnCDrgpjtg4Drgqnri4jri6QuCi0gUjQzIOydmCDrtoDtirjsiqTtirjrnq"
           "kg7ZGc7KSA7Jik7LCoIDAuMDEzNDUg64qUIO2PrOyVhOyGoSDsnbTroaDqsJIg"
           "4oiaNTcgw7cgNTYzLjc3ID0gMC4wMTMzOSDsmYAg7IaM7IiY7KCQIOyVhOuemC"
           "DrhLfsp7gg7J6Q66as6rmM7KeAIOydvOy5mO2VqeuLiOuLpC4g6rCA7KCV7J2E"
           "IOyCrOyaqe2VmOyngCDslYrripQg67Cp67KVKOyerO2RnOynkSnqs7wg7IKs7J"
           "qp7ZWY64qUIOuwqeuylSjtj6zslYTshqEg6rO17IudKeydtCDqsJnsnYAg6rCS"
           "7J2EIOyCsOy2nO2WiOyKteuLiOuLpC4g7J20IOuNsOydtO2EsOyXkOyEnOuKlC"
           "Dtj6zslYTshqEg6rCA7KCV7J20IOq3uOunjO2BvCDsnpgg66ee64qU64uk64qU"
           "IOucu+yeheuLiOuLpC4KLSDqtazqsITsnbQg7KCE7LK0IOqzhOyVvSDsgqzqs6"
           "DsnKjsnYQg7Y+s7ZWo7ZaI64uk64qUIOqyg+ydgCDjgIzsoITssrQg6rOE7JW9"
           "IO2Pieq3oOqzvCDri6TrpbTri6Tqs6Ag66eQ7ZWgIOq3vOqxsOqwgCDsnbQg64"
           "2w7J207YSw7JeQIOyXhuuLpOOAjeuKlCDrnLvsnbTsp4Ag44CM6rCZ64uk44CN"
           "64qUIOucu+ydtCDslYTri5nri4jri6QuIFI0MyDsspjrn7wg6rOE7JW97J20IO"
           "yggeydgCDshLjqt7jrqLztirjripQg6re4656Y7IScIDLqsIDsp4Ag67Cp67KV"
           "IOykkSDtlZjrgpjrpbwg7ISg7YOd7ZW07JW8IO2VqeuLiOuLpC4g7IS46re466"
           "i87Yq4IOuwlOq5peydmCDsoJXrs7Trpbwg67mM66CkIOyYpOqxsOuCmCjsiJjs"
           "tpXCt+uqqO2YlSDsmIjsuKEpLCDshLjqt7jrqLztirjrpbwg642UIOuEk+qyjC"
           "DrrLbslrQg7IKs6rOgIOqxtOyImOulvCDriITsoIHtlZjripQg6rKD7J6F64uI"
           "64ukLgoK7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp7IudIOKAlCDshL"
           "jqt7jrqLztirjrp4jri6Qg6re4IOyEuOq3uOuovO2KuOydmCDqs4Tslb3rp4wg"
           "67O17JuQ7LaU7Lac66GcIO2RnOuzuCAxLDAwMOqwnOulvCDsnqzstpTstpztlb"
           "Qg7ZGc67O466eI64ukIOOAjOyCrOqzoCDqsbTsiJgg7ZWpIMO3IOuFuOy2nCDt"
           "lanjgI3snYQg6rOE7IKw7ZWY6rOgLCDqt7ggMSwwMDDqsJzrpbwg7YGs6riw7I"
           "ic7Jy866GcIOygleugrO2VtCDslYTrnpggMi41JSDsp4DsoJDqs7wg7JyEIDIu"
           "NSUg7KeA7KCQ7JeQ7IScIOygiOuLqO2VtCDqtazqsITsnZgg65GQIOuBneycvO"
           "uhnCDsgrzsirXri4jri6QuCg==",
           "m2":
           "8J+UkyDrr7jshZggMiDsmYTshLHrs7gg4oCUIOOAjEFJIOy9lOuTnCDsi5zsnp"
           "Eg7ZaJ44CNIOyVhOuemOyXkCDrk6TslrTsmZTslrTslbwg7ZWY64qUIOy9lOuT"
           "nOuKlCDsnbQg7KCV64+E7J6F64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrto"
           "Drpbgg7ZSE66Gs7ZSE7Yq4IOyYiAoKICAgIOyghOyehCDri7Tri7nsnpAg7L2U"
           "65Oc7JeQ7IScIOqzoOy5oCDqs7PsnYAgMuqwnOyVvC4g7LKr7Ke4LCDqs4TsiJ"
           "gg6rK966Gc66W8IOybkOuemCDri6jsnIQg6re464yA66Gc7J24IFhkZXNpZ24g"
           "7Jy866GcIOygge2Vqe2WiOuKlOuNsCBCb251c01hbHVzIOyymOufvCDqsJLsnZ"
           "gg67KU7JyE6rCAIO2BsCDsl7Tqs7wgMC8xIOuNlOuvuCDsl7TsnbQKICAgIOqw"
           "meydgCDtjKjrhJDti7Ag7ZWt7J2EIOuwm+ycvOuptCDqs7Xtj4ntlZjsp4Ag7J"
           "WK7Jy864uIIO2RnOykgO2ZlO2VnCBYeiDroZwg67CU6r+UIOykmC4g65GY7Ke4"
           "LCDtjKjrhJDti7Ag7ZWt7J2YIOyEuOq4sOulvCDqs6Drpbwg6re86rGw6rCAIO"
           "yVhOyYiCDsl4bsnLzri4ggQUxQSEFTIDbqsJzrp4jri6QgNS3tj7Trk5wg6rWQ"
           "7LCo6rKA7Kad7J2EIOyLpO2Wie2VtCDspJgg4oCUIOu2hO2VoOydgAogICAgU0"
           "VFRCDroZwg7ISe7J2AIEtGb2xkIO2VmOuCmOulvCDrp4zrk6TslrQgYWxwaGEg"
           "NuqwnOqwgCDqsJnsnYAg67aE7ZWg7J2EIO2VqOq7mCDsk7Dqs6AsIO2VmeyKtS"
           "Dtj7Trk5zsl5DshJzrp4wgUG9pc3NvblJlZ3Jlc3NvciDrpbwg7KCB7ZWp7ZWc"
           "IOuSpCDqsoDspp0g7Y+065Oc7JeQ7IScIOuFuOy2nCDqsIDspJEKICAgIHdfZX"
           "hwIOulvCDspIAg7Y+J6regIO2PrOyVhOyGoSDsnbTtg4jrj4Trpbwg7J6s6rOg"
           "LCDtj7Trk5wgNeqwnCDsoJDsiJjsnZgg7Y+J6reg6rO8IO2RnOykgOyYpOywqC"
           "jtkZzspIDtjrjssKjrpbwgNeydmCDsoJzqs7Hqt7zsnLzroZwg64KY64iIIOqw"
           "kinrpbwgYWxwaGEg66eI64ukIO2VnCDtlonsnLzroZwg66qo7Jy866m0IOuPvC"
           "jtg4DquYPsnYAKICAgIHlfcmF0ZSwg6rCA7KSR7J2AIHdfZXhwKS4g6re4IOuL"
           "pOydjCDtjKjrhJDti7Ag7ZWt7J20IOqwgOyepSDslb3tlZwgYWxwaGEg7JeQ7I"
           "Sc7J2YIOq1kOywqOqygOymnSDsnbTtg4jrj4Trpbwg7ZWcIO2WiSwg6rCA7J6l"
           "IOuCruydgCDqtZDssKjqsoDspp0g7J207YOI64+E66W8IOyWtOuKkCBhbHBoYS"
           "Dsnbjsp4DsmYAg7ZWo6ruYIOyGjOyImOygkAogICAg7JWE656YIDbsnpDrpqzr"
           "oZwg7ZWcIO2WiSwg6re4IOqwgOyepSDslb3tlZwgYWxwaGEg7JeQ7ISc7J2YIO"
           "2RnOykgO2ZlCDqs4TsiJggNDDqsJzsnZgg7KCI64yT6rCSIO2VqeydhCAi7ZGc"
           "7KSA7ZmUIOqzhOyImCA0MOqwnOydmCDtgazquLAg7ZWpID0gKOyIq+yekCkiIO"
           "uhnCDshozsiJjsoJAg7JWE656YIDTsnpDrpqzquYzsp4Ag7ZWcIO2WiSBwcmlu"
           "dAogICAg7ZW0IOykmC4g6re466a87J2AIOuCmOuegO2VnCDrkZAg7Yyo64SQ66"
           "GcIOq3uOugpCDspJgg4oCUIOyZvOyqveydgCDqsIDroZzstpXsnYQgYWxwaGEg"
           "66GcIOuGk+qzoCDroZzqt7gg64iI6riI7J2EIOyTsOqzoCBDT0xTIDQw6rCc7J"
           "2YIOqzhOyImCDqsr3roZzrpbwg7JiF7J2AIOyEoOycvOuhnCDqsrnss5Ag6re4"
           "66awIOuSpAogICAgQm9udXNNYWx1cyDsmYAgVmVoQnJhbmRfQjEyIOuRkCDshK"
           "Drp4wg6rW16rKMIOuNp+q3uOumrOqzoCAwIOyXkCDqsIDroZwg7KCQ7ISg7J2E"
           "IOq4i+qzoCwg7Jik66W47Kq97J2AIGFscGhhIDbqsJzrpbwg6rCA66Gc7LaV7J"
           "eQIOuGk+qzoCDqtZDssKjqsoDspp0g7J207YOI64+E66W8IOunieuMgCA26rCc"
           "66GcCiAgICDqt7jrprAg65KkIO2RnOykgOyYpOywqOulvCDsmKTssKjrp4nrjI"
           "DroZwg7ZGc7Iuc7ZW0IOykmC4g65GQIO2MqOuEkCDrqqjrkZAg65GQIOy2leyX"
           "kCDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICBrZiA9IEtGb2xkKDUsIHNodW"
           "ZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPVNFRUQpICAgICAgICAgICMg67aE7ZWg"
           "7J2AIO2VnCDrsojrp4wg4oCUIGFscGhhIDbqsJzqsIAg7ZWo6ruYIOyUgeuLiO"
           "uLpAoKICAgIGNvZWZfcGF0aCwgcm93cyA9IHt9LCBbXQogICAgZm9yIGEgaW4g"
           "QUxQSEFTOgogICAgICAgIGNvZWZfcGF0aFthXSA9IChQb2lzc29uUmVncmVzc2"
           "9yKGFscGhhPWEsIG1heF9pdGVyPTMwMCwgdG9sPTFlLTYpCiAgICAgICAgICAg"
           "ICAgICAgICAgICAgIC5maXQoWHosIHlfcmF0ZSwgc2FtcGxlX3dlaWdodD13X2"
           "V4cCkuY29lZl8pICAgIyBYZGVzaWduIOydtCDslYTri4jrnbwgWHog4oCUIO2V"
           "qOygleydtCDsnojrjZgg6rOzCiAgICAgICAgZm9sZF9zY29yZXMgPSBbXQogIC"
           "AgICAgIGZvciB0ciwgdGUgaW4ga2Yuc3BsaXQoWHopOgogICAgICAgICAgICBt"
           "ID0gUG9pc3NvblJlZ3Jlc3NvcihhbHBoYT1hLCBtYXhfaXRlcj0zMDAsIHRvbD"
           "0xZS02KS5maXQoCiAgICAgICAgICAgICAgICBYelt0cl0sIHlfcmF0ZVt0cl0s"
           "IHNhbXBsZV93ZWlnaHQ9d19leHBbdHJdKSAgICAgICAgICAgIyDtlZnsirUg7Y"
           "+065Oc7JeQ7ISc66eMIOygge2Vqe2VmOqzoAogICAgICAgICAgICBmb2xkX3Nj"
           "b3Jlcy5hcHBlbmQobWVhbl9wb2lzc29uX2RldmlhbmNlKHlfcmF0ZVt0ZV0sIG"
           "0ucHJlZGljdChYelt0ZV0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAgICAgICAgICAgICAgc2FtcGxlX3dlaWdodD13X2V4cFt0ZV0pKSAgIy"
           "DsoJDsiJjripQg6rKA7KadIO2PtOuTnOyXkOyEnCDsgrDstpztlanri4jri6QK"
           "ICAgICAgICBmb2xkX3Njb3JlcyA9IG5wLmFycmF5KGZvbGRfc2NvcmVzKQogIC"
           "AgICAgIHJvd3MuYXBwZW5kKHsiYWxwaGEiOiBhLCAiY3YiOiBmb2xkX3Njb3Jl"
           "cy5tZWFuKCksICJzZSI6IGZvbGRfc2NvcmVzLnN0ZChkZG9mPTEpIC8gbnAuc3"
           "FydCg1KX0pCgogICAgY29lZl9wYXRoX3RibCA9IHBkLkRhdGFGcmFtZShjb2Vm"
           "X3BhdGgsIGluZGV4PUNPTFMpCiAgICBjdl9hbHBoYSA9IHBkLkRhdGFGcmFtZS"
           "hyb3dzKQogICAgYmVzdCA9IGN2X2FscGhhLmxvY1tjdl9hbHBoYVsiY3YiXS5p"
           "ZHhtaW4oKV0KCiAgICBwcmludChmIuuyjOygkCDsl4bripQg6rWQ7LCo6rKA7K"
           "adIOydtO2DiOuPhCA9IHtjdl9hbHBoYS5sb2NbMCwgJ2N2J106LjZmfSAgKGFs"
           "cGhhID0ge0FMUEhBU1swXTpnfSkiKQogICAgcHJpbnQoZiLstZzsoIAg6rWQ7L"
           "Co6rKA7KadIOydtO2DiOuPhCA9IHtiZXN0WydjdiddOi42Zn0gIChhbHBoYSA9"
           "IHtiZXN0WydhbHBoYSddOmd9KSIpCiAgICBwcmludChmIu2RnOykgO2ZlCDqs4"
           "TsiJggNDDqsJzsnZgg7YGs6riwIO2VqSA9IHtjb2VmX3BhdGhfdGJsW0FMUEhB"
           "U1swXV0uYWJzKCkuc3VtKCk6LjRmfSAgKGFscGhhID0ge0FMUEhBU1swXTpnfS"
           "kiKQoKICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygxLCAyLCBmaWdzaXpl"
           "PShGSUdfVywgMy44KSkKICAgIGZvciBjIGluIENPTFM6ICAgICAgICAgICAgIC"
           "AgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIO2ajOyD"
           "iSDsi6TshKAgNDDqsJwgPSDqs4TsiJgg7ZWY64KY7JSpCiAgICAgICAgYXhlc1"
           "swXS5wbG90KEFMUEhBUywgY29lZl9wYXRoX3RibC5sb2NbY10sIGNvbG9yPUNP"
           "TFsi7Jew7ZqM7IOJIl0sIGx3PTEpCiAgICBmb3IgYyBpbiBbIkJvbnVzTWFsdX"
           "MiLCAiVmVoQnJhbmRfQjEyIl06ICAgICAgICAgICAgICAgICAgICAgICAgICAg"
           "IyDtlZjripjsg4kg6rW17J2AIOyEoCAy6rCc66eMIOuUsOudvOqwkeuLiOuLpA"
           "ogICAgICAgIGF4ZXNbMF0ucGxvdChBTFBIQVMsIGNvZWZfcGF0aF90YmwubG9j"
           "W2NdLCBjb2xvcj1DT0xbIu2VmOuKmOyDiSJdLCBsdz0yKQogICAgYXhlc1swXS"
           "5zZXRfeHNjYWxlKCJsb2ciKQogICAgYXhlc1swXS5heGhsaW5lKDAsIGNvbG9y"
           "PUNPTFsi7ZqM7IOJIl0sIGxzPSI6IikKICAgIGF4ZXNbMF0uc2V0X3hsYWJlbC"
           "gi7KCV6rec7ZmUIOqwleuPhCBhbHBoYSIpCiAgICBheGVzWzBdLnNldF95bGFi"
           "ZWwoIu2RnOykgO2ZlCDqs4TsiJgiKQogICAgYXhlc1sxXS5iYXIoW2Yie2E6Z3"
           "0iIGZvciBhIGluIEFMUEhBU10sIGN2X2FscGhhWyJjdiJdLCB5ZXJyPWN2X2Fs"
           "cGhhWyJzZSJdLAogICAgICAgICAgICAgICAgY29sb3I9Q09MWyLtlZjripjsg4"
           "kiXSwgZWNvbG9yPUNPTFsi64Ko7IOJIl0sIGNhcHNpemU9MykKICAgIGF4ZXNb"
           "MV0uc2V0X3hsYWJlbCgi7KCV6rec7ZmUIOqwleuPhCBhbHBoYSIpCiAgICBheG"
           "VzWzFdLnNldF95bGFiZWwoIuq1kOywqOqygOymnSDtj6zslYTshqEg7J207YOI"
           "64+EIikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSDsmIjsg4Eg7KeI66y47J"
           "2YIOuLteydgCDssqsg67KI7Ke4LCDjgIxmaXQoLi4uKSDsnZgg7LKrIOyduOye"
           "kOqwgCDsm5Drnpgg64uo7JyEIOq3uOuMgOuhnOyduCDshKTqs4TtlonroKzsnb"
           "TrnbzjgI0g7JiA7Iq164uI64ukLiDtlajqu5gg7KCB7Ja0IOuRlAogIOydtO2D"
           "iOuPhOydmCDrs4DtmZTripQg44CM6rGw7J2YIOq3uOuMgOuhnOuLpCDigJQg64"
           "2w7J207YSw6rCAIOqzhOyImCDqsJzsiJjsl5Ag67mE7ZW0IOy2qeu2hO2VtCDt"
           "jKjrhJDti7Ag7ZWt7Jy866GcIOykhOydvCDqsoPsnbQg67OE66GcIOyXhuuLpO"
           "OAjSDsqr3snoXri4jri6QuCiAg7JWE656YIOyIq+yekOuhnCDtmZXsnbjtlbQg"
           "67O07IS47JqULgotIO2VqOygleydgCDsoIHtlansl5Ag7KCE64us7ZWcIO2Wie"
           "ugrOydtOyXiOyKteuLiOuLpC4g44CMQUkg7L2U65OcIOyLnOyekSDtlonjgI0g"
           "7JWE656YIOy9lOuTnOuKlCDtkZzspIDtmZTtlZjsp4Ag7JWK7J2AIFhkZXNpZ2"
           "4g7J2EIOq3uOuMgOuhnCDsoITri6ztlojsirXri4jri6QuIO2MqOuEkO2LsCDt"
           "la3snYAKICDqs4TsiJjsnZgg7YGs6riw7JeQIOu2gOqzvOuQmOuKlOuNsCBCb2"
           "51c01hbHVzIOuKlCA1MO+9njIzMCwg642U66+4IOyXtOydgCAwIOyVhOuLiOup"
           "tCAxIOydtOudvCDri6jsnITqsIAg7Je066eI64ukIOuLpOumheuLiOuLpC4g6r"
           "CS7J2YIOuylOychOqwgAogIO2BsCDsl7TsnYAg6rCZ7J2AIO2aqOqzvOulvCDr"
           "grTripQg642wIO2VhOyalO2VnCDqs4TsiJjqsIAg7JWg7LSI7JeQIOyekeyVhO"
           "yEnCDtjKjrhJDti7Ag7ZWt7J2YIOyYge2WpeydhCDqsbDsnZgg67Cb7KeAIOyV"
           "iuqzoCwg6rCS7J2YIOuylOychOqwgCDsnpHsnYAg642U66+4IOyXtOunjCDtga"
           "zqsowKICDstpXshozrkKnri4jri6QuIOq3uOuemOyEnCDqt7gg7L2U65Oc7J2Y"
           "IOqzhOyImCDtgazquLAg7ZWp7J2AIDTsl5Ag6rCA6rmM7Jq0IOqwkuyXkOyEnC"
           "Dsi5zsnpHtlZjripTrjbAsIOq3uCA064qUICLqs4TsiJjqsIAg7YGs64ukIuqw"
           "gCDslYTri4jrnbwKICAi7Je066eI64ukIOuLqOychOqwgCDri6TrpbTri6Qi64"
           "qUIOucu+yeheuLiOuLpC4g44CM7KSA67mEIOyFgOOAjeydtCBYeiDrpbwg65Sw"
           "66GcIOyDneyEse2VtCDrkZQg7J207Jyg6rCAIOydtOqyg+yeheuLiOuLpC4KLS"
           "Dqs4TsiJjripQg7ZmV7Iuk7Z6IIOy2leyGjOuQqeuLiOuLpC4g7ZGc7KSA7ZmU"
           "7ZWcIFh6IOuhnCDri6Tsi5wg6rOE7IKw7ZWY66m0IO2RnOykgO2ZlCDqs4TsiJ"
           "ggNDDqsJzsnZgg7YGs6riwIO2VqeydtCBhbHBoYSA9IDFlLTA2IOyXkOyEnCAx"
           "LjA4ODAsCiAgYWxwaGEgPSAxLjAg7JeQ7IScIDAuMTI5OSDigJQgOOu2hOydmC"
           "Ax7J20IOuQqeuLiOuLpC4g7Jm87Kq9IOq3uOumvOyXkOyEnCDtmozsg4kg7ISg"
           "IDQw6rCc6rCAIOyYpOuluOyqveycvOuhnCDqsIjsiJjroZ0gMCDsnLzroZwKIC"
           "DsiJjroLTtlZjripQg6rKD7J20IOydtCDsiKvsnpDsnoXri4jri6QuIO2VmOuK"
           "mOyDiSDqtbXsnYAg7ISgIDLqsJwg7KSRIOychOqwgCBCb251c01hbHVzKDAuMz"
           "M5MiDihpIgMC4wMzI0KSwg7JWE656Y6rCACiAgVmVoQnJhbmRfQjEyKDAuMTYz"
           "MiDihpIgMC4wMTM0KSDsnoXri4jri6QuCi0g6re465+w642wIOyYiOy4oeydgC"
           "DrgpjslYTsp4Dsp4Ag7JWK7Iq164uI64ukLiDqtZDssKjqsoDspp0g7J207YOI"
           "64+E64qUIGFscGhhID0gMWUtMDYg7JeQ7IScIDAuNjA4MjQ3LCDstZzsoIDsoJ"
           "DsnbgKICBhbHBoYSA9IDAuMDAxIOyXkOyEnCAwLjYwODI0MSDigJQg6rCc7ISg"
           "7Y+t7J20IDAuMDAwMDA2IOydtOqzoCwg7J2064qUIO2PtOuTnCDqsIQg7ZGc7K"
           "SA7Jik7LCoIDAuMDAyNjE1IOydmCA0MDDrtoTsnZggMQogIOyeheuLiOuLpC4g"
           "7Jik66W47Kq9IOunieuMgCA26rCc7J2YIOuGkuydtCDssKjsnbTqsIAg67O07J"
           "207KeAIOyViuuKlCDqsoPsnbQg6re4IOqwnOyEoO2PreyeheuLiOuLpC4g6re4"
           "65+s64ukIGFscGhhID0gMC4xIOyXkOyEnAogIDAuNjExOTgxLCBhbHBoYSA9ID"
           "EuMCDsl5DshJwgMC42MjE0MTIg66GcIOyYpO2eiOugpCDspp3qsIDtlanri4jr"
           "i6Qg4oCUIO2MqOuEkO2LsCDtla3snbQg6rCV7ZW07KC4IO2VhOyalO2VnCDsi6"
           "DtmLjquYzsp4Ag7LaV7IaM7ZaI64uk64qUCiAg65y77J6F64uI64ukLgotIOq3"
           "uCDsnbTsnKDripQg64uk7J2M6rO8IOqwmeyKteuLiOuLpC4g6rSA7Lih7J20ID"
           "Y3OCwwMTPqsbTsnbjrjbAg6rOE7IiY6rCAIDQx6rCc67+Q7J206528IO2MqOuE"
           "kO2LsCDtla3snLzroZwg7KSE7J28IOqzhOyImCDtgazquLDqsIAg7JWg7LSI7J"
           "eQIOqxsOydmCDsl4bsirXri4jri6QuIOygleq3nO2ZlOuKlCDtjrjtlqXsnYQK"
           "ICDstpTqsIDtlZjripQg64yA7IugIOqzhOyImOydmCDrtoTsgrDsnYQg7KSE7J"
           "2064qUIOygiOy2qeyduOuNsCwg6rOE7IiYIO2VmOuCmOuLuSDqtIDsuKEg7IiY"
           "6rCAIOydtOunjO2BvCDrp47snLzrqbQg7KSE7J28IOu2hOyCsOydtCDrgqjslY"
           "Qg7J6I7KeACiAg7JWK7Iq164uI64ukLgotIOq3uOuemOuPhCDsnbQg7Iuk7ZeY"
           "7J20IOuRmOynuCDrgqDqs7wg7J207Ja07KeA64qUIOu2gOu2hOydtCDsnojsir"
           "Xri4jri6QuIFJlZ2lvbl9SNDMg7J2YIOqzhOyImOuKlCBhbHBoYSA9IDFlLTA2"
           "IOyXkOyEnCAtMC4wMDQ0IOydtOuNmAogIOqyg+ydtCBhbHBoYSA9IDEuMCDsl5"
           "DshJwgLTAuMDAwMCDsnbQg65Cp64uI64ukLiDqs4TsiJjqsIAgMCDsl5Ag64u/"
           "64qU64uk64qUIOqyg+ydgCDqt7gg7KeA7Jet7J20IOyghOyytCDqs4Tslb0g7Y"
           "+J6reg6rO8IOqwmeydgCDsmpTsnKjsnYQKICDrsJvripTri6TripQg65y77J20"
           "6rOgLCDrkZjsp7gg64KgIOqwkOuniCDsgqzsoITrtoTtj6zroZwg7ZWY642YIO"
           "yImOy2leqzvCDsoJXtmZXtnogg6rCZ7J2AIOydvOyeheuLiOuLpC4gTDIg7Yyo"
           "64SQ7YuwIO2VreydgCDqs4TsiJjsl5Ag67aA7Jes7ZWcIO2Pieq3oCAwIOyduA"
           "ogIOygleq3nCDsgqzsoITrtoTtj6zsnbTqs6AsIOq3uOuemOyEnCBSaWRnZSDs"
           "tpTsoJXsnYAg6re4IOyCrOyghOu2hO2PrCDslYTrnpjsnZggTUFQIOyeheuLiO"
           "uLpC4KLSDsoJXqt5ztmZTqsIAg7Zqo6rO866W8IOuCtOuKlCDsobDqsbTsnYAg"
           "6rOE7IiYIOqwnOyImOyXkCDruYTtlbQg642w7J207YSw6rCAIOyggeydhCDrlY"
           "zsnoXri4jri6QuIOqwmeydgCDqs4TsiJggNDHqsJzrpbwg6rOE7JW9CiAgNSww"
           "MDDqsbQo7IKs6rOgIDI0NeqxtCnrp4zsnLzroZwg7LaU7KCV7ZWY66m0IOq1kO"
           "ywqOqygOymnSDsnbTtg4jrj4TqsIAg7Yyo64SQ7YuwIO2VrSDsl4bsnbQgMC42"
           "MTY5LCBhbHBoYSA9IDAuMyDsl5DshJwgMC41ODQxIOuhnAogIDAuMDMyOCDqsJ"
           "Dshoztlanri4jri6Qg4oCUIOuwqeq4iCDrs7gg6rCc7ISg7Y+t7J2YIDUsMDAw"
           "67Cw7J6F64uI64ukLiDsi6Dqt5wg7KeE7LacIOyngOyXreydmCDssqvtlbQg7Y"
           "+s7Yq47Y+066as7Jik6rCAIOydtCDtgazquLDsnoXri4jri6QuCgrsvZTrk5zq"
           "sIAg64u17J2EIOunjOuTnOuKlCDrsKnsi50g4oCUIOydtCDsvZTrk5zsl5Drip"
           "Qg7YyQ64uo7J20IOuRkCDrsogg65Ok7Ja06rCR64uI64ukLiDtlZjrgpjripQg"
           "66y07JeH7J2EIOqwmeydgCDri6jsnITroZwg66ee7LacIOqyg+yduOqwgAoo7Z"
           "Gc7KSA7ZmU66W8IOuovOyggCDtlZjripTqsIAp7J206rOgLCDri6Trpbgg7ZWY"
           "64KY64qUIOustOyXh+ycvOuhnCBhbHBoYSDrpbwg6rOg66W8IOqyg+yduOqwgC"
           "jtm4jroKgg7KCQ7IiY6rCAIOyVhOuLiOudvCDqsoDspp0g7KCQ7IiYKeyeheuL"
           "iOuLpC4KMuqwgOyngCDrqqjrkZAg7YuA66Ck64+EIOyYpOulmOqwgCDrsJzsg5"
           "3tlZjsp4Ag7JWK6rOgIOyIq+yekOuKlCDsgrDstpzrkKnri4jri6QuIOq3uOue"
           "mOyEnCDsoJXqt5ztmZQg7L2U65Oc66W8IOuwm+yVmOydhCDrlYwg7IKs656M7J"
           "20IO2ZleyduO2VoCDtla3rqqnrj4Qg7ZWt7IOBIOydtCAy6rCc7J6F64uI64uk"
           "CuKAlCDsoIHtlansl5Ag7KCE64us7ZWcIO2WieugrOydtCDri6jsnITrpbwg66"
           "ee7LaYIOqyg+yduOqwgCwg6re466as6rOgIOygkOyImOulvCDsgrDstpztlZwg"
           "642w7J207YSw6rCAIOygge2VqeyXkCDsgqzsmqnrkJjsp4Ag7JWK7J2AIOuNsO"
           "ydtO2EsOyduOqwgC4K",
           "m3":
           "8J+UkyDrr7jshZggMyDsmYTshLHrs7gg4oCUIOOAjEFJIOy9lOuTnCDsi5zsnp"
           "Eg7ZaJ44CNIOyVhOuemOyXkCDrk6TslrTsmYDslbwg7ZWgIOy9lOuTnOuKlCDs"
           "nbQg7KCV64+E7J6F64uI64ukCgrilqAg7J20IOy9lOuTnOulvCDrtoDrpbgg7Z"
           "SE66Gs7ZSE7Yq4IOyYiAoKICAgIOydtOuvuCDrp4zrk6TslrTsoLgg7J6I64qU"
           "IGRmLCBGT0xEUywgRk9STVVMQVMsIHlfcmF0ZSwgd19leHAsIExBTV9BTEwg7J"
           "2EIOq3uOuMgOuhnCDsk7Dqs6Ag64uk7IucIOunjOuTpOyngOuKlCDrp5DslYQg"
           "7KSYIOKAlCB5X3JhdGUg7JmAIHdfZXhwIOuKlAogICAgZGYg7JmAIOq4uOydtO"
           "qwgCDqsJnsnYAg64SY7YyM7J20IOuwsOyXtOydtOudvCDtj7Trk5zsnZgg7ZaJ"
           "IOuyiO2YuOuhnCDsnpjrnbwg7JOw66m0IOuPvC4gRk9STVVMQVMg7JeQIOuToC"
           "Dtm4Trs7QgM+qwnChudWxsIMK3IGNvcmUgwrcgZnVsbCnrpbwgRk9MRFMg7J2Y"
           "ICjtlZnsirUsIOqygOymnSkKICAgIO2WiSDrsojtmLggNeyMjSDsnITsl5DshJ"
           "wg7ZWY64KY7JSpIOyLpO2Wie2VtCDspJgg4oCUIOygge2VqeydgCBkZiDrpbwg"
           "6re4IO2VmeyKtSDtlokg67KI7Zi466GcIOyekOuluCDsobDqsIHsl5DshJzrp4"
           "wg7Y+s7JWE7IahIEdMTSDsnLzroZwg7ZWY6rOgIG9mZnNldCDsl5DripQg6re4"
           "IOyhsOqwgeydmCBFeHBvc3VyZSDsl7Tsl5AKICAgIOuhnOq3uOulvCDslIzsmr"
           "Qg6rCS7J2EIOyghOuLrO2VmOqzoCwg7KCQ7IiY64qUIOqygOymnSDsobDqsIHs"
           "l5DshJwg7Jik7ZSE7IWL7J2EIDAg7Jy866GcIOuRlCDsmIjsuKEo7Jew7JyoKe"
           "qzvCB5X3JhdGUg7J2YIOqygOymnSDsobDqsIHsnYQg67mE6rWQ7ZW0IOyCsOy2"
           "nO2VtCDspJguCiAgICDsnbTrlYwgd19leHAg7J2YIOqygOymnSDsobDqsIHsnY"
           "Qg6rCA7KSR7Jy866GcIOykgCDtj4nqt6Ag7Y+s7JWE7IahIOydtO2DiOuPhOul"
           "vCDsk7DrqbQg64+8LiDtm4Trs7Trp4jri6Qg44CMKOydtOumhCkg7Y+J6regID"
           "0gKOyIq+yekCkg7Y+065OcIOqwhCDtkZzspIDsmKTssKggPSAo7Iir7J6QKSDt"
           "j7Trk5zrs4QgKOyIq+yekCA16rCcKeOAjSDsspjrn7wg6rCS66eI64ukIOydtO"
           "umhOqzvCDrk7HtmLjrpbwg67aZ7J24IO2VnCDtlonsnYQg7IaM7IiY7KCQIOyV"
           "hOuemCA27J6Q66as66GcCiAgICBwcmludCDtlbQg7KSYIOKAlCDtkZzspIDsmK"
           "TssKjripQg7Y+065OcIDXqsJwg7KCQ7IiY7J2YIO2RnOykgO2OuOywqOulvCA1"
           "7J2YIOygnOqzseq3vOycvOuhnCDrgpjriIgg6rCS7J207JW8LiDqt7gg64uk7J"
           "2MIOqwmeydgCDtj7Trk5zsl5DshJwgY29yZSDsnZgg7J207YOI64+E7JeQ7ISc"
           "IGZ1bGwg7J2YIOydtO2DiOuPhOulvCDruoAg7Ked7KeA7J2ACiAgICDssKjsnb"
           "QgNeqwnOulvCDrp4zrk6TslrQg6re4IO2Pieq3oOqzvCDtkZzspIDsmKTssKjr"
           "pbwg7ZWcIO2WiSwgIu2Pieq3oCAvIO2RnOykgOyYpOywqCA9ICjsiKvsnpApIO"
           "uwsCIg66W8IOyGjOyImOygkCDslYTrnpggMuyekOumrOuhnCDtlZwg7ZaJIHBy"
           "aW50IO2VmOqzoCwg7Y+J6reg7J20IOqwgOyepSDrgq7snYAg7ZuE67O07J2YIO"
           "2Pieq3oOyXkCDqt7gKICAgIO2RnOykgOyYpOywqOulvCDrjZTtlZwgMS3tkZzs"
           "pIDsmKTssKgg7ISg6rO8IOq3uCDshKAg7JWI7JeQIOuTnOuKlCDtm4Trs7Qg66"
           "qp66Gd64+EIO2VnCDtlokg7Lac66Cl7ZW0IOykmC4g6re4656Y7ZSE64qUIOuC"
           "mOuegO2VnCDrkZAg7Yyo64SQ66GcIOKAlCDsmbzsqr3snYAg6rCA66Gc7LaV7J"
           "eQIG51bGwgwrcgY29yZSDCtyBmdWxsIOydhCDrhpPqs6AKICAgIO2bhOuztOun"
           "iOuLpCDtj7Trk5wg7KCQ7IiYIDXqsJzrpbwg7KCQ7Jy866GcIO2RnOyLnO2VmO"
           "qzoCDqt7gg7Y+J6reg7J2EIOq1teydgCDqsIDroZzshKDsnLzroZwg7ZGc7Iuc"
           "7ZWY6rOgLCDsmKTrpbjsqr3snYAg7Y+065OcIDEg67aA7YSwIDUg6rmM7KeA66"
           "W8IOqwgOuhnOy2leyXkCDrhpPqs6AgY29yZSDsl5DshJwgZnVsbCDsnYQg67qA"
           "IOywqOydtOulvCDrp4nrjIAKICAgIDXqsJzroZwg6re466awIOuSpCAwIOychO"
           "y5mOyZgCDtj4nqt6Ag7JyE7LmY7JeQIOqwgOuhnOyEoOydhCDquIvqs6AsIOuR"
           "kCDtjKjrhJAg66qo65GQIOuRkCDstpXsl5Ag652867Ko7J2EIOuLrOyVhCDspJ"
           "guCgogICAgY3ZfZGV2ID0ge30KICAgIGZvciBuYW1lLCBmb3JtdWxhIGluIEZP"
           "Uk1VTEFTLml0ZW1zKCk6CiAgICAgICAgZm9sZF9zY29yZXMgPSBbXQogICAgIC"
           "AgIGZvciB0ciwgdGUgaW4gRk9MRFM6ICAgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAgICAgICAgIyDtm4Trs7QgM+qwnOqwgCDqsJnsnYAg67aE7ZWg7J2EIO"
           "2VqOq7mCDsgqzsmqntlanri4jri6QKICAgICAgICAgICAgZHRyLCBkdGUgPSBk"
           "Zi5pbG9jW3RyXSwgZGYuaWxvY1t0ZV0KICAgICAgICAgICAgciA9IHNtZi5nbG"
           "0oZm9ybXVsYT1mb3JtdWxhLCBkYXRhPWR0ciwgZmFtaWx5PXNtLmZhbWlsaWVz"
           "LlBvaXNzb24oKSwKICAgICAgICAgICAgICAgICAgICAgICAgb2Zmc2V0PW5wLm"
           "xvZyhkdHJbIkV4cG9zdXJlIl0pKS5maXQoKQogICAgICAgICAgICBwcmVkID0g"
           "bnAuYXNhcnJheShyLnByZWRpY3QoZHRlLCBvZmZzZXQ9bnAuemVyb3MobGVuKG"
           "R0ZSkpKSkgICAjIG9mZnNldCAwIOydtOudvCDsl7DsnKjsnbQg64KY7Ji164uI"
           "64ukCiAgICAgICAgICAgIGZvbGRfc2NvcmVzLmFwcGVuZChtZWFuX3BvaXNzb2"
           "5fZGV2aWFuY2UoeV9yYXRlW3RlXSwgcHJlZCwgc2FtcGxlX3dlaWdodD13X2V4"
           "cFt0ZV0pKQogICAgICAgIGN2X2RldltuYW1lXSA9IG5wLmFycmF5KGZvbGRfc2"
           "NvcmVzKQogICAgICAgIHNlID0gY3ZfZGV2W25hbWVdLnN0ZChkZG9mPTEpIC8g"
           "bnAuc3FydCg1KQogICAgICAgIHByaW50KGYie25hbWU6NXN9IO2Pieq3oCA9IH"
           "tjdl9kZXZbbmFtZV0ubWVhbigpOi42Zn0gIO2PtOuTnCDqsIQg7ZGc7KSA7Jik"
           "7LCoID0ge3NlOi42Zn0iCiAgICAgICAgICAgICAgZiIgIO2PtOuTnOuzhCB7bn"
           "Aucm91bmQoY3ZfZGV2W25hbWVdLCA2KX0iKQoKICAgIGdhcCA9IGN2X2Rldlsi"
           "Y29yZSJdIC0gY3ZfZGV2WyJmdWxsIl0gICAgICAgICAgICAgICAgICAgICAgIC"
           "Mg6rCZ7J2AIO2PtOuTnOyXkOyEnCDruoAg7Ked7KeA7J2AIOywqOydtAogICAg"
           "Z2FwX3NlID0gZ2FwLnN0ZChkZG9mPTEpIC8gbnAuc3FydCg1KQogICAgcHJpbn"
           "QoZiJjb3JlIC0gZnVsbCDsp53sp4DsnYAg7LCo7J20IO2Pieq3oCA9IHtnYXAu"
           "bWVhbigpOi42Zn0gIO2RnOykgOyYpOywqCA9IHtnYXBfc2U6LjZmfSIpCiAgIC"
           "BwcmludChmIu2Pieq3oCAvIO2RnOykgOyYpOywqCA9IHtnYXAubWVhbigpIC8g"
           "Z2FwX3NlOi4yZn0g67CwIikKICAgIGJlc3QgPSBtaW4oY3ZfZGV2LCBrZXk9bG"
           "FtYmRhIGs6IGN2X2RldltrXS5tZWFuKCkpCiAgICB0aHIgPSBjdl9kZXZbYmVz"
           "dF0ubWVhbigpICsgY3ZfZGV2W2Jlc3RdLnN0ZChkZG9mPTEpIC8gbnAuc3FydC"
           "g1KQogICAgcHJpbnQoZiIxLe2RnOykgOyYpOywqCDqt5zsuZnsnZgg7ISgID0g"
           "e3RocjouNmZ9ICAtPiAg6re4IOyEoCDslYjsnZgg7ZuE67O0IHtbayBmb3Igay"
           "BpbiBjdl9kZXYgaWYgY3ZfZGV2W2tdLm1lYW4oKSA8PSB0aHJdfSIpCgogICAg"
           "ZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEwLCAzLj"
           "YpKQogICAgZm9yIGksIG5hbWUgaW4gZW51bWVyYXRlKFsibnVsbCIsICJjb3Jl"
           "IiwgImZ1bGwiXSk6CiAgICAgICAgYXhlc1swXS5wbG90KFtpXSAqIDUsIGN2X2"
           "RldltuYW1lXSwgIm8iLCBjb2xvcj0iI0IzQjNCMyIpICAgICAgICAgICAgICAg"
           "ICAgICAgICMg7Y+065OcIOygkOyImCA16rCcCiAgICAgICAgYXhlc1swXS5wbG"
           "90KFtpXSwgW2N2X2RldltuYW1lXS5tZWFuKCldLCAiXyIsIG1zPTIyLCBtZXc9"
           "MywgY29sb3I9IiMwNTFDMkMiKSAgICMg6re4IO2Pieq3oAogICAgYXhlc1swXS"
           "5zZXRfeHRpY2tzKFswLCAxLCAyXSk7IGF4ZXNbMF0uc2V0X3h0aWNrbGFiZWxz"
           "KFsibnVsbCIsICJjb3JlIiwgImZ1bGwiXSkKICAgIGF4ZXNbMF0uc2V0X3hsYW"
           "JlbCgi7ZuE67O0IOuqqO2YlSIpOyBheGVzWzBdLnNldF95bGFiZWwoIuq1kOyw"
           "qOqygOymnSDtj6zslYTshqEg7J207YOI64+EIikKICAgIGF4ZXNbMV0uYmFyKH"
           "JhbmdlKDUpLCBnYXAsIGNvbG9yPSIjMDBBOUY0IikKICAgIGF4ZXNbMV0uYXho"
           "bGluZSgwLCBjb2xvcj0iIzc1NzU3NSIpOyBheGVzWzFdLmF4aGxpbmUoZ2FwLm"
           "1lYW4oKSwgY29sb3I9IiMwNTFDMkMiLCBscz0iOiIpCiAgICBheGVzWzFdLnNl"
           "dF94dGlja3MocmFuZ2UoNSkpOyBheGVzWzFdLnNldF94dGlja2xhYmVscyhbZi"
           "Ltj7Trk5wge2kgKyAxfSIgZm9yIGkgaW4gcmFuZ2UoNSldKQogICAgYXhlc1sx"
           "XS5zZXRfeGxhYmVsKCLtj7Trk5wiKTsgYXhlc1sxXS5zZXRfeWxhYmVsKCJjb3"
           "JlIC0gZnVsbCDsnbTtg4jrj4Qg7LCo7J20IikKICAgIHBsdC5zaG93KCkKCu2V"
           "tOyEnQoKLSDsmIjsg4EgM+qwnCDspJEg64u17J2AIOuRkCDrsojsp7gsIOOAjO"
           "yInOychOuKlCDqsJnsp4Drp4wsIO2ZleyepSDrqqjtmJXqs7wg7ZW17IusIOuq"
           "qO2YleydmCDssKjsnbTqsIAg7Y+065OcIOqwhCDsgrDtj6zrs7Tri6Qg7J6R64"
           "uk44CNIOyeheuLiOuLpC4g7Y+065OcIO2Pieq3oCDsnbTtg4jrj4TripQgbnVs"
           "bCAwLjYyNDg4OCA+IGNvcmUgMC42MTA2NjIgPiBmdWxsIDAuNjA4MjQ4IOuhnC"
           "DslrTsoJwgQUlDIOqwgCDrp6TquLQg7Iic7JyE7JmAIOqwmeydgCDsiJzshJzs"
           "mIDsirXri4jri6QuIOOAjOykgOu5hCDshYDjgI3snbQg7Lac66Cl7ZW0IOuRlC"
           "DquLDspIDshKAg4oCUIOyghOyytCDqs4Tslb3sl5Ag6rCZ7J2AIOyalOycqCAw"
           "LjEwMDYxNCDtlZjrgpjrpbwg66ek6rK87J2EIOuVjOqwgCAwLjYyNDg4MSDigJ"
           "Qg6rO8IG51bGwg7J2YIO2Pieq3oOydtCDsgqzsi6Tsg4Eg6rCZ7J2AIOqwkuyd"
           "uCDqsoPrj4Qg7ZmV7J247ZW0IOuRkOyEuOyalC4gbnVsbCDsnbQg7JiI7Lih7Z"
           "WY64qUIOqwkuydtCDsoITssrQg6rOE7JW9IOyCrOqzoOycqCDtlZjrgpjsnoXr"
           "i4jri6QuCi0g7ISk66qF67OA7IiY66W8IOuEo+yWtCDqt7gg6riw7KSA7ISgKD"
           "AuNjI0ODgxKeuztOuLpCDsnbTtg4jrj4TqsIAg64Ku7JWE7KeEIO2PreydgCBj"
           "b3JlIOqwgCAwLjAxNDIsIGZ1bGwg7J20IDAuMDE2NiDsnoXri4jri6QuIO2YhO"
           "yerCDruYTqtZAg64yA7IOB7J24IOuRkCDrqqjtmJXsnZgg7LCo7J20IDAuMDAy"
           "NDE0IOuKlCDqt7gg6rCQ7IaM7Y+t7J2YIOyVvSAxNSUg7J6F64uI64ukIOKAlC"
           "DsiJzsnITripQg64KY64mY7JeI7KeA66eMIOywqOydtOydmCDtgazquLDripQg"
           "7YGs7KeAIOyViuyKteuLiOuLpC4KLSDsmbzsqr0g6re466a87JeQ7IScIO2ajO"
           "yDiSDsoJAgNeqwnOqwgCDshLjroZzroZwg7Z2p7Ja07KeEIOuylOychOqwgCDt"
           "m4Trs7QgM+qwnCDsgqzsnbTsnZgg6rCE6rKp67O064ukIOuEk+yKteuLiOuLpC"
           "4g7ZmV7J6lIOuqqO2YleydmCDtj7Trk5wg6rCEIO2RnOykgOyYpOywqOqwgCAw"
           "LjAwMjYxNyDroZwg65GQIOuqqO2YleydmCDssKjsnbQgMC4wMDI0MTQg67O064"
           "ukIO2BrOqzoCwgMS3tkZzspIDsmKTssKgg6rec7LmZ7J2YIOyEoCAwLjYxMDg2"
           "NCDslYjsl5AgZnVsbCDqs7wgY29yZSDqsIAg7ZWo6ruYIOuTpOyWtOyYteuLiO"
           "uLpC4g6rec7LmZ64yA66Gc652866m0IDLqsJwg7KSRIOuLqOyInO2VnCBjb3Jl"
           "IOulvCDshKDtg53tlZjqsowg65Cp64uI64ukLgotIOq3uOufsOuNsCDsmKTrpb"
           "jsqr0g7Yyo64SQ7J2AIOqysOqzvOqwgCDri6TrpoXri4jri6QuIOqwmeydgCDt"
           "j7Trk5zsl5DshJwg67qAIOywqOydtOuKlCDtj7Trk5wgNeqwnCDrqqjrkZAg7J"
           "aR7IiYKDAuMDAxNTAwIH4gMC4wMDMyOTEp7J206rOgLCDqt7ggNeqwnOuhnCDs"
           "grDstpztlZwg7ZGc7KSA7Jik7LCo64qUIDAuMDAwMjk1IOKAlCDtj4nqt6Dsnb"
           "Qg7ZGc7KSA7Jik7LCo7J2YIDguMTnrsLDsnoXri4jri6QuIO2ZleyepSDrqqjt"
           "mJXsnZgg7J207YOI64+E6rCAIOyasOyXsO2eiCDrjZQg64Ku6rKMIOuCmOyYqC"
           "DqsoPsnbQg7JWE64uI652864qUIOucu+yeheuLiOuLpC4KLSDrkZAg6rOE7IKw"
           "7J20IOuLpOuluCDrp5DsnYQg7ZWY64qUIOydtOycoOuKlCDtj7Trk5wg6rCEIO"
           "yCsO2PrOydmCDsoJXssrTsl5Ag7J6I7Iq164uI64ukLiDsgrDtj6zsnZgg64yA"
           "67aA67aE7J2AIO2PtOuTnOuniOuLpCDqsoDspp0g6rOE7JW97J20IOuLpOultO"
           "uLpOuKlCDrjbDshJwg7Jik6rOgLCDqt7gg67OA64+Z7J2AIO2bhOuztCAz6rCc"
           "7JeQIOuPmeydvO2VmOqyjCDsnpHsmqntlanri4jri6QuIOyLpOygnOuhnCA067"
           "KIIO2PtOuTnOyXkOyEnOuKlCDtm4Trs7QgM+qwnOqwgCDrqqjrkZAg64Ku6rOg"
           "KG51bGwgMC42MTYzOTUgwrcgY29yZSAwLjYwMjQzMiDCtyBmdWxsIDAuNTk5Nz"
           "UwKSwgM+uyiCDtj7Trk5zsl5DshJzripQg66qo65GQIOuGkuyKteuLiOuLpCgw"
           "LjYzMjcxOCDCtyAwLjYxNzE0MiDCtyAwLjYxNTY0MSkuIOqwmeydgCDtj7Trk5"
           "wg7JWI7JeQ7IScIOu5vOuptCDsnbQg6rO17Ya1IOuzgOuPmeydtCDsoJzqsbDr"
           "kJjqs6Ag66qo7ZiVIOywqOydtOunjCDrgqjslYQsIOu5hOq1kCDquLDspIDsnZ"
           "gg7ZGc7KSA7Jik7LCo6rCAIOyVvSA567CwIOyekeyVhOynkeuLiOuLpCgwLjAw"
           "MjYxNyDihpIgMC4wMDAyOTUpLgotIOq3uOuemOyEnCDrs7Tqs6DshJzsl5Drip"
           "Qg65GQIOusuOyepeydtCDri6Qg7ZWE7JqU7ZWp64uI64ukIOKAlCDtmZXsnqUg"
           "66qo7ZiV7J20IOydvOq0gOuQmOqyjCDsobDquIgg64Kr64ukLiAi7J286rSA65"
           "CY6rKMIuuKlCDsp53sp4DsnYAg7LCo7J206rCALCAi7KGw6riIIuydgCDtj7Tr"
           "k5wg6rCEIOyCsO2PrOqwgCDrkrfrsJvsuajtlanri4jri6QuCi0g7ZqM7J2Y7J"
           "eQIOygnOy2nO2VoCDslYjsnYAg64uk7J2MIDPqsIDsp4Ag7KSRIO2VmOuCmOyd"
           "tOqzoCwgM+qwgOyngCDrqqjrkZAg6re86rGw66W8IOqwluy2peuLiOuLpC4gKD"
           "EpIO2ZleyepSDrqqjtmJUo6rOE7IiYIDQx6rCcKSDigJQg7Ked7KeA7J2AIOyw"
           "qOydtOqwgCDtj7Trk5wgNeqwnCDrqqjrkZAg6rCZ7J2AIOu2gO2YuOydtOqzoC"
           "wg7KeA7JetwrfruIzrnpzrk5zripQg7JqU7Jyo7ZGc6rCAIOyLpOygnOuhnCDs"
           "gqzsmqntlZjripQg7ISk66qF67OA7IiY7J6F64uI64ukLiAoMikg7ZW17IusIO"
           "uqqO2YlSjqs4TsiJggOeqwnCkg4oCUIDEt7ZGc7KSA7Jik7LCoIOq3nOy5meyd"
           "mCDshKAg7JWI7JeQIOuTpOqzoCwg7J207YOI64+EIOywqOydtCAwLjAwMjQxNC"
           "DripQg7J207YOI64+EIOq4sOykgOyEoOyXkOyEnCAwLjQlIOyImOykgOyduCDr"
           "jbDri6Qg6rOE7IiYIDQx6rCcIOykkeyXkOuKlCDqs4Tslb0gMSwzMjbqsbTsnb"
           "gg7KeA7Jet7J2YIOqzhOyImOuPhCDtj6ztlajrkJjslrQg7J6I7Ja0IOq3uCDq"
           "s4TsiJjqsIAg6re464yA66GcIOqzoOqwnSDssq3qtazslaHsnbQg65Cp64uI64"
           "ukLiAoMykgMuqwnOulvCDsuLXsnLzroZwg4oCUIOq4sOuzuCDqtazsobDripQg"
           "Y29yZSDroZwg7ISk7KCV7ZWY6rOgIOyngOyXrcK367iM656c65OcIOyhsOygle"
           "ydgCDrr7jshZggMiDsl5DshJwg64uk66OsIOygleq3nO2ZlOuhnCDstpXshozt"
           "lbQg642U7ZWp64uI64ukLiDshKDtg53tlZwg7Kq96rO8IOq3vOqxsOulvCAz66"
           "y47J6l7Jy866GcIOyggeyWtCDrkZDshLjsmpQuIOydtCDshKDtg53snbQg65Kk"
           "IO2UhOuhnOygne2KuOydmCDrsKnrspUg7ISg7YOd7Jy866GcIOq3uOuMgOuhnC"
           "DsnbTslrTsp5Hri4jri6QuCgrsvZTrk5zqsIAg64u17J2EIOunjOuTnOuKlCDr"
           "sKnsi50g4oCUIOy9lOuTnOuKlCDtj7Trk5zrp4jri6Qg7ZWZ7Iq1IOu2gOu2hO"
           "ynke2VqeycvOuhnOunjCDqs4TsiJjrpbwg7LaU7KCV7ZWY6rOgIOq3uCDqs4Ts"
           "iJjroZwg6rKA7KadIOu2gOu2hOynke2VqeydmCDsl7DsnKjsnYQg7JiI7Lih7Z"
           "W0IOygkOyImOulvCDsgrDstpztlanri4jri6QuIOydtCDsoIjssKjrpbwg7ZuE"
           "67O0IDPqsJzqsIAg6rCZ7J2AIO2PtOuTnCA16rCc66GcIOuwmOuzte2VmOuvgO"
           "uhnCDsoJDsiJjqsIAg7ZuE67O066eI64ukIDXqsJzslKkg64iE7KCB65CY6rOg"
           "LCDshLjroZwg67Cp7Zal7Jy866GcIO2Pieq3oOydhCDrgrTrqbQgIuydtCDrqq"
           "jtmJXsnbQg7IOIIOqzhOyVveyXkOyEnCDslrzrp4jrgpgg7YuA66as64qU6rCA"
           "IiDqsIAsIO2bhOuztOuBvOumrCDqsJnsnYAg7Y+065Oc7JeQ7IScIOu5vOuptC"
           "Ai65GQIOuqqO2YleydtCDqsJnsnYAg6rOE7JW97JeQ7IScIOyWvOuniOuCmCDr"
           "i6TrpbjqsIAiIOqwgCDrkKnri4jri6QuCg==",
           "과제":
           "8J+UkyDqs7zsoJwg7JmE7ISx67O4IOKAlCDjgIxBSSDsvZTrk5wg7Iuc7J6RIO"
           "2WieOAjSDslYTrnpjsl5Ag65Ok7Ja07JmA7JW8IO2VmOuKlCDsvZTrk5zripQg"
           "7J20IOygleuPhOyeheuLiOuLpAoK4pagIOydtCDsvZTrk5zrpbwg67aA66W4IO"
           "2UhOuhrO2UhO2KuCDsmIgKCiAgICDsnbTrr7gg66eM65Ok7Ja07KC4IOyeiOuK"
           "lCBkZiwgRl9DT1JFLCBSRUcsIEJPT1QsIEIsIFNFRUQg66W8IOq3uOuMgOuhnC"
           "Dsk7Dqs6Ag64uk7IucIOunjOuTpOyngOuKlCDrp5DslYQg7KSYLiBGX0NPUkUg"
           "66W8IO2PrOyVhOyGoSBHTE0g7Jy866GcIO2VnCDrsogg7KCB7ZWp7ZW0IOykmC"
           "4g7J2065WMCiAgICBvZmZzZXQg7JeQ64qUIGRmIOydmCBFeHBvc3VyZSDsl5Ag"
           "66Gc6re466W8IOyUjOyatCDqsJLsnYQg7KCE64us7ZWY6rOgLCDqsJnsnYAg7I"
           "ud7J2EIGNvdl90eXBlPSJjbHVzdGVyIiDsmYAgY292X2t3ZHM9eyJncm91cHMi"
           "OiBSRUd9IOuhnAogICAg7ZWcIOuyiCDrjZQg7KCB7ZWp7ZW07IScIEJvbnVzTW"
           "FsdXMg6rOE7IiY7J2YIOuRkCDtkZzspIDsmKTssKjrpbwg64KY64iIIOuwsOyI"
           "mOulvCAi7ZWg7Kad7KCQ7IiYIOqzhOyImOydmCDtkZzspIDsmKTssKgg67Cw7I"
           "iYID0gKOyIq+yekCkg67CwIiDroZwg7IaM7IiY7KCQIOyVhOuemCAy7J6Q66as"
           "6rmM7KeAIHByaW50IO2VtCDspJguCiAgICDqt7gg64uk7J2MIOqzhOyVveydhC"
           "DsmIjsuKHqsJIg7Iic7Jy866GcIOq1rOqwhCAxNeqwnOuhnCDrtoTtlaDtlbQg"
           "6rWs6rCE66eI64ukIENsYWltTmIg7J2YIOu2hOyCsOydhCDtj4nqt6DsnLzroZ"
           "wg64KY64iE6rOgLCDqt7gg6rCS7J20IDEg7J2EIOuEmOuKlCDqtazqsITsnbQg"
           "66qHIOqwnOyduOyngCAi67aE7IKww7ftj4nqt6DsnbQgMSDsnITsqr3snbgKIC"
           "AgIOy5uCDsiJggPSAo7Iir7J6QKSDsubgiIOycvOuhnCDtlZwg7ZaJLCDqtazq"
           "sIQgMTXqsJzsnZgg7Y+J6reg7J24IOqzvOyCsO2PrCDPhsyCIOyZgCDqt7gg7K"
           "Cc6rOx6re864+EIO2VnCDtlokgcHJpbnQg7ZW0IOykmC4g7J207YOI64+EIOye"
           "lOywqOyXkOyEnCDsoITssrQg7Y+J6reg7J2EIOu6gCDrkqQgUkVHIOuhnCDrrL"
           "bslrQKICAgIOyngOyXreuniOuLpCDsp4Dsl63rs4Qg7Y+J6reg7J2EIOuCtOqz"
           "oCwg6rCZ7J2AIOyelOywqOulvCDsi6TsoJwg7KeA7Jet67OEIOqzhOyVvSDsiJ"
           "jsmYAg7YGs6riw6rCAIOqwmeydgCDrtoDrtoTsp5HtlakgMjLqsJzsl5AgU0VF"
           "RCDroZwg7Iuc7J6R7ZWY64qUIOuCnOyImOuhnCBCIO2ajCDri6Tsi5wg67Cw67"
           "aE7ZW0IOyWu+ydgCDrtoDrtoTsp5Htlanrs4Qg7Y+J6reg7J2YIDIuNSUg7JmA"
           "IDk3LjUlIOuwseu2hOychOulvAogICAg6re4IOyngOyXreydmCDquLDspIDqta"
           "zqsITsnLzroZwg7IK87JWEIOq4sOykgOq1rOqwhCDrsJbsnLzroZwg64KY6rCE"
           "IOyngOyXreydtCDrqocg6rOz7J247KeA64+EIO2VnCDtlokg7Lac66Cl7ZW0IO"
           "ykmC4g6rCA7KCVIOygkOqygO2RnOuKlCDrk7HrtoTsgrDqs7wg64+F66a9IDLq"
           "sJwg7ZaJ7Jy866GcIOustOyXh+ycvOuhnCDrtKTrgpggwrcg7KeE64uo6rCSIM"
           "K3IOq1rOqwhCDquLjsnbTsl5Ag66ek6ri4IOuwsOyImCDCtwogICAg7YyQ7KCV"
           "IDTqsJwg7Je07J2EIOuLtOyVhCDsnbjsh4TtlZjqs6AsIEJPT1Qg7J2YIFJlZ2"
           "lvbiBSNDMg7ZaJ7JeQ7IScIOuztOyglSDsoIQg6rWs6rCEIOq4uOydtOyZgCDi"
           "iJrPhsyCIOulvCDqs7HtlZwg67O07KCVIO2bhCDqtazqsIQg6ri47J2064+EIO"
           "qxtC/rhYQg64uo7JyE66GcIO2VqOq7mCDsoIHslrQg7KSYLiDqt7jrnpjtlITr"
           "ipQg7ZWcIOyepeunjAogICAg4oCUIOyEuOuhnOuhnCDsp4Dsl60gMjLqsJzrpb"
           "wg67Cw7LmY7ZWY6rOgIOyngOyXreuniOuLpCDquLDspIDqtazqsITsnZgg65GQ"
           "IOuBneydhCDqtbXsnYAg6rCA66GcIOunieuMgOuhnCDquIvqs6Ag7KeA7Jet67"
           "OEIO2Pieq3oCDsnITsuZjsl5Ag66eI7Luk66W8IO2VmOuCmOyUqSDtkZzsi5zt"
           "lZwg65KkLCAwIOychOy5mOyXkCDshLjroZwg7KCQ7ISg7J2EIOq4i+qzoAogIC"
           "Ag7IS466Gc7LaVIOuIiOq4iOyXkCDsp4Dsl60g7J2066aE7J2EIOyggeqzoCDr"
           "kZAg7LaV7JeQIOudvOuyqOydhCDri6zslYQg7KSYLgoKICAgICMgMSkg6rCZ7J"
           "2AIOyLneydhCAy7ZqMIOygge2Vqe2VqeuLiOuLpCDigJQg7Y+s7JWE7IahIO2R"
           "nOykgOyYpOywqCAx6rCcLCDsp4Dsl63snLzroZwg66y27J2AIOq1sOynkSDtkZ"
           "zspIDsmKTssKggMeqwnAogICAgZml0X3BvaXMgPSBzbWYuZ2xtKGZvcm11bGE9"
           "Rl9DT1JFLCBkYXRhPWRmLCBmYW1pbHk9c20uZmFtaWxpZXMuUG9pc3NvbigpLA"
           "ogICAgICAgICAgICAgICAgICBvZmZzZXQ9bnAubG9nKGRmWyJFeHBvc3VyZSJd"
           "KSkuZml0KCkKICAgIGZpdF9jbHVzdGVyID0gc21mLmdsbShmb3JtdWxhPUZfQ0"
           "9SRSwgZGF0YT1kZiwgZmFtaWx5PXNtLmZhbWlsaWVzLlBvaXNzb24oKSwKICAg"
           "ICAgICAgICAgICAgICAgb2Zmc2V0PW5wLmxvZyhkZlsiRXhwb3N1cmUiXSkpLm"
           "ZpdChjb3ZfdHlwZT0iY2x1c3RlciIsCiAgICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY292X2t3ZHM9eyJncm91cH"
           "MiOiBSRUd9KQogICAgc2VfcmF0aW8gPSBmbG9hdChmaXRfY2x1c3Rlci5ic2Vb"
           "IkJvbnVzTWFsdXMiXSAvIGZpdF9wb2lzLmJzZVsiQm9udXNNYWx1cyJdKSAgIC"
           "AgICMg6rCZ7J2AIOqzhOyImOyXkOyEnCDrgpjriJXri4jri6QKICAgIHByaW50"
           "KGYi7ZWg7Kad7KCQ7IiYIOqzhOyImOydmCDtkZzspIDsmKTssKgg67Cw7IiYIC"
           "jqtbDsp5EgLyDtj6zslYTshqEpID0ge3NlX3JhdGlvOi4yZn0g67CwIikKCiAg"
           "ICAjIDIpIOyYiOy4oeqwkiDsiJzsnLzroZwg6rWs6rCEIDE16rCcIOKAlCDqta"
           "zqsITrp4jri6Qg44CM67aE7IKwIMO3IO2Pieq3oOOAjQogICAgeSA9IGRmWyJD"
           "bGFpbU5iIl0udG9fbnVtcHkoZmxvYXQpCiAgICBiaW5zID0gbnAuYXJyYXlfc3"
           "BsaXQobnAuYXJnc29ydChucC5hc2FycmF5KGZpdF9wb2lzLmZpdHRlZHZhbHVl"
           "cywgZHR5cGU9ZmxvYXQpKSwgMTUpCiAgICB2bSA9IG5wLmFycmF5KFt5W2Jpbl"
           "9pZHhdLnZhcihkZG9mPTEpIC8geVtiaW5faWR4XS5tZWFuKCkgZm9yIGJpbl9p"
           "ZHggaW4gYmluc10pCiAgICBwcmludChmIuu2hOyCsMO37Y+J6reg7J20IDEg7J"
           "yE7Kq97J24IOy5uCDsiJggPSB7aW50KCh2bSA+IDEpLnN1bSgpKX0g7Lm4IikK"
           "ICAgIHByaW50KGYi6rO87IKw7Y+sIM+GzIIgPSB7dm0ubWVhbigpOi40Zn0gwr"
           "cg7Y+t7JeQIOunpOq4uCDrsLDsiJgg4oiaz4bMgiA9IHtucC5zcXJ0KHZtLm1l"
           "YW4oKSk6LjRmfSDrsLAiKQoKICAgICMgMykg7J207YOI64+EIOyelOywqOulvC"
           "Dsp4Dsl63snLzroZwg66y26rOgLCDqsJnsnYAg7J6U7LCo66W8IOustOyekeyc"
           "hOuhnCDtnanslrQg6riw7KSA6rWs6rCE7J2EIOunjOuTreuLiOuLpAogICAgcm"
           "VzaWQgPSBucC5hc2FycmF5KGZpdF9wb2lzLnJlc2lkX2RldmlhbmNlLCBkdHlw"
           "ZT1mbG9hdCkKICAgIHJlc2lkID0gcmVzaWQgLSByZXNpZC5tZWFuKCkgICAgIC"
           "AgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIDAg7J2EIOqw"
           "gOyatOuNsCDrkaHri4jri6QKICAgIHJlZ2lvbnMgPSBzb3J0ZWQoc2V0KFJFRy"
           "kpCiAgICBzaXplcyA9IG5wLmFycmF5KFtpbnQoKFJFRyA9PSByKS5zdW0oKSkg"
           "Zm9yIHIgaW4gcmVnaW9uc10pCiAgICBvYnNlcnZlZCA9IG5wLmFycmF5KFtyZX"
           "NpZFtSRUcgPT0gcl0ubWVhbigpIGZvciByIGluIHJlZ2lvbnNdKQoKICAgIHJu"
           "ZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhTRUVEKQogICAgY29sbGVjdGVkID"
           "0gbnAuZW1wdHkoKEIsIGxlbihyZWdpb25zKSkpCiAgICBjdXRfcG9pbnRzID0g"
           "bnAuY3Vtc3VtKHNpemVzKVs6LTFdCiAgICBmb3IgYiBpbiByYW5nZShCKToKIC"
           "AgICAgICBzaHVmZmxlZCA9IHJuZy5wZXJtdXRhdGlvbihyZXNpZCkgICAgICAg"
           "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICMg6rWw7KeRIOyDgeq0gOydtC"
           "Dsl4bripQg6rK97JqwIO2VnCDrsowKICAgICAgICBjb2xsZWN0ZWRbYl0gPSBb"
           "cGllY2UubWVhbigpIGZvciBwaWVjZSBpbiBucC5zcGxpdChzaHVmZmxlZCwgY3"
           "V0X3BvaW50cyldCiAgICBiYW5kX2xvID0gbnAucGVyY2VudGlsZShjb2xsZWN0"
           "ZWQsIDIuNSwgYXhpcz0wKQogICAgYmFuZF9oaSA9IG5wLnBlcmNlbnRpbGUoY2"
           "9sbGVjdGVkLCA5Ny41LCBheGlzPTApCiAgICBvdXRzaWRlID0gKG9ic2VydmVk"
           "IDwgYmFuZF9sbykgfCAob2JzZXJ2ZWQgPiBiYW5kX2hpKQogICAgcHJpbnQoZi"
           "LrrLTsnpHsnIQg7J6s67Cw7LmYIOq4sOykgOq1rOqwhCDrsJbsnLzroZwg64KY"
           "6rCEIOyngOyXrSA9IHtpbnQob3V0c2lkZS5zdW0oKSl9IC8ge2xlbihyZWdpb2"
           "5zKX0g6rOzIikKCiAgICAjIDQpIOqwgOyglSDsoJDqsoDtkZwgMuqwnCDtlonq"
           "s7wg67O07KCVIOyghO2bhOydmCDqtazqsIQg6ri47J20CiAgICBkaWFnX3RhYm"
           "xlID0gcGQuRGF0YUZyYW1lKFsKICAgICAgICB7IuustOyXh+ycvOuhnCDrtKTr"
           "gpgiOiAi7JiI7Lih6rCSIOyInCDsl7Tri6TshK8g7Lm47J2YIOu2hOyCsMO37Y"
           "+J6regIiwKICAgICAgICAgIuynhOuLqOqwkiI6IGYiz4bMgiA9IHt2bS5tZWFu"
           "KCk6LjRmfSIsCiAgICAgICAgICLtj63sl5Ag66ek6ri4IOuwsOyImCI6IGYi4o"
           "iaz4bMgiA9IHtucC5zcXJ0KHZtLm1lYW4oKSk6LjRmfSDrsLAiLCAi7YyQ7KCV"
           "IjogIuqzoOyzkCDsnb3qs6Ag6rKw66Gg7J2AIOq3uOuMgOuhnCJ9LAogICAgIC"
           "AgIHsi66y07JeH7Jy866GcIOu0pOuCmCI6ICLsnbTtg4jrj4Qg7J6U7LCo7J2Y"
           "IOyngOyXreuzhCDtj4nqt6Agwrcg6rWw7KeRIO2RnOykgOyYpOywqCIsCiAgIC"
           "AgICAgICLsp4Tri6jqsJIiOiBmIuq4sOykgOq1rOqwhCDrsJYge2ludChvdXRz"
           "aWRlLnN1bSgpKX3qs7Mgwrcg67Cw7IiYIHtzZV9yYXRpbzouMmZ9IOuwsCIsCi"
           "AgICAgICAgICLtj63sl5Ag66ek6ri4IOuwsOyImCI6ICLqs4TsiJjrp4jri6Qg"
           "64uk66aEIiwgIu2MkOyglSI6ICLtlZwg7Iir7J6Q66GcIOuztOygle2VoCDsiJ"
           "gg7JeG7J2MIn1dLAogICAgICAgIGluZGV4PVsi65Ox67aE7IKwKOqzvOyCsO2P"
           "rCkiLCAi64+F66a9KOyngOyXrSDqtbDsp5Eg7IOB6rSAKSJdKQogICAgcHJpbn"
           "QoZGlhZ190YWJsZS50b19zdHJpbmcoKSkKCiAgICByNDMgPSBCT09ULnNldF9p"
           "bmRleCgi7IS46re466i87Yq4IikubG9jWyJSZWdpb24gUjQzIl0KICAgIHByaW"
           "50KGYiUmVnaW9uIFI0MyDtj60gOiDrs7TsoJUg7KCEIHtyNDNbJ+2PrSddOi42"
           "Zn0g4oaSIOuztOyglSDrkqQgIgogICAgICAgICAgZiJ7cjQzWyftj60nXSAqIG"
           "5wLnNxcnQodm0ubWVhbigpKTouNmZ9IOqxtC/rhYQiKQoKICAgICMgNSkg6re4"
           "66a8IO2VnCDsnqUg4oCUIOyngOyXrSAyMuqwnOydmCDsp4Dsl63rs4Qg7Y+J6r"
           "eg6rO8IOustOyekeychCDsnqzrsLDsuZgg6riw7KSA6rWs6rCECiAgICB5cG9z"
           "ID0gbnAuYXJhbmdlKGxlbihyZWdpb25zKSlbOjotMV0KICAgIGZpZywgYXggPS"
           "BwbHQuc3VicGxvdHMoZmlnc2l6ZT0oRklHX1cgKiAwLjgsIDUuMCkpCiAgICBh"
           "eC5obGluZXMoeXBvcywgYmFuZF9sbywgYmFuZF9oaSwgY29sb3I9Q09MWyLtlZ"
           "jripjsg4kiXSwgbHc9NCkgICAgICAgICAgICAgICAjIOustOyekeychCDsnqzr"
           "sLDsuZgg6riw7KSA6rWs6rCECiAgICBheC5wbG90KG9ic2VydmVkLCB5cG9zLC"
           "AibyIsIGNvbG9yPUNPTFsi64Ko7IOJIl0pICAgICAgICAgICAgICAgICAgICAg"
           "ICAgICMg7Iuk7KCcIOyngOyXreuzhCDtj4nqt6AKICAgIGF4LmF4dmxpbmUoMC"
           "wgY29sb3I9Q09MWyLtmozsg4kiXSwgbHM9IjoiKQogICAgYXguc2V0X3l0aWNr"
           "cyh5cG9zKQogICAgYXguc2V0X3l0aWNrbGFiZWxzKHJlZ2lvbnMpCiAgICBheC"
           "5zZXRfeGxhYmVsKCLsnbTtg4jrj4Qg7J6U7LCo7J2YIOyngOyXreuzhCDtj4nq"
           "t6AgKOyghOyytCDtj4nqt6DsnYQg67qAIOqwkikiKQogICAgYXguc2V0X3lsYW"
           "JlbCgi7KeA7JetIikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSDqtbDsp5Eg"
           "7IOB6rSAKOqwmeydgCDsp4Dsl60g6rOE7JW97J2YIOyDgeq0gCnsnbQg7Jik64"
           "qYIOq1rOqwhCDquLjsnbTrpbwg6rCA7J6lIO2BrOqyjCDspp3qsIDsi5ztgrXr"
           "i4jri6QuIOqzvOyCsO2PrOuhnCDrhJPslrTsp4DripQg6rWs6rCEIOq4uOydtO"
           "uKlCDiiJrPhsyCID0gMS4wMjk1LCDqs6cg7JW9IDMlIOyduOuNsCwg7KeA7Jet"
           "IOq1sOynkSDsg4HqtIDsnYAg7ZWg7Kad7KCQ7IiYIOqzhOyImOydmCDtkZzspI"
           "DsmKTssKjrpbwgNS4zNOuwsOuhnCwg7Jq07KCE7J6QIOuCmOydtOuKlCA2LjE4"
           "67Cw66GcIOymneqwgOyLnO2CteuLiOuLpC4KLSDtlZwg7Iir7J6Q66GcIOuztO"
           "ygle2VoCDsiJgg7J6I64qUIOqyg+qzvCDsl4bripQg6rKD7J20IOuCmOuJqeuL"
           "iOuLpC4g4oiaz4bMgiDripQg66qo65OgIOqzhOyImOyXkCDqsJnsnYAg67Cw7I"
           "iY6528IOq1rOqwhCDquLjsnbTsl5Ag7ZWcIOuyiCDqs7HtlZjrqbQg65CY7KeA"
           "66eMLCDqtbDsp5Eg67Cw7IiY64qUIOqzhOyImOuniOuLpCDri6TrpoXri4jri6"
           "Qg4oCUIOuPhOyLnO2ZlCDrk7HquIkgRiDripQg7Jik7Z6I66CkIDAuNjHrsLDr"
           "oZwg6rCQ7IaM7ZWp64uI64ukLgotIOygkOy2lOygleydgCDsmIHtlqXsnYQg67"
           "Cb7KeAIOyViuqzoCDqtazqsITrp4wg64us65287KeR64uI64ukLiDqs4TsiJgg"
           "OeqwnCDspJEg6rOE7IiY7JeQ7IScIO2RnOykgOyYpOywqOydmCAy67Cw66W8IO"
           "uNlO2VmOqzoCDruoAg6rWs6rCE7J20IDAg7J2EIO2PrO2VqO2VmOyngCDslYrr"
           "ipQg6rKD7J2AIO2PrOyVhOyGoSDtkZzspIDsmKTssKjsl5DshJzrj4Qg6rWw7K"
           "eRIO2RnOykgOyYpOywqOyXkOyEnOuPhCA46rCc66GcIOuPmeydvO2VqeuLiOuL"
           "pC4g64us65287KeA64qUIOqyg+ydgCDtmZXsi6DsnZgg7YGs6riw7J6F64uI64"
           "ukIOKAlCDsmrTsoITsnpAg64KY7J207J2YIHx6fCDqsIAgMTguNDAg7JeQ7ISc"
           "IDIuOTgg66GcIOqwkOyGjO2VqeuLiOuLpC4KLSDqt7jrnpjshJwg67O06rOg7I"
           "Sc7JeQIOyggeydhCDqtazqsIQg6ri47J2064qUIOydtOugh+qyjCDrgpjrianr"
           "i4jri6QuIOuvuOyFmCAxIOydmCBSZWdpb24gUjQzIOq1rOqwhCDquLjsnbQgMC"
           "4wNTE2NTEg6rG0L+uFhOyXkCDiiJrPhsyCIOulvCDqs7HtlZjrqbQgMC4wNTMx"
           "NzQg6rG0L+uFhOydtOqzoCwg6rWw7KeRIOyDgeq0gOyXkCDsnZjtlZwg7ISx67"
           "aE7J2AIOqzseyFiOydtCDslYTri4jrnbwg44CM7KeA7Jet7J2EIOuqqO2YleyX"
           "kCDtj6ztlajtlZjqsbDrgpgg6rWw7KeRIO2RnOykgOyYpOywqOuhnCDri6Tsi5"
           "wg7IKw7Lac7ZWY652844CN64qUIOyjvOyEneycvOuhnCDsoIHsirXri4jri6Qu"
           "IOuLpOunjCDsnbQg6rOx7IWI7J2AIOuMgOyhsOyaqeyeheuLiOuLpC4g67Cx67"
           "aE7JyEIOu2gO2KuOyKpO2KuOueqeydgCDqs4Tslb3snYQg6re464yA66GcIOuL"
           "pOyLnCDrvZHslYQg67aE7IKwID0g7Y+J6regIOqwgOygleydhCDsk7Dsp4Ag7J"
           "WK6rOgLCDshLjqt7jrqLztirggOOqwnOydmCDrtoDtirjsiqTtirjrnqkg7ZGc"
           "7KSA7Jik7LCo6rCAIO2PrOyVhOyGoSDqs7Xsi50g4oiaKOyCrOqzoCDtlakpIM"
           "O3IOuFuOy2nCDtlansnZgg7Y+J6regIDEuMDTrsLDrnbwg6rO87IKw7Y+s66eM"
           "7YG87J2AIOydtOuvuCDqtazqsIQg6ri47J207JeQIOuTpOyWtCDsnojsirXri4"
           "jri6QuIOKIms+GzIIg66W8IOqzse2VtOyVvCDtlZjripQg6rKD7J2AIO2PrOyV"
           "hOyGoSDqsIDsoJXsnLzroZwg6rOE7IKw7ZWcIMKxMiBTRSDqtazqsITsnoXri4"
           "jri6QuCi0g6rOE7JW9IOyImOqwgCDsoIHsnYAg7KeA7Jet7J287IiY66GdIOq4"
           "sOykgOq1rOqwhOydtCDrhJPslrQg6re4IOuwluycvOuhnCDrspfslrTrgpjquL"
           "Ag7Ja066C17Iq164uI64ukLiDqs4Tslb0gMSwzMjbqsbTsnbggUjQzIOydgCDs"
           "p4Dsl63rs4Qg7Y+J6reg7J20ICswLjAxNjAg7J24642w64+EIOq4sOykgOq1rO"
           "qwhCDslYjsnoXri4jri6QuIOuwmOuMgOuhnCAxNjAsNjAx6rG07J24IFIyNCDr"
           "ipQg4oiSMC4wMTUzIOycvOuhnCDsoIjrjJPqsJLsnbQg642UIOyekeydgOuNsC"
           "DquLDspIDqtazqsIQg67CW7J6F64uI64ukLgoK7L2U65Oc6rCAIOuLteydhCDr"
           "p4zrk5zripQg67Cp7IudIOKAlCDqsJnsnYAg7Iud7J2EIDLtmowg7KCB7ZWp7Z"
           "W0IO2RnOykgOyYpOywqOunjCDqtZDssrTtlZwg6rKD7J20IOuwsOyImOydtOqz"
           "oCwg7J207YOI64+EIOyelOywqOulvCDsp4Dsl63snLzroZwg66y27J2AIO2Pie"
           "q3oOydhCDjgIzqsJnsnYAg7J6U7LCo66W8IOustOyekeychOuhnCDrsLDrtoTt"
           "lojsnYQg65WMIOuCmOyYpOuKlCDtj4nqt6DjgI0gMSwwMDDqsJzsmYAg67mE6r"
           "WQ7ZWcIOqyg+ydtCDquLDspIDqtazqsITsnoXri4jri6QuIO2VnOyqveydgCDs"
           "grDtj6zrpbwsIOuLpOuluCDsqr3snYAg7Jyg7ZqoIO2RnOuzuCDsiJjrpbwg7L"
           "ih7KCV7ZWp64uI64ukLgo=",
           }

#  힌트는 (한 문장 힌트, 튜터에게 물을 것) 짝입니다 — 채점이 ❌ 인 항목에만 나옵니다
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
           "eyJtMSI6eyLsp4Dsl60gUjQzIOydmCA5NSUg6rWs6rCEIO2VmO2VnCAo6rG0L+"
           "uFhCkiOlsi7J6s7ZGc67O4IOyCrOqzoOycqOydhCDrp4zrk6Ag64KY64iX7IWI"
           "7J2YIOu2hOuqqOulvCDrqLzsoIAg7ZmV7J247ZWY7IS47JqUIOKAlCDqs4Tslb"
           "0g7IiY66GcIOuCmOuIhOuptCDri6jsnITqsIAg6rG0L+qzhOyVveydtCDrkJjq"
           "s6Ag6rCS7J20IOygiOuwmCDslYTrnpjroZwg6rCQ7IaM7ZWp64uI64ukLiDrto"
           "TrqqjripQg6re4IOyerO2RnOuzuOydmCDrhbjstpwg7ZWp7J6F64uI64ukIiwi"
           "7J20IOy9lOuTnOqwgCDsnqztkZzrs7jrp4jri6Qg7IKs6rOg7Jyo7J2YIOu2hO"
           "uqqOuhnCDsgqzsmqntlZwg6rCS7J20IOustOyXh+yduOqwgCJdLCLsp4Dsl60g"
           "UjQzIOydmCA5NSUg6rWs6rCEIOyDge2VnCAo6rG0L+uFhCkiOlsi7ZGc67O4IO"
           "2VmOuCmOulvCDrqocg6rG07Jy866GcIOy2lOy2nO2WiOuKlOyngCDtmZXsnbjt"
           "lZjshLjsmpQg4oCUIOyerO2RnOuzuCDtgazquLDqsIAg7JuQIO2RnOuzuOuztO"
           "uLpCDtgazrqbQg67OA64+ZIOuylOychOqwgCDsi6TsoJzrs7Tri6Qg7J6R6rKM"
           "IOy2lOygleuQmOyWtCDsg4HtlZzsnbQg7KKB6rKMIOyCsOy2nOuQqeuLiOuLpC"
           "4g6re466as6rOgIOq1rOqwhOydgCDsoJXroKztlZwg7J6s7ZGc67O47J2YIOyc"
           "hOyqvSAyLjUlIOyngOygkOyXkOyEnCDsoIjri6jtlanri4jri6QiLCLsnbQg7L"
           "2U65Oc6rCAIO2RnOuzuCDtlZjrgpjrpbwg66qHIOqxtOycvOuhnCDstpTstpzt"
           "lojqs6Ag7IOB7ZWc7J2EIOyWtOuKkCDrsLHrtoTsnITsl5DshJwg7KCI64uo7Z"
           "aI64qU6rCAIl0sInBsb3QiOlsi6rCA66Gc7LaV7J20IOyXsOqwhCDsgqzqs6Ds"
           "nKjsnbTqs6Ag7IS466Gc7LaV7J20IOyEuOq3uOuovO2KuCDsnbTrpoQgOOqwnO"
           "yduOyngCDCtyDshLjqt7jrqLztirjrp4jri6Qg6rWs6rCE7J2YIOuRkCDrgZ3s"
           "nYQg7J6H64qUIOqwgOuhnCDrp4nrjIDrpbwg6re47JeI64qU7KeAIMK3IOygkO"
           "y2lOyglSDrp4jsu6Trpbwg7ZGc7Iuc7ZaI64qU7KeAIMK3IOyghOyytCDqs4Ts"
           "lb0g6riw7KSA7ISgIOychOy5mOyXkCDshLjroZwg7KCQ7ISg7J2EIOq3uOyXiO"
           "uKlOyngCDCtyDrkZAg7LaV7JeQIOudvOuyqOydhCDri6zslZjripTsp4Ag7LCo"
           "66GA66GcIO2ZleyduO2VmOyEuOyalC4g6re466a87J2AIO2RnOulvCDrp4zrk5"
           "zripQg6rKD6rO8IOqwmeydgCDtlITroaztlITtirjsl5Ag7ZWo6ruYIOyngOyL"
           "nO2VqeuLiOuLpCIsIuuCtCDqt7jrprzsnZgg7Ja064qQIOy2lcK37Ja064qQIO"
           "yEoOydtCDquLDspIDqs7wg64uk66W47KeAIl0sImVycm9yIjpbIuyYpOulmCDr"
           "rLjsnqUg7LKrIO2WieydhCDqt7jrjIDroZwgQUkg7Yqc7YSw7JeQIOuztOyXrC"
           "Dso7zqs6AsIOykgOu5hCDshYDsnbQg7IOd7ISx7ZWcIOydtOumhChkZiDCtyBT"
           "RUdNRU5UUyDCtyBCIMK3IFNFRUQgwrcgTEFNX0FMTCnsnYAg64uk7IucIOyDne"
           "yEse2VmOyngCDrp5Dqs6Ag6re464yA66GcIOyCrOyaqe2VmOudvOqzoCDrqoXs"
           "i5ztlZjshLjsmpQiLCLsnbQg7Jik66WY6rCAIOyWtOuKkCDtlonsl5DshJwg7J"
           "mcIOuCrOuKlOyngCJdfSwibTIiOnsi6rWQ7LCo6rKA7KadIOydtO2DiOuPhOqw"
           "gCDqsIDsnqUg64Ku7J2AIOyngOygkOydmCDsnbTtg4jrj4QiOlsi44CMQUkg7L"
           "2U65OcIOyLnOyekSDtlonjgI0g7JWE656YIOy9lOuTnOuKlCDqs4TsiJgg6rK9"
           "66Gc6rmM7KeA66eMIOunjOuTreuLiOuLpCDigJQg6rWQ7LCo6rKA7Kad7J2AIO"
           "yDiOuhnCDsp4Dsi5ztlbTslbwg7ZWp64uI64ukLiDrtoTtlaDsnYQg7ZWcIOuy"
           "iCDrp4zrk6TslrQgYWxwaGEgNuqwnOqwgCDtlajqu5gg7IKs7Jqp7ZWY6rKMIO"
           "2VmOqzoCwg7KCQ7IiY64qUIOygge2VqeyXkCDsgqzsmqntlZjsp4Ag7JWK7J2A"
           "IOqygOymnSDtj7Trk5zsl5DshJzrp4wg7IKw7Lac7ZWY7IS47JqULiDtm4jroK"
           "gg642w7J207YSw7JeQ7IScIOyCsOy2nO2VnCDsoJDsiJjroZwgYWxwaGEg66W8"
           "IOqzoOultOuptCDtjKjrhJDti7Ag7ZWt7J20IOqwgOyepSDslb3tlZwgYWxwaG"
           "Eg6rCAIO2VreyDgSDstZzsoIDsoJDsnLzroZwg7IKw7Lac65Cp64uI64ukIiwi"
           "7J20IOy9lOuTnOqwgCBhbHBoYSDrpbwg6rOg66W8IOygkOyImOulvCDslrTrip"
           "Ag642w7J207YSw7JeQ7IScIOyCsOy2nO2WiOuKlOyngCJdLCLrsozsoJDsnbQg"
           "6rGw7J2YIOyXhuuKlCBhbHBoYSDsl5DshJzsnZgg7ZGc7KSA7ZmUIOqzhOyImC"
           "A0MOqwnOydmCDtgazquLAg7ZWpIjpbWyIx7LCoIOKAlCDsoIHtlansl5Ag7KCE"
           "64us7ZWcIO2WieugrCDtlZwg7ZaJ66eMIO2ZleyduO2VmOyEuOyalDogZml0KC"
           "4uLikg7J2YIOyyqyDsnbjsnpDsnoXri4jri6QiLCIy7LCoIOKAlCDqt7gg7ZaJ"
           "66Cs7J2AIOyXtOuniOuLpCDri6jsnITqsIAg64uk66aF64uI64ukLiDtjKjrhJ"
           "Dti7Ag7ZWt7J2AIOqzhOyImOydmCDtgazquLDsl5Ag67aA6rO865CY66+A66Gc"
           "LCDjgIzspIDruYQg7IWA44CN7J20IOunjOuTpOyWtCDrkZQg7ZGc7KSA7ZmU7Z"
           "WcIO2WieugrOuhnCDrsJTqv5Qg7KCE64us7ZWY7IS47JqUIl0sIuydtCDsvZTr"
           "k5zqsIAg7KCB7ZWp7JeQIOuEmOq4tCDtlonroKzsnZgg7Je065Ok7J20IOyEnO"
           "uhnCDqsJnsnYAg64uo7JyEIOychOyXkCDsnojripTsp4AiXSwicGxvdCI6WyLs"
           "mKTrpbjsqr0g7Yyo64SQ7J2EIO2ZleyduO2VmOyEuOyalCDigJQg6rCA66Gc7L"
           "aV7JeQIGFscGhhIDbqsJzrpbwg66y47J6Q7Je0IOudvOuyqOuhnCDrsLDsuZjt"
           "lZjqs6Ag7IS466Gc7LaV7JeQIOq1kOywqOqygOymnSDsnbTtg4jrj4Trpbwg65"
           "GUIOyEuOuhnCDrp4nrjIAgNuqwnOyduOyngCDCtyDrp4nrjIDrpbwgMCDsl5Ds"
           "hJwg7Iuc7J6R7ZWY6rKMIOuRkOyXiOuKlOyngCDCtyDrkZAg7LaV7JeQIOudvO"
           "uyqOydhCDri6zslZjripTsp4Ag7LCo66GA66GcIO2ZleyduO2VqeuLiOuLpC4g"
           "6re466a87J2AIOqwkuydhCDrp4zrk5zripQg6rKD6rO8IOqwmeydgCDtlITroa"
           "ztlITtirjsl5Ag7ZWo6ruYIOyngOyLnO2VqeuLiOuLpCIsIuuCtCDqt7jrprzs"
           "nZgg7Ja064qQIOy2lcK37Ja064qQIOunieuMgOqwgCDquLDspIDqs7wg64uk66"
           "W47KeAIl0sImVycm9yIjpbIuyYpOulmCDrrLjsnqUg7LKrIO2WieydhCDqt7jr"
           "jIDroZwgQUkg7Yqc7YSw7JeQIOuztOyXrCDso7zqs6AsIOykgOu5hCDshYDsnb"
           "Qg7IOd7ISx7ZWcIOydtOumhChkZiDCtyBYZGVzaWduIMK3IFh6IMK3IENPTFMg"
           "wrcgeV9yYXRlIMK3IHdfZXhwIMK3IEFMUEhBUyDCtyBTRUVEKeydgCDri6Tsi5"
           "wg7IOd7ISx7ZWY7KeAIOunkOqzoCDqt7jrjIDroZwg7IKs7Jqp7ZWY65286rOg"
           "IOuqheyLnO2VmOyEuOyalCIsIuydtCDsmKTrpZjqsIAg7Ja064qQIO2WieyXkO"
           "yEnCDsmZwg64Ks64qU7KeAIl19LCJtMyI6eyLtmZXsnqUg66qo7ZiVKGZ1bGwp"
           "7J2YIO2PtOuTnCDtj4nqt6Ag7J207YOI64+EIjpbIuydtCDrr7jshZjsnYAg7Z"
           "6M7Yq4IOyXhuydtCDtlZnsg53snbQg7KeB7KCRIOuqheyEuO2VmOuKlCDrr7js"
           "hZjsnoXri4jri6Qg4oCUIOuCtCBQUk9NUFQg66W8IOuLpOyLnCDsnb3qs6AgM+"
           "qwgOyngOqwgCDri6Qg7KCB7ZiAIOyeiOuKlOyngCDtmZXsnbjtlZjshLjsmpQu"
           "IOygkOyImOulvCDslrTripAg7Y+065Oc7JeQ7IScIOyCsOy2nO2VoOyngCwg6r"
           "KA7KadIO2PtOuTnOydmCDsmIjsuKHsnYQg7Ja064qQIOuLqOychOuhnCDrsJvs"
           "nYTsp4AsIO2PtOuTnCA16rCc66W8IOyWtOuWu+qyjCDtlZjrgpjroZwg7JqU7J"
           "W97ZWg7KeA7J6F64uI64ukIiwi64K0IOyYiOy4oeydtCDruYjrj4Qg7YOA6rmD"
           "6rO8IOqwmeydgCDri6jsnIQg7JyE7JeQIOyeiOuKlOqwgCJdLCJjb3JlIC0gZn"
           "VsbCDsp53sp4DsnYAg7LCo7J207J2YIO2Pieq3oCAvIO2RnOykgOyYpOywqCAo"
           "67CwKSI6WyLssKjsnbTrpbwg66eM65Oc64qUIOyInOyEnOulvCBQUk9NUFQg7J"
           "eQIOuqheyLnO2WiOuKlOyngCDtmZXsnbjtlZjshLjsmpQg4oCUIOyWtOuKkCDt"
           "m4Trs7Tsl5DshJwg7Ja064qQIO2bhOuztOulvCDrubzripTsp4AsIOqwmeydgC"
           "Dtj7Trk5zrgbzrpqwg67m864qU7KeALCDrgpjriIwg7ZGc7KSA7Jik7LCo66W8"
           "IOustOyXh+yXkOyEnCDsgrDstpztlZjripTsp4AgM+qwgOyngOyeheuLiOuLpC"
           "IsIuuCtOqwgCDrgpjriIgg7ZGc7KSA7Jik7LCo6rCAIOywqOydtCA16rCc7JeQ"
           "7IScIOyCsOy2nOuQnCDqsoPsnbjqsIAiXSwicGxvdCI6WyLrkZAg7Yyo64SQ7J"
           "eQIOustOyXh+ydhCDrsLDsuZjtlaDsp4AgUFJPTVBUIOyXkCDri6Qg7KCB7JeI"
           "64qU7KeAIO2ZleyduO2VmOyEuOyalCDigJQg7KCQ6rO8IOunieuMgOulvCDrqo"
           "cg6rCcIOq3uOumtOyngCwg6rCA66Gc7ISgIDLqsJzsnZgg7JyE7LmYLCDrkZAg"
           "7Yyo64SQ7J2YIOqwgOuhnOy2lcK37IS466Gc7LaVIOydtOumhOyeheuLiOuLpC"
           "IsIuuCtCDqt7jrprzsnZgg7Ja064qQIOy2lcK37Ja064qQIOunieuMgOqwgCDq"
           "uLDspIDqs7wg64uk66W47KeAIl0sImVycm9yIjpbIuyYpOulmCDrrLjsnqUg7L"
           "KrIO2WieunjCDrlrzslrQg7J297Jy87IS47JqUIOKAlCDspIDruYQg7IWA7J20"
           "IOyDneyEse2VnCDsnbTrpoQoZGYgwrcgRk9MRFMgwrcgRk9STVVMQVMgwrcgeV"
           "9yYXRlIMK3IHdfZXhwIMK3IExBTV9BTEwp7J2AIOuLpOyLnCDsg53shLHtlZjs"
           "p4Ag66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqntlanri4jri6QiLCLsnbQg7Jik66"
           "WY6rCAIOyWtOuKkCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwi6rO87KCc"
           "Ijp7Iu2VoOymneygkOyImCDqs4TsiJjsnZgg7ZGc7KSA7Jik7LCoIOuwsOyImC"
           "DigJQg6rWw7KeRIMO3IO2PrOyVhOyGoSAo67CwKSI6WyLrsLDsiJjqsIAgMSDr"
           "toDqt7zsnbTrqbQg6rWw7KeRIOudvOuyqOydhCDsoITri6ztlZjsp4Ag7JWK7J"
           "2AIOqyg+yeheuLiOuLpCDigJQgYGNvdl90eXBlPVwiY2x1c3RlclwiYCDsl5Ag"
           "YGNvdl9rd2RzPXtcImdyb3Vwc1wiOiBSRUd9YCDrpbwg7ZWo6ruYIOyghOuLrO"
           "2WiOuKlOyngCwg6re4IGBSRUdgIOqwgCDsp4Dsl60g652867Ko7J247KeAIO2Z"
           "leyduO2VmOyEuOyalC4g67Cw7IiY64qUIOqwmeydgCDqs4TsiJjsl5DshJwg65"
           "GQIO2RnOykgOyYpOywqOulvCDrgpjriIgg6rCS7J206528LCBgSEMwYCDqsJns"
           "nYAg6rCV6rG0IO2RnOykgOyYpOywqOuhnCDsgrDstpztlZjrqbQgMS4wMCDrto"
           "Dqt7zsnbQg64KY7Ji164uI64ukIiwi64K06rCAIOuCmOuIiCDrkZAg7ZGc7KSA"
           "7Jik7LCo6rCAIOqwmeydgCDqs4TsiJjsnZgg6rKD7J246rCAIl0sIuu2hOyCsM"
           "O37Y+J6reg7J20IDEg7JyE7Kq97J24IOy5uCDsiJggKOy5uCkiOlsi7KCE7LK0"
           "IOqzhOyVveydhCDtlZwg67aA67aE7KeR7ZWp7Jy866GcIOyCsOy2nO2VmOuptC"
           "DrtoTsgrDqs7wg7Y+J6reg7J20IOqxsOydmCDqsJnsnYAg6rCS7J20IOuQqeuL"
           "iOuLpCDigJQg6rOE7JW9IDTqsbQg7KSRIDPqsbTsnbQg7IKs6rOgIDDqsbTsnb"
           "TquLAg65WM66y47J6F64uI64ukLiDjgIzrp4zrk6Qg6rKD44CN7JeQIOyngOyg"
           "leuQnCDqtazqsIQg7IiY66eM7YG8IOyYiOy4oeqwkiDsiJzsnLzroZwg67aE7Z"
           "Wg7ZW0IOq1rOqwhOuniOuLpCDqs4TsgrDtlZjqs6AsIOOAjOu2hOyCsCDDtyDt"
           "j4nqt6DjgI3snbQgMSDsnYQg64SY64qUIOq1rOqwhOydmCDqsJzsiJjrpbwg7K"
           "eR6rOE7ZWY7IS47JqULiDqtazqsIQg7IiY66W8IOuKmOumrOqxsOuCmCDspITs"
           "nbTrqbQg6rOE7IKwIOqysOqzvOqwgCDri6zrnbzsp5Hri4jri6QiLCLrgrTqsI"
           "Ag67aE7IKw6rO8IO2Pieq3oOydhCDsgrDstpztlZwg7ZGc67O47J20IOuqhyDq"
           "sJwg6rWs6rCE7J246rCAIl0sInBsb3QiOlsi6rCA66Gc7LaV7J20IOyngOyXre"
           "uzhCDtj4nqt6DsnbTqs6Ag7IS466Gc7LaV7J20IOyngOyXrSAyMuqwnOyduOyn"
           "gCDCtyDsp4Dsl63rp4jri6Qg6riw7KSA6rWs6rCE7J2YIOuRkCDrgZ3snYQg7J"
           "6H64qUIOqwgOuhnCDrp4nrjIDrpbwg6re47JeI64qU7KeAIMK3IOyLpOygnCDs"
           "p4Dsl63rs4Qg7Y+J6regIOychOy5mOyXkCDsoJDsnYQg7ZWY64KY7JSpIO2RnO"
           "yLnO2WiOuKlOyngCDCtyAwIOychOy5mOyXkCDshLjroZzshKDsnYQg6re47JeI"
           "64qU7KeAIMK3IOuRkCDstpXsl5Ag652867Ko7J2EIOuLrOyVmOuKlOyngCDssK"
           "jroYDroZwg7ZmV7J247ZWY7IS47JqUIiwi64K0IOq3uOumvOydmCDslrTripAg"
           "7LaVwrfslrTripAg7ISg7J20IOq4sOykgOqzvCDri6Trpbjsp4AiXSwiZXJyb3"
           "IiOlsi7Jik66WYIOusuOyepSDssqsg7ZaJ66eMIOuWvOyWtCDsnb3snLzshLjs"
           "mpQg4oCUIOykgOu5hCDshYDsnbQg7IOd7ISx7ZWcIOydtOumhChkZiDCtyBGX0"
           "NPUkUgwrcgUkVHIMK3IEJPT1QgwrcgQiDCtyBTRUVEKeydgCDri6Tsi5wg7IOd"
           "7ISx7ZWY7KeAIOunkOqzoCDqt7jrjIDroZwg7IKs7Jqp7ZWp64uI64ukIiwi7J"
           "20IOyYpOulmOqwgCDslrTripAg7ZaJ7JeQ7IScIOyZnCDrgqzripTsp4AiXX19")
_ASK = {"m1": "full", "m2": "blank",       # 뒤로 갈수록 걷힙니다 — 미션 3 은 항목만 남습니다
        "m3": "self", "과제": "bare"}
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 항목 (문장으로 만드는 것은 내 몫입니다)",
             "self": "◼ 스스로 점검할 항목 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " 「AI 코드 시작 행」 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. 「AI 코드 시작 행」 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : 「AI 코드 시작 행」 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 결정 16 이 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 행만 고릅니다 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 행이 하나라도 있으면 그
    행들만 보고, 그런 행이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 경로를 열어 두지 않으면
    `plt.hlines(...)` 로 화면에 분명히 그어 놓은 구간 막대가 「막대가 하나도 없다」 로 채점돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax):
    """축의 선을 곡선(점 10개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다"""
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _pt_sets(ax):
    """산점도의 점 좌표 — `scatter`(PathCollection)와 `plot(x, y, "o")` 를 함께 셉니다"""
    marker = _marker_lines(ax)
    pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
          for c in ax.collections if not hasattr(c, "get_segments")]   # 선 모음은 점으로 세지 않습니다
    pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in marker]
    return marker, pt_sets


def _spans(ax):
    """가로로 누운 구간 막대를 (왼쪽 끝, 오른쪽 끝) 짝으로 모읍니다.

    `hlines`(LineCollection) · `plot([lo, hi], [y, y])`(Line2D) · `errorbar(xerr=…)` ·
    `barh`(Rectangle) 어느 방법으로 그렸어도 같은 자리에서 셉니다 — 그리는 방법을 하나로
    못 박으면 AI 가 고른 다른 방법이 「막대가 없다」 로 읽혀 학생이 애먼 곳을 고칩니다.
    세로로 선 토막(수직 기준선)은 구간이 아니라 여기서 빠집니다.
    """
    picked = []
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1)) for ln in ax.lines]
    cands += _seg_lines(ax)
    for xd, yd in cands:
        if xd.size != 2 or yd.size != 2:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        if not np.isclose(yd[0], yd[1]) or float(xd.max() - xd.min()) <= 0:
            continue                                   # 가로로 누워 있고 길이가 있는 토막만
        picked.append((float(xd.min()), float(xd.max())))
    for p in getattr(ax, "patches", []):               # `barh(left=lo, width=hi-lo)` 로 그린 구간
        try:
            x0, bar_w, heights = float(p.get_x()), float(p.get_width()), float(p.get_height())
        except (AttributeError, TypeError, ValueError):
            continue
        if bar_w > 0 and heights > 0 and np.isfinite(x0):
            picked.append((x0, x0 + bar_w))
    return picked


def _near(val, ref, factor=1.5):
    """실측값이 기준의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {want}개)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])
                trough = float(xd[yd.argmin()])
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(충격반응) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec and len(heights) != spec["n"]:
            gaps.append(f"막대가 {len(heights)}개다 (기준 {spec['n']}개)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "interval":
        spans = _spans(ax)
        if "n_spans" in spec:
            n_lo, n_hi = spec["n_spans"]
            if not (n_lo <= len(spans) <= n_hi):
                gaps.append(f"가로 구간 막대가 {len(spans)}개다 (기준 {n_lo}개 — 항목 하나에 막대"
                            f" 하나, 강조로 덧그린 막대는 {n_hi - n_lo}개까지 넘어갑니다)")
        if spans:
            widths = [right - left for left, right in spans]
            left_end, right_end = min(left for left, _ in spans), max(right for _, right in spans)
            if "span_max" in spec:
                lo, hi = spec["span_max"]
                if not (lo <= max(widths) <= hi):
                    gaps.append(f"가장 넓은 막대의 폭이 {max(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "span_min" in spec:
                lo, hi = spec["span_min"]
                if not (lo <= min(widths) <= hi):
                    gaps.append(f"가장 좁은 막대의 폭이 {min(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(left_end, lo) and _near(right_end, hi)):
                    gaps.append(f"막대가 차지하는 가로축 값이 {left_end:.4g}~{right_end:.4g} 다"
                                f" (기준 {lo:g}~{hi:g})")
        if "n_points" in spec:
            _, pt_sets = _pt_sets(ax)
            n_pts = int(sum(p.shape[0] for p in pt_sets))
            n_lo, n_hi = spec["n_points"]
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점추정 마커가 {n_pts}개다 (기준 {n_lo}개 — 항목 하나에 마커 하나)")
        if "vline_x" in spec:
            _, vline = _lines(ax)
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append(f"기준 구간({v0:g} ~ {v1:g}) 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "scatter":
        marker, pt_sets = _pt_sets(ax)
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            cell = spec["n_points"]
            n_lo, n_hi = cell if isinstance(cell, tuple) else (cell, cell)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 항목 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in marker], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


def _ref_plot_m1():
    """미션 1 의 참조 그림 — 「준비 셀」의 df · SEGMENTS · SEED · LAM_ALL 로 세그먼트별 구간 막대를 그립니다.

    참조 그림은 눈으로 대 보는 용도라 재표본을 300벌만 뽑습니다(채점에 쓰는 값이 아닙니다).
    """
    d, cells, seed, ref = _get("df"), _get("SEGMENTS"), _get("SEED"), _get("LAM_ALL")
    if d is None or not cells or seed is None or ref is None:
        return None
    name, left, right, n_pts = [], [], [], []
    for cell, mask in cells.items():
        cell_tbl = d[mask]
        y = cell_tbl["ClaimNb"].to_numpy(dtype=float)
        E = cell_tbl["Exposure"].to_numpy(dtype=float)
        rng = np.random.default_rng(seed)
        resample = np.empty(300)
        for b in range(300):
            i = rng.integers(0, len(y), len(y))
            resample[b] = y[i].sum() / E[i].sum()
        name.append(cell)
        left.append(float(np.percentile(resample, 2.5)))
        right.append(float(np.percentile(resample, 97.5)))
        n_pts.append(float(y.sum() / E.sum()))
    ypos = np.arange(len(name))[::-1]
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.6))
    a = f.add_subplot(111)
    a.hlines(ypos, left, right, color=COL["하늘색"], lw=4)
    a.plot(n_pts, ypos, "o", color=COL["남색"])
    a.axvline(ref, color=COL["회색"], ls=":")
    a.set_yticks(ypos)
    a.set_yticklabels(name)
    a.set_xlabel("연간 사고율 (건/년)")
    a.set_ylabel("요율표의 세그먼트")
    return f


def _ref_plot_m2():
    """미션 2 의 참조 그림 — 「준비 셀」의 Xz · y_rate · w_exp · ALPHAS 로 alpha 별 이탈도 막대를 그립니다.

    참조 그림은 눈으로 대 보는 용도라 계약 20만 건만 뽑아 3-fold 로 잽니다(채점에 쓰는 값이
    아닙니다). 전체 678,013건을 5-fold 로 다시 재면 채점 한 번마다 열 몇 초가 더 걸립니다.
    """
    Z, y, w, grid = _get("Xz"), _get("y_rate"), _get("w_exp"), _get("ALPHAS")
    if Z is None or y is None or w is None or not grid:
        return None
    rng = np.random.default_rng(_get("SEED") or 47)
    picked = rng.choice(len(y), min(200000, len(y)), replace=False)
    Z, y, w = Z[picked], y[picked], w[picked]
    kf = KFold(3, shuffle=True, random_state=47)
    heights, err = [], []
    for penalty in grid:
        fold_scores = []
        for tr, te in kf.split(Z):
            m = PoissonRegressor(alpha=penalty, max_iter=300, tol=1e-6).fit(
                Z[tr], y[tr], sample_weight=w[tr])
            fold_scores.append(mean_poisson_deviance(y[te], m.predict(Z[te]), sample_weight=w[te]))
        heights.append(float(np.mean(fold_scores)))
        err.append(float(np.std(fold_scores, ddof=1) / np.sqrt(len(fold_scores))))
    f = _orig_figure(figsize=(FIG_W * 0.55, 3.4))
    a = f.add_subplot(111)
    a.bar([f"{penalty:g}" for penalty in grid], heights, yerr=err,
          color=COL["하늘색"], ecolor=COL["남색"], capsize=3)
    a.set_xlabel("정규화 강도 alpha")
    a.set_ylabel("교차검증 포아송 이탈도")
    return f


def _ref_plot_m3():
    """미션 3 의 참조 그림 — 「준비 셀」의 FOLDS · FORMULAS 로 후보별 폴드 점수와 짝지은 차이를 그립니다.

    참조 그림은 눈으로 대 보는 용도라 계약 25만 건만 뽑아 다시 다섯 폴드로 나눠 잽니다(채점에
    쓰는 값이 아닙니다). 678,013건을 그대로 쓰면 참조 그림 한 장에 40초가 더 걸립니다.
    """
    d, formulas, y, w = _get("df"), _get("FORMULAS"), _get("y_rate"), _get("w_exp")
    if d is None or not formulas or y is None or w is None:
        return None
    rng = np.random.default_rng(_get("SEED") or 47)
    picked = rng.choice(len(d), min(250000, len(d)), replace=False)
    small_tbl, y, w = d.iloc[picked].reset_index(drop=True), y[picked], w[picked]
    folds = list(KFold(5, shuffle=True, random_state=47).split(small_tbl))
    scores = {}
    for name, formula in formulas.items():
        cell = []
        for tr, te in folds:
            tr_idx, va_idx = small_tbl.iloc[tr], small_tbl.iloc[te]
            fit = smf.glm(formula=formula, data=tr_idx, family=sm.families.Poisson(),
                          offset=np.log(tr_idx["Exposure"])).fit()
            y_hat = np.asarray(fit.predict(va_idx, offset=np.zeros(len(va_idx))))
            cell.append(mean_poisson_deviance(y[te], y_hat, sample_weight=w[te]))
        scores[name] = np.array(cell)
    if not {"core", "full"} <= set(scores):
        return None
    diff_arr = scores["core"] - scores["full"]
    f = _orig_figure(figsize=(FIG_W, 3.4))
    left = f.add_subplot(1, 2, 1)
    for i, name in enumerate(scores):
        left.plot([i] * len(scores[name]), scores[name], "o", color=COL["연회색"])
        left.plot([i], [scores[name].mean()], "_", ms=20, mew=3, color=COL["남색"])
    left.set_xticks(range(len(scores)))
    left.set_xticklabels(list(scores))
    left.set_xlim(-0.5, len(scores) - 0.5)
    left.set_xlabel("후보 모형")
    left.set_ylabel("교차검증 포아송 이탈도")
    right = f.add_subplot(1, 2, 2)
    right.bar(range(len(diff_arr)), diff_arr, color=COL["하늘색"])
    right.axhline(0, color=COL["회색"])
    right.axhline(float(diff_arr.mean()), color=COL["남색"], ls=":")
    right.set_xticks(range(len(diff_arr)))
    right.set_xticklabels([f"폴드 {i + 1}" for i in range(len(diff_arr))])
    right.set_xlabel("폴드")
    right.set_ylabel("core - full 이탈도 차이")
    return f


def _ref_plot_task():
    """과제의 참조 그림 — 「준비 셀」의 df · F_CORE · REG 로 지역별 잔차와 무작위 재배치 기준구간을 그립니다.

    참조 그림은 눈으로 대 보는 용도라 계약 20만 건만 뽑아 재배치도 300벌만 돕니다(채점에 쓰는
    값이 아닙니다). 678,013건을 그대로 쓰면 참조 그림 한 장에 열 몇 초가 더 걸립니다.
    """
    d, formula, label = _get("df"), _get("F_CORE"), _get("REG")
    if d is None or not formula or label is None:
        return None
    rng = np.random.default_rng(_get("SEED") or 47)
    sampled = np.sort(rng.choice(len(d), min(200000, len(d)), replace=False))
    small, small_label = d.iloc[sampled], np.asarray(label)[sampled]
    fit = smf.glm(formula=formula, data=small, family=sm.families.Poisson(),
                  offset=np.log(small["Exposure"])).fit()
    resid = np.asarray(fit.resid_deviance, dtype=float)
    resid = resid - resid.mean()
    regions = sorted(set(small_label))
    sizes = np.array([int((small_label == r).sum()) for r in regions])
    observed = np.array([resid[small_label == r].mean() for r in regions])
    cut_points = np.cumsum(sizes)[:-1]
    collected = np.empty((300, len(regions)))
    for b in range(300):
        collected[b] = [piece.mean() for piece in np.split(rng.permutation(resid), cut_points)]
    left, right = np.percentile(collected, 2.5, axis=0), np.percentile(collected, 97.5, axis=0)
    ypos = np.arange(len(regions))[::-1]
    f = _orig_figure(figsize=(FIG_W * 0.8, 4.6))
    a = f.add_subplot(111)
    a.hlines(ypos, left, right, color=COL["하늘색"], lw=4)
    a.plot(observed, ypos, "o", color=COL["남색"])
    a.axvline(0, color=COL["회색"], ls=":")
    a.set_yticks(ypos)
    a.set_yticklabels(regions)
    a.set_xlabel("이탈도 잔차의 지역별 평균 (전체 평균을 뺀 값)")
    a.set_ylabel("지역")
    return f


_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2,   # 미션마다 참조 그림을 그리는 함수 하나
        "m3": _ref_plot_m3, "과제": _ref_plot_task}


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 문장으로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt



def _prompt1(n=70):
    """지금 셀의 지시 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    if not txt:
        return "PROMPT 입력란이 비어 있습니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    stripped = txt.replace("—", "").replace("·", "").strip()
    if stripped in _PROMPT_PLACEHOLDERS:
        return "PROMPT 입력란이 자리표시자 그대로입니다"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 행부터 채점 표를 닫는 `─` 구분선 행까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                     # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 행에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행이 있으면 그 행만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 출력하면 어느 숫자가 답인지 채점 코드가 판별하지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        hits = [v for v in (_nums("\n".join(tail)) if tail else nums) if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점 코드가 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 패널이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    if failed:
        _TRY[key] = _TRY.get(key, 0) + 1     # 결정 16 — 다음 ❌ 부터 기준 범위를 인쇄합니다
    for name, hkey in failed:
        hint = tbl.get(hkey, ("",))[0]
        first_tier = (hint,) if isinstance(hint, str) else tuple(hint)   # 함정 미션은 1차·2차로 나뉩니다
        for tiers, txt in enumerate(first_tier, 1):
            if txt:
                tail = f" {tiers}단" if len(first_tier) > 1 else ""
                print(f"   힌트({name}){tail} : {txt}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("(IPython 밖입니다 — 훅 없이 judge(태그, 출력, 그림들) 만 준비했습니다)")

## 세그먼트별 신뢰구간 길이의 차이

### 미션 1 — 세그먼트 8개 사고율의 95% 신뢰구간 산출

#### 할 일 — 미션 1 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 셀을 클릭한 채 AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
4. 실행 결과(출력)에 표시되는 채점 결과를 확인합니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문** — 요율표에 사고율을 점 하나로 적으면 「얼마나 믿을 만하냐」 에 답할 수 없는데, 세그먼트마다 그 값은 어디부터 어디까지입니까?

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Generic_forest_plot.png" width="562" height="363"/>
<p align="center"><em>▲ 점추정에 구간을 함께 표시해 여러 항목을 나란히 배치한 그림(forest plot) — 가로 막대가 각 항목의 구간, 가운데 네모가 점추정, 세로 점선이 비교 기준선이다. 오늘 세그먼트 8개로 그릴 그림도 이 형태다 (출처: Wikipedia, Generic forest plot.png)</em></p>

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 전처리 끝난 계약 표, 678,013행 | 고정 |
| `SEGMENTS` | 세그먼트 이름과 그 세그먼트의 계약을 선택하는 마스크를 짝지은 사전, 8개 | 고정 |
| `B` | 재표본 수 1,000 — 이론 2. 부트스트랩이 실무 하한으로 제시한 값 | 고정 |
| `SEED` | 난수 시드 | 고정 |
| `LAM_ALL` | 전체 계약 사고율 $\hat{\lambda}$ = 0.100614, 단위 건/년 | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **지역 R43 의 95% 구간 하한** — 단위 건/년 · 소수점 아래 5자리 · 재표본 사고율을 정렬해 아래 2.5% 지점에서 절단한 값
2. **지역 R43 의 95% 구간 상한** — 단위 건/년 · 소수점 아래 5자리 · 같은 분포의 위 2.5% 지점에서 절단한 값
3. **함께 출력할 표 하나** — 세그먼트 8개마다 `세그먼트` · `n` · `claims` · `exposure` · `lam_hat` · `lo95` · `hi95` · `se` · `폭` · `상대폭` 을 한 행으로 담습니다
   - 채점 기준에 포함되는 값은 아니며, 과제에서 같은 표를 같은 열 이름으로 다시 생성해 사용합니다
   - `se` 는 재표본 분포의 표준편차이고, 구간 길이 열 `폭` 은 `hi95` 에서 `lo95` 를 뺀 값, 상대 구간 길이 열 `상대폭` 은 그 값을 점추정 `lam_hat` 으로 나눈 값입니다
   - 표에 한 행 더 — Area C 와 Region R43 의 `상대폭` 을 나란히 출력해, 아래 「실행 전 예상」에서 고른 값과 대조합니다
4. **그래프 한 장** — 가로축은 연간 사고율(건/년), 세로축은 세그먼트 이름 8개인 구간 그림
   - 가로 막대 : 세그먼트마다 하한에서 상한까지 하나씩, 모두 8개
   - 마커 : 세그먼트마다 점추정 위치에 하나씩
   - 세로선 : `LAM_ALL` 위치에 세로축 전체를 가로지르는 선 하나
   - 가로축 이름 : 「연간 사고율 (건/년)」
   - 세로축 이름 : 「요율표의 세그먼트」

**출력 형식** — 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장이고, 표는 파일로 내보내지 않습니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` · `SEGMENTS` · `B` · `SEED` · `LAM_ALL` 은 이미 있으니 다시 만들지 말라고 적습니다.
- **재추출 방법** — 표본 하나를 원 표본과 같은 건수로 복원추출할 것, 사고율의 분모가 계약 수가 아니라 노출 합이라는 점, 난수는 `SEED` 로 고정할 것.
- **출력** — 값 2개의 이름과 단위, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로축·세로축에 무엇을 배치할지 · 가로 막대와 마커 · 세로선의 위치 · 가로축·세로축 이름.

**막히면** — AI 튜터에게 「부트스트랩 분포에서 95% 구간의 두 끝을 어느 백분위로 자르나요」 처럼 물어보세요.

#### 실행 전 예상 — 상대 구간 길이 차이의 배수

Area C 에는 계약 191,880건이 누적되어 있고 지역 R43 은 계약 1,326건뿐입니다. 아래에서 하나를 고르고 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] Area C(계약 191,880건)와 지역 R43(계약 1,326건)의 상대 구간 길이를 비교하면 R43 쪽이 대략 몇 배로 길까요?**
>
> 1. 두 배 남짓
> 2. 열 배 안팎
> 3. 백 배 이상
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

여기에 구간 길이를 정하는 것이 **계약 수**인지 **사고 건수**인지도 적어 두면 채점 뒤에 대조할 기준이 생깁니다.

<details class="lms-extra"><summary>배경 — 이론 2. 부트스트랩과의 연결, 세그먼트 8개의 선정 근거</summary>

이론 2. 부트스트랩의 재추출과 백분위 구간을 재무팀의 질문에 그대로 적용합니다. 세그먼트는 계약 수가 많은 Area A 부터 F 까지 6개와, 어제 확장 모형이 실제로 사용한 설명변수이면서 계약 수가 적은 지역 R43 · R42 2개로 선정해 두었습니다. 이론 1. 표준오차와 이론 2. 부트스트랩이 구간을 설정하지 못하는 예로 든 계약 12건인 세그먼트는 재표집 대신 정확 포아송 구간을 사용하므로 오늘 8개에서 제외했습니다.

이론 4. 신뢰구간을 이용한 의사결정은 지역 R43 을 「계약 1,326건 중 54건이 사고를 겪은」 지역으로 다루었습니다. 「준비 셀」이 출력하는 사고 57건은 그 계약에서 발생한 사고 건수의 합이라 3건이 더 많습니다. 오늘 사고율의 분자는 건수입니다.

포아송 분포에서는 분산이 평균과 같아, 구간 길이를 정하는 것은 계약 수가 아니라 사고 건수입니다. 사고율 추정값의 표준오차 SE(λ̂) = √(사고 합) ÷ 노출 합을 점추정 λ̂ 으로 나누면 노출 합이 약분되어 상대 표준오차가 1 ÷ √(사고 합) 으로 정리되기 때문입니다. 유도는 이론 1. 표준오차에 있습니다.

</details>

<details class="lms-extra"><summary>주의 — AI 튜터에 위임할 작업과 직접 확인할 항목</summary>

AI 튜터가 코드를 작성하는 위치는 마지막으로 클릭한 코드 셀입니다. 다른 셀을 클릭한 채 지시하면 의도하지 않은 셀이 수정됩니다.

왼쪽은 문법과 코드 구조 구성이라 위임해도 되는 작업이고, 오른쪽은 **틀려도 오류가 발생하지 않아** 사람이 확인해야 하는 항목입니다.

| AI 튜터에 위임할 작업 | 내가 반드시 확인할 항목 |
|---|---|
| 복원추출 반복문과 난수 생성기 호출 | 표본 하나를 원 표본과 **같은 건수**로 추출했는가 |
| 재표본마다 사고율을 계산해 배열에 집계하기 | 나눗셈의 분모가 계약 수가 아니라 **노출 합**인가 |
| 백분위로 구간을 절단하고 열 10개인 표로 집계하기 | 시드를 고정해 **다시 실행해도 같은 구간**이 산출되는가 |
| 세그먼트 8개를 한 그림에 그리기 | 산출한 구간이 **점추정을 포함하는가** |

원 표본보다 크게 추출하면 구간이 실제보다 좁게 추정됩니다. 백분위 구간은 점추정에서 좌우로 뻗은 길이가 같지 않아도 성립합니다.

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다
from sklearn.datasets import fetch_openml

# freMTPL2freq — 프랑스 자동차보험 계약별 사고 빈도 (678,013 x 12). OpenML 에서 바로 내려받습니다
df = fetch_openml(data_id=41214, as_frame=True).frame
df["IDpol"] = df["IDpol"].astype(int)
for c in ["Area", "VehBrand", "VehGas", "Region"]:
    df[c] = df[c].astype(str).str.strip("'").astype("category")

# 4일 동안 고정해 온 전처리 2가지
df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)    # 사고 5건 이상은 극소수라 상한 4 로 절사
df["Exposure"] = df["Exposure"].clip(upper=1.0)            # 노출은 1년 상한

LAM_ALL = df["ClaimNb"].sum() / df["Exposure"].sum()       # 쓸 변수 1 — 전체 계약(회사 전체) 사고율 (건/년)
B, SEED = 1000, 47                                         # 쓸 변수 2 — 재표본 수와 난수 시드

SEGMENTS = {f"Area {a}": (df["Area"] == a) for a in list("ABCDEF")}   # 쓸 변수 3 — 세그먼트 이름 -> 불리언 마스크
SEGMENTS["Region R43"] = (df["Region"] == "R43")              # 계약 수가 가장 적은 지역
SEGMENTS["Region R42"] = (df["Region"] == "R42")              # 그다음으로 계약 수가 적은 지역

for seg_name, mask in SEGMENTS.items():
    d = df[mask]
    print(f"{seg_name:12s} 계약 {len(d):7,d}건  사고 {int(d['ClaimNb'].sum()):5,d}건  "
          f"노출 {d['Exposure'].sum():10,.1f}년  "
          f"점추정 {d['ClaimNb'].sum() / d['Exposure'].sum():.6f} 건/년")

print(f"\n전체 계약 사고율 LAM_ALL = {LAM_ALL:.6f} 건/년 · 재표본 수 B = {B} · 난수 시드 SEED = {SEED}")
print("쓸 수 있는 이름 : df (전처리 끝난 계약 표) · SEGMENTS (세그먼트 이름 -> 마스크 8개) · "
      "B (재표본 수) · SEED (난수 시드) · LAM_ALL (전체 계약 사고율, 건/년)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 1 의 「AI 코드 시작 행」 아래 코드가 세그먼트마다 95% 구간의 두 끝을 만드는 방식으로 맞는 것은?**
>
> 1. 점추정에 부트스트랩 분포의 표준편차를 1.96배 해서 좌우로 똑같이 더하고 빼, 구간이 언제나 점추정을 한가운데 둔다
> 2. 복원추출 행에 그 세그먼트의 계약 수를 표본 크기로 전달해 「사고 합 ÷ 노출 합」 을 재표본 수만큼 모은 뒤, 크기순으로 순위를 매겨 아래위 2.5% 지점을 두 끝으로 삼는다
> 3. 복원추출 행의 표본 크기로 그 세그먼트의 계약 수가 아니라 표 전체의 행 수를 전달해 같은 계산을 되풀이한 뒤 아래위 2.5% 를 자른다

## 계수 40개의 정규화

### 미션 2 — 계수 40개에 대한 패널티 항 적용 (함정 미션)

#### 할 일 — 미션 2 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 「PROMPT 셀」을 **먼저 그대로 ▶** 해 전임 담당자 코드의 채점 결과(❌)를 확인합니다.
3. 아래 고칠 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
4. 미션 1 과 같은 순서로 AI 튜터에게 코드를 받아 실행하고, 채점 결과를 확인합니다. ❌ 면 힌트대로 PROMPT 를 고쳐 다시 합니다.

**이 미션의 질문** — 미션 1 의 R43 구간처럼 지역 계수도 단독으로 추정하기 어려운데, 계수 40개에 패널티 항을 적용하면 새 계약의 예측이 개선됩니까?

**쓸 수 있는 것**

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 전처리 끝난 계약 표, 678,013행 | 고정 |
| `Xdesign` | 설계행렬, 열 40개 — 원래 단위 | 고정 |
| `Xz` | 표준화한 설계행렬 — 평균 0 · 표준편차 1 | 고정 |
| `COLS` | 열 이름 40개 | 고정 |
| `y_rate` | 빈도 타깃, 단위 건/년 | 고정 |
| `w_exp` | 노출 가중, 단위 년 | 고정 |
| `ALPHAS` | 패널티 항의 세기 격자 6개 | 고정 |
| `SEED` | 난수 시드 | 고정 |

설계행렬은 범주형을 0/1 더미 열로 펼친 숫자 행렬입니다. `Area_F` 처럼 도시화 등급 F 에 해당하면 1, 아니면 0 인 열이 그 더미 열입니다.

#### 수정 전 확인 — 오답을 만드는 위치

> **[예측] 「여기부터 AI 코드」 행 아래 코드에서 답을 틀리게 만드는 곳은 어디이고 왜인가요? (`BonusMalus` 50～230 · `Area_F` 0/1)**
>
> 1. `fit(...)` 의 첫 인자가 원래 단위 그대로인 설계행렬이라 — 패널티 항이 값의 범위가 큰 열을 거의 누르지 못한다
> 2. `alpha` 격자를 여섯 개만 설정해서 — 격자가 좁아 최저점을 놓친다
> 3. `max_iter=300` 이 모자라서 — 수렴 전에 멈춘다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

#### 고칠 것

「여기부터 AI 코드」 행 아래에 전임 담당자 코드가 이미 들어 있습니다. 오류 없이 실행되는데 값이 틀렸고 그림이 없습니다. 아래 4개가 고친 뒤에 나와야 할 결과입니다. 고치기 전에 이탈도가 개선될지 그대로일지, 그리고 계수 크기 합이 어떻게 달라질지를 한 문장씩 적어 두면 뒤에서 대조할 기준이 됩니다.

1. **교차검증 이탈도가 가장 낮은 지점의 이탈도** — 소수점 아래 6자리
   - 출력할 때 값 앞에 붙일 이름은 `교차검증 이탈도가 가장 낮은 지점의 이탈도` 입니다 — 이 글자 그대로 복사해 「이름 = 값」 한 행으로 출력합니다. 채점이 이 문자열로 값줄을 찾습니다
   - 지금 : 이 값이 화면에 출력되지 않습니다
   - 고친 뒤 : 교차검증을 새로 지시합니다 — 폴드 5개로 무작위 분할한 결과를 `SEED` 로 **한 번만 만들어** `ALPHAS` 6개가 함께 사용합니다. 점수는 적합에 사용하지 않은 검증 폴드에서만, 그 폴드의 `w_exp` 를 `sample_weight` 로 전달한 `mean_poisson_deviance` 로 산출한 뒤, 그 6개 중 가장 낮은 값을 출력합니다
   - 적합은 `PoissonRegressor(alpha=…, max_iter=300, tol=1e-6)` 로 하고 `sample_weight` 에 그 학습 폴드의 `w_exp` 를 전달합니다
2. **벌점이 거의 없는 alpha 에서의 표준화 계수 40개의 크기 합** — 소수점 아래 4자리
   - 지금 : 4 에 가까운 값이 출력됩니다 — 열마다 단위가 달라 그대로 더할 수 없는 값이라, 크고 작음을 말할 수 없습니다
   - 고친 뒤 : 표준화한 `Xz` 를 전달해 `ALPHAS` 의 첫 값에서 계수 40개 절댓값 합을 출력합니다 — 단위가 같아진 값이라 더할 수 있고, 이론 6. 정규화와 교차검증의 그림이 적어 둔 **1.0880** 이 나오면 수정한 위치가 맞습니다
   - 출력할 때 값 앞에 붙일 이름은 `벌점이 거의 없는 alpha 에서의 표준화 계수 40개의 크기 합` 입니다 — 이 글자 그대로 복사해 「이름 = 값」 한 행으로 출력합니다. 채점이 이 문자열로 값줄을 찾습니다(「벌점」 은 패널티 항의 다른 이름입니다)
3. **그래프 한 장** — 패널 2개
   - 왼쪽 **계수 경로** : 가로축은 정규화 강도 alpha(로그 눈금), 세로축은 표준화 계수인 선 그래프
     - 선 : `COLS` 40개에 하나씩, 그 가운데 `BonusMalus` 와 `VehBrand_B12` 2개 선만 색을 달리합니다
     - 가로축 이름 : 「정규화 강도 alpha」 · 세로축 이름 : 「표준화 계수」
   - 오른쪽 **alpha 별 이탈도** : 가로축은 alpha 6개를 배치한 범주형 축, 세로축은 교차검증 포아송 이탈도인 막대 그래프
     - 막대 : alpha 하나에 하나씩, 모두 6개. 0 에서 시작하고 폴드 간 ±1 표준오차를 오차막대로 표시합니다 (폴드 = 위 1번의 그 분할 단위)
     - 가로축 이름 : 「정규화 강도 alpha」 · 세로축 이름 : 「교차검증 포아송 이탈도」
4. **함께 출력할 한 행** — 패널티 항이 거의 없는 첫 alpha 의 교차검증 이탈도도 같은 자릿수로 출력합니다
   - 채점 기준에 포함되는 값은 아니며, 1번 값과 비교하면 「패널티 항을 적용해 얻은 것」의 크기를 확인할 수 있습니다

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_logistic_path_001.png" width="640" height="384"/>
<p align="center"><em>▲ 계수 경로를 읽는 법 — 세로축이 계수이고 선 하나가 변수 하나다. 이 예시의 가로축은 패널티 항의 세기의 역수 C 라 오른쪽으로 갈수록 패널티 항이 약하고, 왼쪽으로 갈수록 강해져 선들이 0 으로 수렴한다 (출처: scikit-learn documentation)</em></p>

<details class="lms-extra"><summary>자세히 — 이 예시와 오늘 그릴 그림의 차이</summary>

오늘 왼쪽 패널에 그릴 그림은 가로축이 정규화 강도 alpha 라 방향이 반대입니다 — 오른쪽으로 갈수록 패널티 항이 강합니다. 그래서 선이 0 으로 수렴하는 쪽도 반대편입니다. 또 이 예시는 L1 패널티 항(Lasso)이라 계수가 정확히 0 이 되지만, 오늘 적용하는 L2 는 0 이 되지는 않고 0 에 접근하기만 합니다. 오늘 사용하는 패널티 항은 L2 하나입니다.

</details>

**통과 기준** — 값 2개가 위에 인라인 코드로 적어 둔 이름 그대로 「이름 = 값」 형태로 한 행씩 출력되고, 그래프가 막대 6개인 오른쪽 이탈도 패널을 위 요구대로 갖추면 통과입니다. 채점은 오른쪽 이탈도 패널만 확인합니다(왼쪽 계수 경로 패널은 이해를 돕는 참고용입니다).

#### 프롬프트에 넣을 것(힌트)

- **고칠 위치** — 적합에 전달하는 행렬을 무엇으로 바꿀지, 패널티 항의 세기를 무슨 점수로 고를지.
- **손대지 말 범위** — `Xdesign` · `Xz` · `COLS` · `y_rate` · `w_exp` · `ALPHAS` · `SEED` 는 이미 있으니 다시 만들지 말라고 적습니다.
- **출력** — 값 2개의 이름(위 인라인 코드 그대로)과 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 두 패널에 각각 무엇을 배치할지 · 왼쪽 가로축의 로그 눈금 · 막대 수와 오차막대 · 두 패널의 가로축·세로축 이름.

**막히면** — AI 튜터에게 「같은 분할을 alpha 6개가 함께 쓰게 하려면 KFold 를 어디서 만들어야 하나요」 처럼 물어보세요.

<details class="lms-extra"><summary>배경 — 이론 6. 정규화와 교차검증, 표준화와 표준화 계수</summary>

이론 6. 정규화와 교차검증의 **패널티 항**(L2 · Ridge 정규화)으로 계수를 0 쪽으로 수축시키고, 그 세기인 **정규화 강도 alpha** 를 선택합니다. 모형의 계수는 절편까지 41개이고, 패널티 항은 절편을 제외한 40개에 부과됩니다. L2 패널티 항은 계수에 평균 0 인 정규 사전분포를 부여한 것과 같고, 둘째 날 배운 수축을 회귀계수에 적용한 형태입니다.

**표준화**는 각 열을 평균 0 · 표준편차 1 로 바꾸어 열마다 다른 단위를 통일하는 처리입니다. 단위가 통일된 계수가 **표준화 계수**이고, 그 40개의 절댓값 합이 모형이 사용하는 계수의 총량입니다.

패널티 항은 **계수의 크기**에 부과됩니다. 값의 범위가 큰 열은 같은 효과를 내는 데 필요한 계수가 애초에 작아 패널티 항의 영향을 거의 받지 않고, 값의 범위가 작은 열만 크게 축소됩니다. **패널티 항이 공평하려면 계수의 단위가 먼저 같아야 합니다.**

표준화는 「준비 셀」이 전체 데이터로 한 번 계산해 두었고, 실무에서는 `Pipeline` 으로 묶어 폴드 안에서 계산합니다.

</details>

<details class="lms-extra"><summary>주의 — AI 튜터에 위임할 작업과 직접 확인할 항목</summary>

이 미션은 값이 틀려도 오류가 발생하지 않습니다.

| AI 튜터에 위임할 작업 | 내가 반드시 확인할 항목 |
|---|---|
| 교차검증 반복문과 폴드 분할 | 적합에 전달한 행렬이 표준화한 `Xz` 인가 |
| `ALPHAS` 6개를 순회하는 반복문 | 분할을 **한 번만 만들어** alpha 6개가 함께 사용했는가 |
| 두 패널을 한 그림으로 그리기 | 점수를 적합에 사용하지 않은 **검증 폴드에서만** 산출했는가 |

</details>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_bias_variance_001.png" width="640" height="512"/>
<p align="center"><em>▲ 정규화의 편향과 분산의 절충 — 왼쪽 열의 위아래 패널 2개만 보면 된다. 위 패널은 같은 문제를 여러 표본으로 풀 때 표본마다 달라지는 예측이고, 그 산포가 분산이다. 아래 패널의 초록 variance 곡선이 그 산포를 수치로 분해한 것이다 (출처: scikit-learn documentation)</em></p>

<details class="lms-extra"><summary>자세히 — 정규화가 줄이는 것</summary>

패널티 항은 계수를 참값에서 이동시키는 대신 이 산포를 줄이고, 이득이 손해보다 큰지는 표본 크기에 따라 결정됩니다. 오른쪽 열의 모형 `Bagging(Tree)` 은 오늘 사용하지 않습니다.

</details>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 4일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

# 쓸 변수 1 — 설계행렬. 범주형을 0/1 더미 열로 펼쳐 모형에 그대로 들어가는 숫자 행렬입니다 (열 40개)
Xdesign = pd.get_dummies(df[["BonusMalus", "DrivAge", "Area", "VehGas", "VehBrand", "Region"]],
                         drop_first=True).astype(float)
Xdesign["logDensity"] = np.log(df["Density"])
COLS = list(Xdesign.columns)                                    # 쓸 변수 2 — 그 열 이름 40개

Xz = StandardScaler().fit_transform(Xdesign)                    # 쓸 변수 3 — 열마다 평균 0 · 표준편차 1 로 맞춘 같은 행렬
y_rate = (df["ClaimNb"] / df["Exposure"]).to_numpy()            # 쓸 변수 4 — 빈도 타깃 (건/년), offset 대신 쓰는 형태
w_exp = df["Exposure"].to_numpy()                               # 쓸 변수 5 — 노출 가중 (년)
ALPHAS = [1e-6, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]                    # 쓸 변수 6 — 벌점 세기 격자 여섯 개
SEED = 47                                                       # 쓸 변수 7 — 분할 난수 시드

print("설계행렬 Xdesign :", Xdesign.shape, "| 열 이름 앞 6개 :", COLS[:6])
print(f'원래 단위  BonusMalus {df["BonusMalus"].min():.0f}~{df["BonusMalus"].max():.0f}'
      f' · Area_F {Xdesign["Area_F"].min():.0f}~{Xdesign["Area_F"].max():.0f}'
      f' — 두 열이 같은 벌점을 받으면 공평할까요')
print("표준화한 Xz 의 BonusMalus 열 평균/표준편차 :",
      round(float(Xz[:, COLS.index("BonusMalus")].mean()), 6),
      "/", round(float(Xz[:, COLS.index("BonusMalus")].std()), 6))
print("빈도 타깃 y_rate 의 노출 가중평균 :", round(float(np.average(y_rate, weights=w_exp)), 6),
      " (전체 계약 사고율과 같아야 합니다)")
print("쓸 수 있는 이름 : df · Xdesign (설계행렬) · Xz (표준화한 설계행렬) · COLS (열 이름 40개) · "
      "y_rate (빈도 타깃, 건/년) · w_exp (노출 가중, 년) · ALPHAS (벌점 격자 6개) · SEED (난수 시드)")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 돌아가지만 답이 틀렸습니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 튜터가 「AI 코드 시작 행」 아래를 다시 씁니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — PROMPT 를 채우고 실행하면 어느 값이 틀렸는지 채점이 짚어 줍니다
coef_path = {}
for a in ALPHAS:                                     # 벌점을 키워 가며 계수를 한 벌씩 모읍니다
    fit = PoissonRegressor(alpha=a, max_iter=300, tol=1e-6).fit(
        Xdesign.to_numpy(), y_rate, sample_weight=w_exp)
    coef_path[a] = fit.coef_
coef_path_tbl = pd.DataFrame(coef_path, index=COLS)

print(f"계수 40개의 크기 합 (벌점 거의 없음, alpha = {ALPHAS[0]:g}) = {coef_path_tbl[ALPHAS[0]].abs().sum():.4f}")
print(f"계수 40개의 크기 합 (가장 센 벌점, alpha = {ALPHAS[-1]:g}) = {coef_path_tbl[ALPHAS[-1]].abs().sum():.4f}")

> **[문제] 미션 2 에서 고친 뒤의 코드가 계수 경로를 만드는 방식으로 맞는 것은?**
>
> 1. 원래 단위 그대로의 설계행렬을 적합에 넘긴다 — 패널티 항은 계수의 제곱합에 걸리니 열의 단위가 달라도 세기는 공평하다
> 2. 열마다 평균 0 · 표준편차 1 로 맞춘 설계행렬을 적합에 넘긴다 — 그래야 50～230 을 오가는 열과 0/1 더미 열이 같은 세기의 패널티 항을 받는다
> 3. 설계행렬은 원래 단위 그대로 두고 패널티 항의 세기를 열마다 다르게 준다 — 단위 차이는 alpha 쪽에서 보정한다

## AIC 순위와 교차검증 순위의 일치 여부

### 미션 3 — 어제 후보 3개의 교차검증 순위 재산출

#### 할 일 — 미션 3 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 미션 1 과 같은 순서로 AI 튜터에게 코드를 받아 실행하고, 채점 결과를 확인합니다. ❌ 면 힌트대로 PROMPT 를 고쳐 다시 합니다.

**미션 2 의 결과** — 패널티 항을 최적 값으로 적용해도 교차검증 이탈도는 0.000006 만 감소했습니다. 관측 678,013건에 계수는 41개라 계수 하나당 증거가 이미 충분하기 때문입니다. **정규화의 효과를 결정하는 것은 데이터 크기가 아니라 계수 하나당 증거량입니다** — 같은 계수 41개를 계약 5,000건만으로 추정한 대조 실험은 미션 2 의 완성본 풀이 마지막 항목에 있고, 거기서는 이탈도가 0.0328 낮아졌습니다.

**이 미션의 질문** — 어제 AIC 가 매긴 후보 3개의 순위는 이 표본의 로그우도로 계산한 것인데, 새 계약의 예측으로 다시 산출해도 그 순위가 그대로입니까?

**쓸 수 있는 것**

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 전처리 끝난 계약 표, 678,013행 | 고정 |
| `FOLDS` | 고정 분할 5쌍 — (학습, 검증) 행 번호 | 고정 |
| `FORMULAS` | 후보 3개의 formula — `null` · `core` · `full` | 고정 |
| `y_rate` | 빈도 타깃, 단위 건/년 | 고정 |
| `w_exp` | 노출 가중, 단위 년 | 고정 |
| `LAM_ALL` | 전체 계약 사고율 $\hat{\lambda}$ = 0.100614, 단위 건/년 | 고정 |
| 이탈도 기준선 | 전체 계약에 같은 요율 `LAM_ALL` 하나를 매겼을 때의 포아송 이탈도 0.624881 — 설명변수를 하나도 쓰지 않은 `null` 후보의 폴드 점수가 이 값 부근에 위치합니다 | 「준비 셀」 출력값, 읽고 대조 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **확장 모형(full)의 폴드 평균 이탈도** — 소수점 아래 6자리 · 폴드 5개의 점수를 평균한 값
   - 점수는 미션 2 와 같은 지표입니다. 다만 검증 폴드의 노출 `w_exp` 를 가중으로 전달한 평균 포아송 이탈도이고, 관측 수가 아니라 노출 합으로 나눈 값입니다
2. **core - full 짝지은 차이의 평균 / 표준오차** — 단위 배(표준오차의 배수) · 소수점 아래 2자리
   - 같은 폴드에서 `core` 의 이탈도에서 `full` 의 이탈도를 뺀 차이 5개를 만들고, 그 평균을 그 5개로 산출한 표준오차로 나눈 값입니다
   - 표준오차는 5개 값의 표준편차를 $\sqrt{5}$ 로 나눈 값입니다
3. **함께 출력할 3개 행** — 후보 3개마다 폴드 평균 · 폴드 간 표준오차 · 폴드별 점수 5개를 한 행씩 출력합니다
   - 채점 기준에 포함되는 값은 아니며, 평균 간 간격과 폴드 간 산포를 함께 확인하는 항목입니다
   - 여기에 한 행 더 — 폴드 평균이 가장 낮은 후보의 평균에 그 후보의 폴드 간 표준오차를 더한 1-표준오차 선과, 그 선 안에 드는 후보 이름을 함께 출력합니다
4. **그래프 한 장** — 패널 2개
   - 왼쪽 **후보별 폴드 점수** : 가로축은 후보 3개를 배치한 범주형 축, 세로축은 교차검증 포아송 이탈도인 산점도
     - 점 : 후보마다 폴드별 이탈도 5개, 그리고 평균 위치에 짧은 가로선 하나
     - 가로축 이름 : 「후보 모형」 · 세로축 이름 : 「교차검증 포아송 이탈도」
   - 오른쪽 **폴드별 짝지은 차이** : 가로축은 폴드 1 ～ 5, 세로축은 `core` 에서 `full` 을 뺀 이탈도 차이인 막대 그래프
     - 막대 : 폴드 하나에 하나씩, 모두 5개
     - 가로선 : 0 위치에 실선 하나, 차이의 평균 위치에 점선 하나. 2개 모두 가로축 전체를 가로지릅니다
     - 가로축 이름 : 「폴드」 · 세로축 이름 : 「core - full 이탈도 차이」
5. **함께 지킬 조건** — 3개 모두 지킵니다
   - 분할은 `FOLDS` 의 (학습, 검증) 행 번호 5쌍을 그대로 사용합니다. 후보 3개가 같은 폴드를 사용해야 순위가 분할의 우연에 좌우되지 않습니다
   - 적합은 학습 행 번호로 나눈 폴드에서만 `smf.glm(…, family=sm.families.Poisson(), offset=np.log(학습 조각["Exposure"]))` 으로 합니다
   - 검증 예측은 `predict(검증 조각, offset=np.zeros(len(검증 조각)))` 으로 받습니다. 오프셋을 0 으로 두면 노출 1년 기준 연율이 산출되어 `y_rate` 와 단위가 일치합니다

**출력 형식** — 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장입니다. 모형 3개 × 폴드 5개 = 적합 15회이므로 약 40초가 소요됩니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` · `FOLDS` · `FORMULAS` · `y_rate` · `w_exp` · `LAM_ALL` 은 이미 있으니 다시 만들지 말라고 적습니다.
- **짝지은 차이를 구하는 방법** — 후보마다 점수 5개를 모은 뒤 같은 폴드끼리 빼야 한다는 것, 나눌 표준오차가 그 차이 5개에서 산출된다는 것.
- **출력** — 값 2개의 이름과 단위, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 두 패널에 각각 무엇을 배치할지 · 점과 막대의 수 · 가로선 2개의 위치 · 두 패널의 가로축·세로축 이름.

**막히면** — AI 튜터에게 「검증 조각의 예측을 빈도 타깃과 같은 단위로 받으려면 offset 에 무엇을 전달하나요」 처럼 물어보세요.

#### 실행 전 예상 — 교차검증 순위의 변화

어제 AIC 가 매긴 순위는 `full` < `core` < `null` 이었습니다. 작을수록 좋은 점수이므로 AIC 가 가장 작은 후보는 `full` 입니다. 하나를 고르고 왜 그렇게 판단했는지 한 문장 적으세요.

> **[예측] 같은 후보 셋을 5-fold 교차검증 이탈도로 다시 순위 매기면 어떻게 될까요?**
>
> 1. 순위가 같고, 확장 모형의 우위도 폴드 간 산포보다 크게 나온다
> 2. 순위는 같지만, 확장 모형과 핵심 모형의 차이가 폴드 간 산포보다 작다
> 3. 순위가 반대가 되어 핵심 모형이 1위가 된다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>배경 — 폴드 점수 5개의 요약 통계량 3개 (이론 6. 정규화와 교차검증)</summary>

이번에는 설명변수의 개수를 비교합니다. 다만 `FOLDS` 는 무작위 분할이라 같은 지역의 계약이 학습과 검증에 함께 들어갑니다. 이론 5. 잔차 진단에서 지역 22개 가운데 17개가 무작위 범위 밖이었으므로 이 점수는 낙관적이고, 지역 더미를 쓴 `full` 쪽이 특히 그렇습니다.

**폴드 간 표준오차**는 한 모형의 점수가 폴드마다 얼마나 변동하는지를 측정하고, **짝지은 차이**는 같은 폴드에서 두 모형의 점수를 뺀 값 5개입니다. 평균 차이가 폴드 간 표준오차 안에 들어오면 교차검증은 AIC 와 다른 답을 준 것이 아니라 **아무 답도 주지 않은 것**입니다. 이번 미션에서 그 경우에 해당하는지부터 확인합니다.

이탈도는 단위 없는 점수이므로 위 표의 기준선 0.624881 과 견줘 읽습니다. 나머지 후보 2개가 그 기준선에서 얼마나 감소하는지가 읽을 값입니다.

</details>

<details class="lms-extra"><summary>주의 — AI 튜터에 위임할 작업과 직접 확인할 항목</summary>

이 미션은 조건을 어겨도 오류 없이 숫자가 산출됩니다.

| AI 튜터에 위임할 작업 | 내가 반드시 확인할 항목 |
|---|---|
| 폴드 5쌍을 순회하는 반복문과 적합 호출 | 검증 예측의 `offset` 이 0 이어서 연율 단위인가 |
| 이탈도를 계산해 후보별로 모으는 코드 | 적합이 **학습 행 번호 조각에서만** 이루어졌는가 |
| 짝지은 차이와 표준오차 계산 | 짝지은 차이를 **같은 폴드끼리** 뺐는가 |

</details>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/KfoldCV.gif" width="640" height="147"/>
<p align="center"><em>▲ 관측 12개(n = 12)를 폴드 3개(k = 3)로 나눠 3회 학습하면 관측 하나하나가 정확히 한 번씩 검증에 사용된다 — 이번 미션은 같은 절차를 폴드 5개로 실행하고, 이 계약 자료에서는 폴드 하나가 계약 약 13만 5천 건이다 (출처: Wikimedia Commons)</em></p>

<details class="lms-extra"><summary>자세히 — 회차별 검증 폴드의 위치</summary>

파란 블록이 그 회차의 검증 폴드이고 빨간 블록이 학습 폴드이며, 파란 블록이 회차마다 오른쪽으로 한 폴드씩 이동합니다. 「준비 셀」이 분할을 <code>shuffle=True</code> 로 만드는 것은 원래 순서대로 나누면 폴드마다 계약의 성격이 편중될 수 있기 때문입니다.

</details>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_grid_search_stats_003.png" width="600" height="450"/>
<p align="center"><em>▲ 두 모형의 교차검증 점수 차이를 분포 하나로 그린 그림(scikit-learn 의 모형 비교 예제). 최빈값은 0 의 오른쪽 0.01 부근에 위치하지만 왼쪽 꼬리가 0 을 넘어가, 한쪽의 점수가 더 높다고 단정하지는 못한다 (출처: scikit-learn documentation)</em></p>

<details class="lms-extra"><summary>자세히 — 그림의 가로축과 부호 방향</summary>

가로축은 두 모형의 평균 점수 차이이고, 세로축은 그 차이가 그 값일 확률밀도입니다. 그림의 가로축 μ 는 이 그림에서만 쓰는 표기이고, 이 과정에서 μ 는 기대 건수입니다. 다만 이 예시의 점수는 정확도라 클수록 좋고, 우리 이탈도는 작을수록 좋아 부호를 반대로 읽습니다.

</details>

In [ ]:
# 미션 3 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 3 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 4일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

SEED = 47                                                       # 쓸 변수 1 — 분할 난수 시드
FOLDS = list(KFold(5, shuffle=True, random_state=SEED).split(df))   # 쓸 변수 2 — (학습, 검증) 인덱스 5쌍
y_rate = (df["ClaimNb"] / df["Exposure"]).to_numpy()            # 쓸 변수 3 — 빈도 타깃 (건/년)
w_exp = df["Exposure"].to_numpy()                               # 쓸 변수 4 — 노출 가중 (년)
LAM_ALL = float(df["ClaimNb"].sum() / df["Exposure"].sum())     # 쓸 변수 5 — 전체 계약 사고율 (건/년)

FORMULAS = {                                                    # 쓸 변수 6 — 모형 이름 -> formula 문자열
    "null": "ClaimNb ~ 1",
    "core": "ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)",
    "full": "ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)"
            " + C(VehBrand) + C(Region) + np.log(Density)"}
print("어제 정한 후보 셋을 이 셀이 그대로 다시 적습니다\n")

for i, (tr, te) in enumerate(FOLDS):
    print(f"폴드 {i + 1} : 학습 {len(tr):,}건 · 검증 {len(te):,}건 · 겹침 {len(set(tr) & set(te)):,}건")
print("\n검증할 후보 셋")
for name, formula in FORMULAS.items():
    print(f"  {name:5s} : {formula}")
print(f"\n이탈도 기준선 — 전체 계약에 같은 요율 {LAM_ALL:.6f} 건/년 하나를 매겼을 때 : "
      f"{mean_poisson_deviance(y_rate, np.full(len(df), LAM_ALL), sample_weight=w_exp):.6f}"
      "  (단위 없는 점수라 이 값과 견줘 읽습니다)")
print("쓸 수 있는 이름 : df (전처리 끝난 계약 표) · FOLDS (고정 분할 5쌍) · "
      "FORMULAS (후보 셋의 formula) · y_rate (빈도 타깃, 건/년) · w_exp (노출 가중, 년) · "
      "LAM_ALL (전체 계약 사고율, 건/년)")

In [ ]:
# 미션 3 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 폴드별 점수를 모형이 아니라 폴드 방향으로 읽으면 후보 3개의 점수가 같은 폴드에서 동시에 증가·감소한다. 그래서 폴드 간 표준오차는 두 모형의 점수 차이를 판정하기에는 정밀도가 낮은 기준이고, 같은 폴드 안에서 뺀 짝지은 차이가 정밀도가 훨씬 높은 기준이다.**
>
> 1. O (맞다)
> 2. X (틀리다)

## 보고할 신뢰구간 길이의 보정

> **[과제] 과제 제출 — 보고할 신뢰구간 길이의 보정**
>
> - **무엇을** 등분산·독립 2개의 가정을 확인해 값 2개 — 표준오차 배수와 분산÷평균이 1 위쪽인 칸 수 — 와 근거 표 1개·그림 한 장을 만듭니다.
> - **무엇으로** 오늘 쓴 계약 678,013건 하나로 하고, 「준비 셀」이 미션 1 의 세그먼트 8개 구간 표와 핵심 모형의 식을 다시 생성해 둡니다.
> - **인정** 채점이 값 2개와 그래프를 통과시킨 노트북에 브리핑을 최대 4문장으로 작성한 것입니다.
> - **예** 점검표 한 행은 「등분산(과산포) · 구간 15개의 분산÷평균 · √φ̂ 배 · 판정」 형태이고, 값 이름도 「분산÷평균이 1 위쪽인 칸 수」로 적습니다. 미션 1 의 Region R43 구간이라면 보정 전과 보정 후의 구간 길이를 건/년으로 적습니다.
> 
> - 내가 쓴 `PROMPT` 를 지우지 말고 그대로 둔 채 「PROMPT 셀」을 실행해, 값 2개와 그래프가 모두 통과로 열린 상태로 제출할 것
> - 가정 점검표에 「무엇으로 봤나 · 진단값 · 구간 길이에 매길 배수 · 판정」 4개 열을 두고 한 행씩 적을 것
> - 미션 1 의 세그먼트 8개 구간 하나를 골라 보정 전과 보정 후의 구간 길이를 건/년으로 적을 것
> - 브리핑을 아래 「제출 전 마지막 한 장」 양식에 결론·근거·한계 순서로 최대 4문장까지 작성할 것
>
> 마감: 2026-10-02T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | 「PROMPT 셀」의 `PROMPT` 에 목표·쓸 변수·출력 형식이 내 말로 적혀 있다 | 25 |
> | 검증 설계 | 값 2개와 그래프가 모두 통과로 열린 상태이고, 진단값을 산출한 표본이 무엇인지 밝혀져 있다 | 25 |
> | 오류 탐지 | 과산포 배수와 군집 상관 배수를 섞지 않고, 구간 길이에 한 번 곱해 보정할 수 있는 것과 없는 것을 구분해 적었다 | 25 |
> | 해석·보고 | 브리핑에 결론·근거·한계가 각 한 문장씩 있고, 한계만 2문장까지 늘려 최대 4문장 안에 끝난다 | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 과제 — 보고할 신뢰구간 길이의 보정

#### 할 일 — 과제 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 미션 1 과 같은 순서로 AI 튜터에게 코드를 받아 실행하고, 채점 결과를 확인합니다. ❌ 면 힌트대로 PROMPT 를 고쳐 다시 합니다.
4. 아래 「제출 전 마지막 한 장」 양식에 브리핑을 결론·근거·한계 순서로 최대 4문장까지 작성해 제출합니다.

**이 과제의 질문** — 미션 1 에서 산출한 구간은 등분산과 독립 2개의 가정을 전제하는데, 그 가정이 성립하지 않는다면 구간 길이를 얼마나 넓혀 보고해야 합니까?

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Residuals_for_Linear_Regression_Fit.png" width="600" height="400"/>
<p align="center"><em>▲ 잔차가 무엇인지 한 장으로 — 가로축이 설명변수, 세로축이 관측값이고 빨간 직선이 적합한 모형이다. 파란 점에서 직선까지 내려긋는 검은 선분 하나하나가 그 관측의 잔차이고, 위로 뻗었으면 모형이 낮게 예측한 것이다. 오늘은 이 선분을 지역으로 묶어 평균을 계산하고, 그 평균이 0 부근에 있는지를 확인한다 (출처: Wikipedia, Residuals for Linear Regression Fit.png)</em></p>

**쓸 수 있는 것**

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `df` | 전처리 끝난 계약 표, 678,013행 | 고정 |
| `F_CORE` | 어제의 핵심 모형 식, 계수 9개 | 그대로 적합한다 |
| `REG` | 지역 라벨 22개 수준 | 고정 |
| `BOOT` | 미션 1 의 세그먼트 8개 구간 표 — 열은 `세그먼트` · `n` · `claims` · `exposure` · `lam_hat` · `lo95` · `hi95` · `se` · `폭`(구간 길이) · `상대폭`(상대 구간 길이) | 고정 |
| `B` | 재배치·재표본 수 1,000 — 미션 1 의 재표본 수와 같은 값 | 고정 |
| `SEED` | 난수 시드 | 고정 |

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_003.png" width="600" height="400"/>
<p align="center"><em>▲ 예측값 순으로 구간을 분할해 비교한 그림(scikit-learn 의 포아송 회귀 예제). 가로축은 예측 위험이 낮은 쪽부터 매긴 계약의 순위, 세로축은 그 구간의 평균 사고 빈도이고, 파란 × 가 모형의 예측, 주황 ● 가 실제 관측이다. 패널 제목의 모형 이름 4개는 오늘 사용하지 않으니 한 패널만 보면 된다 (출처: scikit-learn documentation)</em></p>

이 예시는 구간이 10개이고, 오늘 과제는 15개로 분할한 뒤 구간마다 평균 대신 「분산 ÷ 평균」 을 계산합니다.

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다.

1. **할증점수 계수의 표준오차 배수 — 군집 ÷ 포아송** — 단위 배 · 소수점 아래 2자리 · 같은 계수에서 군집 표준오차를 포아송 표준오차로 나눈 값
2. **분산÷평균이 1 위쪽인 칸 수** — 단위 칸 · 정수 · 계약을 예측값 순으로 구간 15개로 분할하고, 구간마다 사고 건수의 「분산 ÷ 평균」 을 산출해 1 을 넘는 구간의 개수를 집계합니다
   - 여기서 한 칸은 이론 5. 잔차 진단이 「15개 구간」이라 부른 그 구간 하나이고, 출력할 이름은 위 굵은 글자 그대로 씁니다
   - `BOOT` 의 `세그먼트` 열과는 다른 대상입니다 — 여기서 집계하는 칸은 예측값 순으로 나눈 구간 15개입니다
3. **함께 출력할 표 한 장** — 「무엇으로 봤나 · 진단값 · 구간 길이에 매길 배수 · 판정」 4개 열에 가정 2개를 한 행씩 적습니다
   - 채점 기준에 포함되는 값은 아니며, 브리핑의 근거가 됩니다
   - 등분산(과산포) : 구간 15개의 분산÷평균을 평균한 값이 과산포 φ̂ 이고, 구간 길이에 매길 배수는 그 제곱근 √φ̂ 입니다
   - 독립(지역 군집 상관) : 기준구간 밖으로 나간 지역 수와 1번 배수가 진단값이고, 구간 길이에 매길 배수는 계수마다 다릅니다
   - 기준구간 밖으로 나간 지역 수는 4번 그래프에서 마커가 그 지역의 막대 밖에 위치한 지역을 집계한 값입니다 — 「무작위 재배치 기준구간 밖으로 나간 지역 = ○ / 22 곳」 한 행으로 함께 출력합니다
4. **그래프 한 장** — 가로축은 이탈도 잔차의 지역별 평균, 세로축은 지역 22개인 구간 그림
   - 무작위 재배치 기준구간 : 지역 라벨을 무작위로 재배치해 다시 산출한 지역별 평균들의 2.5 ～ 97.5% 범위입니다
   - 가로 막대 : 지역마다 그 기준구간의 두 끝을 잇는 하나씩, 모두 22개
   - 마커 : 지역마다 실제 지역별 평균 위치에 하나씩
   - 세로선 : 0 위치에 세로축 전체를 가로지르는 선 하나
   - 가로축 이름 : 「이탈도 잔차의 지역별 평균 (전체 평균을 뺀 값)」
   - 세로축 이름 : 「지역」
5. **보정 전과 보정 후의 구간 길이 한 행** — 단위 건/년 · `BOOT` 에서 세그먼트 하나를 골라 그 `폭` 열의 값과, 거기에 √φ̂ 를 곱한 보정 후 값을 나란히 출력합니다
   - 채점 기준에 포함되는 값은 아니며, 제출 조건과 브리핑 두 번째 문장이 그대로 이 2개 값을 묻습니다
   - 백분위 부트스트랩은 계약을 그대로 다시 뽑아 분산 = 평균 가정을 쓰지 않으므로, 여기서 곱하는 √φ̂ 는 과산포를 한 번 더 반영하는 대조용입니다. 세그먼트 8개의 부트스트랩 표준오차가 포아송 공식 √(사고 합) ÷ 노출 합의 평균 1.04배라 과산포만큼은 이미 구간 길이에 들어 있고, √φ̂ 를 곱해야 하는 것은 포아송 가정으로 계산한 ±2 SE 구간입니다
6. **함께 지킬 조건** — 4개 모두 지킵니다
   - 모형은 `F_CORE` 를 `smf.glm(…, family=sm.families.Poisson(), offset=np.log(df["Exposure"]))` 로 한 번만 적합합니다
   - 군집 표준오차는 같은 식을 `fit(cov_type="cluster", cov_kwds={"groups": REG})` 로 한 번 더 적합해 받습니다. `HC0` 같은 강건 표준오차로는 배수가 1.00 부근으로 산출되어 군집 상관이 드러나지 않습니다
   - 잔차는 이탈도 잔차 `.resid_deviance` 를 쓰고, 지역별 평균을 계산하기 전에 전체 평균을 빼 0 을 가운데 둡니다
   - 기준구간은 `SEED` 로 고정해 `B` 회 실행합니다. 같은 잔차를 실제 지역별 계약 수와 크기가 같은 부분집합 22개에 다시 배분해 부분집합별 평균을 모으면, 그 가운데 95% 범위가 기준구간입니다

**출력 형식** — 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장이고, 표는 파일로 내보내지 않습니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `df` · `F_CORE` · `REG` · `BOOT` · `B` · `SEED` 는 이미 있으니 다시 만들지 말라고 적습니다.
- **적합 2회** — 같은 식을 표준오차만 바꿔 2회 적합할 것, 나눌 두 표준오차가 같은 계수의 것이라는 점.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지와 구간을 몇 개로 분할할지, 그리고 기준구간 밖으로 나간 지역 수와 `BOOT` 에서 고른 세그먼트의 보정 전·보정 후 구간 길이를 건/년으로 함께 출력할 것.
- **그래프** — 가로축·세로축에 무엇을 배치할지 · 가로 막대와 마커 · 세로선의 위치 · 가로축·세로축 이름.

**막히면** — AI 튜터에게 「같은 계수의 포아송 표준오차와 군집 표준오차를 어떻게 나란히 꺼내나요」 처럼 물어보세요.

<details class="lms-extra"><summary>주의 — AI 튜터에 위임할 작업과 직접 확인할 항목</summary>

적합을 2회 실행하는 과제라, 값이 틀려도 오류가 발생하지 않는 항목이 2개 있습니다.

| AI 튜터에 위임할 작업 | 내가 반드시 확인할 항목 |
|---|---|
| 같은 식을 2회 적합하고 표준오차를 꺼내기 | 나눈 두 표준오차가 **같은 계수**의 것인가 |
| 예측값 순으로 구간 15개를 분할하는 코드 | 잔차에서 **전체 평균을 빼고** 지역으로 묶었는가 |
| 재배치 반복문과 백분위 절단 | 기준구간을 `SEED` 로 고정해 다시 실행해도 같은가 |

</details>

<details class="lms-extra"><summary>배경 — 과산포와 군집 상관의 구분</summary>

이론 5. 잔차 진단이 잔차 위에서 가정 3개를 확인합니다. 백분위 부트스트랩은 정규성을 사용하지 않으므로 남는 가정은 등분산과 독립 2개이고, ±z SE 로 만든 구간은 3개를 모두 전제합니다.

**과산포**는 실제 분산이 포아송 모형이 가정하는 분산, 곧 평균보다 큰 상태입니다. 가정대로면 「분산 ÷ 평균」 이 1 이고, 이 비 φ̂ 의 제곱근만큼 표준오차가 증가합니다. 모든 계수에 같은 배수가 적용되므로 구간 길이에 한 번 곱해 보정할 수 있습니다.

**군집 상관**은 같은 지역의 계약이 함께 변동하는 상태입니다. 관측은 678,013건인데 서로 독립인 부분집합은 22개뿐인 셈이라, 표준오차가 반영하는 「유효 표본 수」 가 감소합니다. 이쪽은 산포의 문제가 아니라 표본 수의 문제라, 계수마다 배수가 달라 한 숫자로 보정할 수 없습니다.

같은 모형의 실측값은 이론 5. 잔차 진단의 표에 있습니다. 할증점수 계수의 표준오차가 0.00031 에서 0.00163 으로 커졌습니다.

**이탈도 잔차**는 관측 하나가 이탈도에 기여한 값에 부호를 부여한 값이고, 원 잔차나 피어슨 잔차보다 지역별 평균을 읽기에 적합합니다.

</details>

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Heteroscedasticity.png" width="556" height="357"/>
<p align="center"><em>▲ 등분산이 성립하지 않는 경우 — 가로축이 설명변수, 세로축이 관측값이고 점 하나가 관측 하나다. 왼쪽에서는 점이 좁은 범위 안에 모여 있다가 오른쪽으로 갈수록 위아래로 크게 흩어진다. 값이 큰 쪽에서 산포가 더 크다는 뜻이고, 오늘 과제가 예측값 순으로 구간을 분할해 계산하는 이유가 이 그림에 있다 (출처: Wikipedia, Heteroscedasticity.png)</em></p>

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 과제만 따로 실행해도 단독으로 동작합니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)      # 4일 동안 고정해 온 전처리 2가지
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

B, SEED = 1000, 47                                              # 쓸 변수 1 — 재배치·재표본 수와 난수 시드
F_CORE = "ClaimNb ~ BonusMalus + DrivAge + C(Area) + C(VehGas)"  # 쓸 변수 2 — 어제의 핵심 모형 식 (계수 9개)
REG = df["Region"].astype(str).to_numpy()                        # 쓸 변수 3 — 지역 라벨 (22수준)

_SEGMENTS = {f"Area {a}": (df["Area"] == a) for a in list("ABCDEF")}   # 미션 1 의 여덟 세그먼트
_SEGMENTS["Region R43"] = (df["Region"] == "R43")
_SEGMENTS["Region R42"] = (df["Region"] == "R42")
_rows = []
for seg_name, mask in _SEGMENTS.items():                                 # 미션 1 의 구간 표를 이 셀이 그대로 다시 만듭니다
    d = df[mask]
    y, E = d["ClaimNb"].to_numpy(float), d["Exposure"].to_numpy(float)
    rng = np.random.default_rng(SEED)
    s = np.empty(B)
    for b in range(B):
        i = rng.integers(0, len(y), len(y))                      # 복원추출 — 미션 1 과 같은 시드·같은 벌수
        s[b] = y[i].sum() / E[i].sum()
    _rows.append({"세그먼트": seg_name, "n": len(d), "claims": int(y.sum()), "exposure": float(E.sum()),
                  "lam_hat": float(y.sum() / E.sum()),
                  "lo95": float(np.percentile(s, 2.5)), "hi95": float(np.percentile(s, 97.5)),
                  "se": float(s.std(ddof=1))})
BOOT = pd.DataFrame(_rows)                                       # 쓸 변수 4 — 미션 1 의 여덟 세그먼트 구간 표
BOOT["폭"] = BOOT["hi95"] - BOOT["lo95"]
BOOT["상대폭"] = BOOT["폭"] / BOOT["lam_hat"]

print("어제의 핵심 모형 F_CORE :", F_CORE)
print("  어제 채점표의 core 행 — 로그우도 -144,065.91 · AIC 288,149.81 (계수 9개)")
print(f"지역 라벨 REG : {len(set(REG))}수준 — 핵심 모형에 지역이 들어 있지 않아 "
      "지역 차이가 계수로 흡수되지 않고 잔차에 남습니다")
_r43 = BOOT.set_index("세그먼트").loc["Region R43"]
print(f"미션 1 에서 본 그 폭 — Region R43 : {_r43['lo95']:.5f} ~ {_r43['hi95']:.5f} 건/년 "
      f"(폭 {_r43['폭']:.6f} · 상대폭 {_r43['상대폭']:.4f})")
print(f"재배치·재표본 수 B = {B} · 난수 시드 SEED = {SEED}")
print("쓸 수 있는 이름 : df (전처리 끝난 계약 표) · F_CORE (어제의 핵심 모형 식) · "
      "REG (지역 라벨 22수준) · BOOT (미션 1 의 여덟 세그먼트 구간 표) · "
      "B (재배치·재표본 수) · SEED (난수 시드)")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

**제출 전 마지막 한 장** — 값 2개와 그래프가 모두 통과로 열렸으면, 아래 양식대로 브리핑을 작성해 노트북 끝에 남깁니다. 결론·근거·한계 각 1문장, 한계만 2문장까지 — 최대 4문장입니다. 채점자는 이 브리핑을 가장 먼저 읽습니다.

> 브리핑:
> 1. 결론 — 구간 길이에 한 번 곱해 보정한 배수 ○○ 배와, 한 번 곱해 보정할 수 없는 항목을 나란히 적습니다
> 2. 근거 — 가정 2개 중 무엇이 얼마나 성립하지 않는지 한 문장으로. 배수를 숫자와 단위로
> 3. 한계 — 군집이 22개뿐이라 배수 자체도 근사값이라는 것, 그리고 계약이 적은 세그먼트를 어떻게 적을지
>    (상위 세그먼트로 통합할지, 둘째 날의 수축으로 줄일지, 사람 심사로 보낼지)

3번 문장은 **적용 범위**에서 나뉩니다 — 이 결정을 요율표 전체에 적용할지, 일부 세그먼트에만 적용할지입니다. 요율표 전체에 한 모형을 사용하면 설명이 단순한 대신 계약이 적은 세그먼트의 변동이 큰 상태를 그대로 유지하고, 계약이 적은 세그먼트를 따로 다루면 그 세그먼트의 변동은 작아지는 대신 주석이 늘어납니다. 따로 다루기로 했다면 미션 1 의 `상대폭` 으로 세그먼트를 2개 등급으로 나눠 적으세요. 그대로 요율을 매길 세그먼트와, 위 3번의 3가지 가운데 하나로 보낼 세그먼트입니다.

## 오늘의 정리와 사전분포 세기의 결정 방법 예고

오늘 내용을 정리합니다. 오늘 아침 재무팀이 질문한 「이 수치가 얼마나 믿을 만하냐」 에 대한 답은 한 문장입니다. **세그먼트마다 점추정 옆에 95% 구간을 나란히 적고, 그 구간이 전제하는 가정까지 확인해 구간 길이를 다시 적습니다.**

이론 4. 신뢰구간을 이용한 의사결정의 둘째 문장도 같이 적용됩니다 — 실무가 「이 정도 차이는 없는 것으로 본다」 고 정해 둔 범위를 ROPE 라 하고, 산출한 구간이 그 범위를 넘는지까지 적어야 결정이 완결됩니다. 오늘 과제에서는 그 기준과 대조할 구간 길이를 보정해 두었습니다.

오늘 남는 원칙은 한 문장입니다 — **점추정은 가정이 성립하지 않아도 크게 달라지지 않지만 구간은 달라집니다.** 과제에서 할증점수 계수의 표준오차가 약 5배로 증가하자 계수를 그 표준오차로 나눈 확신의 크기(|z|)도 그만큼 감소했습니다. 보고서에서 먼저 영향을 받는 항목은 항상 구간 길이입니다.

| 오늘의 질문 | 우리가 얻은 답 |
|---|---|
| 세그먼트마다 신뢰구간 길이가 왜 다른가? | 재표본 1,000개의 백분위 구간으로 산출하니 계약 191,880건의 Area C 와 계약 1,326건의 지역 R43 의 상대 구간 길이가 13배 차이를 보였습니다. 구간 길이를 정하는 것은 계약 수가 아니라 **누적된 사고 건수**이고, √(Area C 사고 합 9,875건 ÷ 지역 R43 사고 합 57건) ≈ 13.2 가 그대로 13배로 나타납니다 |
| 계수 40개에 정규화가 필요한가? | 패널티 항을 최적으로 골라도 이탈도 개선이 0.000006 에 그쳤고, 계약 5,000건 대조 실험에서는 0.0328 이었습니다. **정규화의 효과를 결정하는 것은 데이터 크기가 아니라 계수 하나당 증거량입니다** |
| AIC 순위가 교차검증 순위와 일치하는가? | 무작위 분할 기준으로는 순위가 그대로였습니다. 다만 확장 모형의 이탈도 차이 0.002414 가 폴드 간 표준오차 0.002617 안이었고, 같은 폴드에서 뺀 짝지은 차이로 보면 그 평균이 표준오차의 8.19배였습니다. **평균만 적으면 절반만 보고하는 것입니다** |
| 그 구간을 그대로 보고해도 되는가? | 과산포는 φ̂ ≈ 1.06 이라 구간 길이를 √φ̂ 배, 곧 약 3% 늘리는 데 그쳤지만, 지역 군집 상관은 할증점수 계수의 표준오차를 약 5배로 증가시켰습니다. **구간 길이에 한 번 곱해 보정할 수 있는 것과 없는 것을 구분해 적는 것**이 오늘의 마지막 단계입니다 |

<details class="lms-extra"><summary>답 확인 — 폴드별 점수에 나타나는 공통 변동</summary>

내 화면의 폴드별 점수 5개를 모형 방향이 아니라 폴드 방향으로 읽어 보면 같은 폴드에서 후보 3개의 점수가 동시에 증가·감소합니다. 그 공통 변동이 폴드 간 표준오차의 대부분이고, 같은 폴드 안에서 빼면 제거되어 차이의 표준오차가 훨씬 작아집니다. 두 문장은 서로 다른 기준으로 산출한 결과라 보고서에는 둘 다 적습니다.

</details>

4일 동안의 내용은 3문장입니다. 전체 계약 사고율 0.100614 건/년에서 시작해 계약이 적은 세그먼트를 수축으로 조정했습니다. 이어서 계약 특성으로 계약마다 다른 사고율을 계산했습니다. 오늘은 그 값마다 구간을 산출하고 그 구간이 어떤 가정 위에서 나온 것인지까지 적었습니다.

다만 오늘까지 만든 것은 요율을 이루는 두 요소 가운데 빈도 하나입니다. 실제 요율은 빈도 × 평균 지급액으로 만들고, 미션 밖에서 부트스트랩 예제로 쓴 지급액 30건(평균 1,773.79유로)이 나머지 하나인 평균 지급액의 자료입니다.

## 내일 예고 — 수축 정도의 결정 주체

오늘 구간 길이를 산출하고 가정을 확인하고 나면 남는 질문이 하나입니다 — **계약이 적은 세그먼트의 넓은 구간을 좁히려면 그 세그먼트 밖의 정보를 빌려 와야 하는데, 얼마나 빌릴지를 무엇이 정하는가.** 둘째 날에는 그 세기를 우리가 직접 선택했습니다. 내일은 **데이터가 그 세기를 정하게** 합니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/MSE_of_ML_vs_JS.png" width="560" height="456"/>
<p align="center"><em>▲ 둘째 날에 다룬 그림을 한 번 더 — 여러 세그먼트를 한꺼번에 추정할 때, 빨간 수축 추정값의 평균제곱오차는 참값이 사전분포의 중심 부근일 때 파란 MLE 보다 크게 낮고 멀어질수록 MLE 에 수렴한다. 화면의 범례 ML 이 그 파란 최대우도, JS 가 빨간 수축이다 (출처: Wikipedia, MSE of ML vs JS.png, 퍼블릭 도메인)</em></p>

가로축 ‖θ‖² 는 참값이 사전분포의 중심에서 떨어진 거리의 제곱이고, 세로축 R 은 평균제곱오차라 아래로 내려갈수록 좋습니다. 빨간 JS 가 파란 ML 아래에 있는 구간이 수축 추정의 평균제곱오차가 더 작은 범위입니다.

오늘의 마무리 과제는 2문장입니다. 채점이 열어 준 완성본과 내 코드가 달랐던 부분 한 곳, 그리고 과제에서 구간을 얼마나 넓히기로 했는지와 그 이유 한 문장.

팀 채널에 공유해 두고 다른 사람의 2문장을 읽는 것이 오늘의 마지막 복습입니다. 함께 볼 팀이 없다면 내일의 나에게 남기는 메모로 적어 두면 같은 역할을 합니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 모형추정 Day 4 완료</h2>
<div style="color:#656565">세그먼트마다 구간 길이를 산출하고, 계수에 패널티 항을 적용해 보고, 순위를 새 데이터에서 다시 매기고, 그 구간이 전제하는 가정까지 확인했습니다.<br/>내일은 계약이 적은 세그먼트의 구간을 좁힐 사전분포의 세기를 데이터가 고르게 합니다.</div>
</div>

> **[문제] 이사회 보고 가능 여부의 판단 근거 3문장**
>
> 재무팀이 마지막으로 다시 질문합니다. "그래서 이 요율표, 이사회에 그대로 제출해도 됩니까?" 오늘 만든 것만으로 답해 보세요. (가) 세그먼트 하나의 숫자를 보고할 때 무엇을 함께 적어야 하는지, (나) 계약이 적은 세그먼트와 많은 세그먼트를 같은 표에 올릴 때 어떤 주석이 필요한지, (다) 요율표에 사용할 모형을 고를 때 현재 표본에서 계산한 점수 하나만으로 판단할 수 없는 이유 — 3문장으로 적어 보세요.